# Full-video baseline and targeted review comparison

Import this notebook, attach the private dataset containing `video.mp4`, enable Internet and a GPU, and configure the `HF_TOKEN` Kaggle secret. Run from the top.

The first output is the unchanged current baseline. A second, checkpointed pass reviews only uncertain, weak, short, skipped, or explicitly selected control regions. It writes supplemental hypotheses and never replaces baseline text, speaker IDs, timings, confidence values, or evidence. Complete audio windows use Whisper's native decoder; voice matching compares decoder and WhisperX-alignment timing crops. Conflicting identities stay `Uncertain`.

The review pass does not enable the rejected denoiser, spectral filters, bounded-Silero transcription, mouth-motion attribution, or hard decoder-warning cutoff. The two officers may remain unresolved.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time, zipfile

# Local: set this to the folder containing video.mp4 and the target .npy files.
# Kaggle: leave None; the attached dataset is found automatically.
DATA_DIR = None
RUN_FULL_VIDEO = True
RUN_GAP_RECOVERY = False  # Older separate text-only pass; targeted review already includes gaps
RUN_TARGETED_REVIEW = True
RUN_HEIGHT_CONTROL = False  # Opening sanity check is enough before the unchanged full baseline
REVIEW_WEAK_CONFIDENCE = 0.35
REVIEW_SHORT_SECONDS = 1.0
# External evaluation controls only; these are not rules inside the review algorithm.
REVIEW_CONTROL_REGIONS = [(640, 665), (665, 690), (1085, 1150)]
BATCH_SIZE = 4
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    # Fail before lengthy installation if this session has no GPU.
    probe = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    if probe is None or probe.returncode or 'GPU ' not in probe.stdout:
        raise RuntimeError('No GPU in this session. Enable GPU T4 x2 in Settings, then rerun this cell.')
    print(probe.stdout)
    matches = list(Path('/kaggle/input').rglob('video.mp4'))
    if DATA_DIR is None:
        if len(matches) != 1:
            raise RuntimeError('Attach the private video dataset; expected one video.mp4, or set DATA_DIR explicitly.')
        DATA_DIR = matches[0].parent
    BASE = Path('/kaggle/working')
else:
    # Convenience for the current local project; otherwise use the chosen folder.
    local_project = Path('/home/think/projects/whisperx_diarization')
    DATA_DIR = DATA_DIR or (local_project if local_project.exists() else Path.cwd())
    BASE = Path.cwd()/'diarization-run'
DATA = Path(DATA_DIR).expanduser().resolve()
for name in ('video.mp4',):
    if not (DATA/name).is_file():
        raise RuntimeError(f'Missing {name}. Set DATA_DIR to the project or extracted dataset folder.')
BASE.mkdir(parents=True, exist_ok=True)
WORK = BASE/'diarization'
WORK.mkdir(exist_ok=True)
RESULTS = BASE/'results'
RESULTS.mkdir(exist_ok=True)
CACHE = BASE/'stage-cache'
VENV = BASE/'diarization-venv'
PYTHON = str(VENV/('Scripts/python.exe' if os.name == 'nt' else 'bin/python'))
print('Input folder:', DATA)
print('Results folder:', RESULTS)
print('Setup will use:', PYTHON)
REFERENCE = WORK/'target-reference'
REFERENCE.mkdir(exist_ok=True)


## Install and verify the environment
No activation or separate `wrapt`/ONNX repair cells are needed. Setup checks whether pip works, rather than only checking that Python exists. It uses virtualenv's bundled pip when creation or repair is needed. Installing dependencies can take several minutes; progress appears below.

In [ ]:
EMBEDDED_FILES = {'chainofrules.py': '"""Evidence-based refactor of the recovered runnable chainofrules.py.\n'
                    'Run from the existing project directory with HF_TOKEN set in the environment.\n'
                    'Accepts any video and matching target embedding files.\n'
                    'All rows in each embedding file are reference samples of the same target.\n'
                    'Defaults retain short.mp4, large-v2, ECAPA and buffalo_l.\n'
                    'Confidence values are heuristic evidence strengths, not calibrated probabilities.\n'
                    '"""\n'
                    'import os\n'
                    'import argparse\n'
                    'import json\n'
                    'import re\n'
                    'import gc\n'
                    'from pathlib import Path\n'
                    'from importlib.metadata import version\n'
                    'import pandas as pd\n'
                    'import onnxruntime as ort\n'
                    'from cloud_runtime import StageCache, file_digest, create_face_analyzer, '
                    'create_full_audio_vad\n'
                    'from dataclasses import dataclass, field, asdict\n'
                    'import cv2\n'
                    'import numpy as np\n'
                    'import torch\n'
                    'import torchaudio\n'
                    'import whisperx\n'
                    'from whisperx.diarize import DiarizationPipeline\n'
                    'from speechbrain.inference.speaker import SpeakerRecognition\n'
                    'from insightface.app import FaceAnalysis\n'
                    'from collections import defaultdict\n'
                    '\n'
                    '\n'
                    '@dataclass(frozen=True)\n'
                    'class Baseline:\n'
                    '    raw_speaker_track: str\n'
                    '    speaker: str\n'
                    '\n'
                    '@dataclass(frozen=True)\n'
                    'class Evidence:\n'
                    '    source: str\n'
                    '    target_score: float\n'
                    '    confidence: float\n'
                    '    details: dict = field(default_factory=dict)\n'
                    '\n'
                    '@dataclass\n'
                    'class TimelineSegment:\n'
                    '    start: float\n'
                    '    end: float\n'
                    '    text: str\n'
                    '    baseline: Baseline\n'
                    '    words: list = field(default_factory=list)\n'
                    '    evidence: list = field(default_factory=list)\n'
                    '    final_speaker: str = "Uncertain"\n'
                    '    final_confidence: float = 0.0\n'
                    '    reasons: list = field(default_factory=list)\n'
                    '\n'
                    '\n'
                    'def normalize_vector(value):\n'
                    '    value = np.asarray(value, dtype=np.float32).reshape(-1)\n'
                    '    norm = np.linalg.norm(value)\n'
                    '    if not np.all(np.isfinite(value)) or norm <= 0:\n'
                    '        raise ValueError("Embedding must be finite and nonzero")\n'
                    '    return value / norm\n'
                    '\n'
                    '\n'
                    'def short_voice_crop(segment, previous=None, following=None, media_duration=None):\n'
                    '    """Recover small timing gaps without including neighboring utterances."""\n'
                    '    start, end = segment.start, segment.end\n'
                    '    if end - start < 0.4:\n'
                    '        lower = previous.end if previous is not None else 0.0\n'
                    '        upper = following.start if following is not None else media_duration\n'
                    '        start = max(lower, start - 0.15)\n'
                    '        end = min(end + 0.15, upper) if upper is not None else end\n'
                    '        # Overlapping transcript boundaries are not safe padding opportunities.\n'
                    '        if start > segment.start or end < segment.end:\n'
                    '            return segment.start, segment.end\n'
                    '    return start, end\n'
                    '\n'
                    '\n'
                    'def add_question_response_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """A conversational hypothesis, never a police-specific identity rule."""\n'
                    '    if previous is None or segment.end - segment.start > 1.0:\n'
                    '        return\n'
                    '    gap = segment.start - previous.end\n'
                    '    if not 0.0 <= gap <= 0.6 or mapping_confidence < 0.75:\n'
                    '        return\n'
                    '    if previous.final_speaker in ("Uncertain", "NonTarget_Unknown", '
                    '"Unknown_Speaker"):\n'
                    '        return\n'
                    '    if previous.final_confidence < 0.65:\n'
                    '        return\n'
                    '    question = previous.text.strip().lower()\n'
                    '    # Narrow to addressed yes/no questions; punctuation alone is insufficient.\n'
                    '    direct_question = re.match(\n'
                    '        r"^(?:(?:ok|okay|all right)[.,]?\\s+)?"\n'
                    '        r"(?:do you|did you|have you|are you|were you|can you|could you|"\n'
                    '        r"would you|will you|don\'t you|didn\'t you|haven\'t you|aren\'t you)\\b", '
                    'question)\n'
                    '    if not question.endswith("?") or direct_question is None:\n'
                    '        return\n'
                    '    answer = re.sub(r"[^a-z\' ]", " ", segment.text.lower()).split()\n'
                    '    if not answer or len(answer) > 4 or answer[0] not in ("yes", "no", "yeah", "yep", '
                    '"nope", "nah"):\n'
                    '        return\n'
                    '    question_track = target_track if previous.final_speaker == "Target_Speaker" else '
                    'previous.final_speaker\n'
                    '    if question_track not in tracks:\n'
                    '        return\n'
                    '    candidates = sorted(track for track in tracks if track != question_track)\n'
                    '    candidate = candidates[0] if len(candidates) == 1 else None\n'
                    '    segment.evidence.append(Evidence("question_response", 0.0, 0.20,\n'
                    '        {"question_start": previous.start, "question_track": question_track,\n'
                    '         "candidate_tracks": candidates, "candidate_track": candidate,\n'
                    '         "gap": gap, "assumption": "Immediate brief answer may be a different speaker; '
                    'not voice-verified."}))\n'
                    '\n'
                    '\n'
                    '\n'
                    '\n'
                    'def add_brief_exchange_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """Tentative acknowledgement or confirmation, with independent voice agreement."""\n'
                    '    if previous is None or mapping_confidence < .75 or segment.end - segment.start >= '
                    '.4:\n'
                    '        return\n'
                    '    if not 0 <= segment.start - previous.end <= .6:\n'
                    '        return\n'
                    '    words = re.findall(r"[a-z\']+", segment.text.lower())\n'
                    '    preceding = re.findall(r"[a-z\']+", previous.text.lower())\n'
                    '    acknowledgement = words in (["okay"], ["ok"], ["oh", "okay"], ["oh", "ok"])\n'
                    '    confirmation = (preceding in (["really"], ["seriously"]) and '
                    'previous.text.strip().endswith("?")\n'
                    '                    and words in (["yes"], ["yeah"], ["yep"], ["no"], ["nope"]))\n'
                    '    if not acknowledgement and not confirmation:\n'
                    '        return\n'
                    '    previous_track = target_track if previous.final_speaker == "Target_Speaker" else '
                    'previous.final_speaker\n'
                    '    candidates = sorted(set(tracks) - {previous_track})\n'
                    '    if previous_track not in tracks or len(candidates) != 1:\n'
                    '        return\n'
                    '    if acknowledgement and (previous.final_confidence < .65 or '
                    'previous.text.strip().endswith("?")):\n'
                    '        return\n'
                    '    if confirmation:\n'
                    '        # A weak question is usable only when its baseline agrees, a target face is\n'
                    '        # tracked, and it was not itself attributed through conversational inference.\n'
                    '        face = next((e for e in previous.evidence if e.source == '
                    '"target_face_visible"), None)\n'
                    '        if (previous.final_confidence < .25 or previous.baseline.raw_speaker_track != '
                    'previous_track\n'
                    '            or previous_track != target_track or face is None\n'
                    '            or not face.details.get("target_visible_hint", False)\n'
                    '            or any("inference" in reason for reason in previous.reasons)):\n'
                    '            return\n'
                    '    voice = next((e for e in segment.evidence if e.source == "local_voice"), None)\n'
                    '    profiles = voice.details.get("track_similarities", {}) if voice is not None else '
                    '{}\n'
                    '    if (voice is None or voice.confidence > .30 or len(profiles) < 2\n'
                    '        or max(profiles.values()) >= .30 or voice.details.get("best_track") != '
                    'candidates[0]):\n'
                    '        return\n'
                    '    segment.evidence.append(Evidence("question_response", 0, .20,\n'
                    '        {"candidate_track": candidates[0], "previous_track": previous_track,\n'
                    '         "assumption": "Brief acknowledgement or confirmation may change speaker; weak '
                    'voice agrees, not verified."}))\n'
                    '\n'
                    '\n'
                    'def add_echo_question_evidence(segment, previous, tracks, target_track, '
                    'mapping_confidence):\n'
                    '    """A brief quoted question can suggest another speaker, never establish one."""\n'
                    '    if previous is None or not segment.text.strip().endswith("?"):\n'
                    '        return\n'
                    '    tokens = lambda text: re.findall(r"[a-z\']+", text.lower())\n'
                    '    phrase, statement = tokens(segment.text), tokens(previous.text)\n'
                    '    if not 2 <= len(phrase) <= 5 or statement[-len(phrase):] != phrase:\n'
                    '        return\n'
                    '    if not 0 <= segment.start - previous.end <= 0.8 or segment.end - segment.start > '
                    '1.2:\n'
                    '        return\n'
                    '    if previous.final_confidence < 0.65 or mapping_confidence < 0.75:\n'
                    '        return\n'
                    '    previous_track = target_track if previous.final_speaker == "Target_Speaker" else '
                    'previous.final_speaker\n'
                    '    candidates = sorted(set(tracks) - {previous_track})\n'
                    '    if previous_track not in tracks or len(candidates) != 1:\n'
                    '        return\n'
                    '    segment.evidence.append(Evidence("echo_question", 0, 0.20,\n'
                    '        {"candidate_track": candidates[0], "previous_track": previous_track,\n'
                    '         "assumption": "Brief repeated question may come from the listener; not '
                    'voice-verified."}))\n'
                    '\n'
                    '\n'
                    'def bbox_iou(left, right):\n'
                    '    x1, y1 = max(left[0], right[0]), max(left[1], right[1])\n'
                    '    x2, y2 = min(left[2], right[2]), min(left[3], right[3])\n'
                    '    intersection = max(0.0, x2-x1) * max(0.0, y2-y1)\n'
                    '    left_area = max(0.0, left[2]-left[0]) * max(0.0, left[3]-left[1])\n'
                    '    right_area = max(0.0, right[2]-right[0]) * max(0.0, right[3]-right[1])\n'
                    '    return intersection / max(left_area + right_area - intersection, 1e-9)\n'
                    '\n'
                    '\n'
                    'def collect_visual_evidence(segment, cap, fps, face_analyzer, target_face_centroid):\n'
                    '    """Track a recently recognized face through head turns; mouth motion is only a '
                    'hint."""\n'
                    '    if not np.isfinite(fps) or fps <= 0:\n'
                    '        segment.evidence.append(Evidence("visual_context", 0, 0, {"reason": '
                    '"invalid_fps"}))\n'
                    '        return\n'
                    '    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n'
                    '    # Lead-in frames establish identity; only frames inside speech measure mouth '
                    'motion.\n'
                    '    times = np.arange(max(0.0, segment.start - 0.5), segment.end, 0.125)\n'
                    '    indices = np.unique(np.rint(times * fps).astype(int))\n'
                    '    best, anchor_best, direct_matches, frames_read = None, None, 0, 0\n'
                    '    anchor = None\n'
                    '    observations, apertures, target_frames = [], [], []\n'
                    '    for index in indices:\n'
                    '        if frame_count > 0 and index >= frame_count:\n'
                    '            continue\n'
                    '        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n'
                    '        ok, frame = cap.read()\n'
                    '        if not ok:\n'
                    '            continue\n'
                    '        frames_read += 1\n'
                    '        time = index / fps\n'
                    '        candidates = []\n'
                    '        for face in face_analyzer.get(frame):\n'
                    '            embedding = getattr(face, "embedding", None)\n'
                    '            if embedding is None:\n'
                    '                continue\n'
                    '            embedding = normalize_vector(embedding)\n'
                    '            similarity = float(np.dot(target_face_centroid, embedding))\n'
                    '            if segment.start <= time <= segment.end:\n'
                    '                best = similarity if best is None else max(best, similarity)\n'
                    '            candidates.append((similarity, face, embedding))\n'
                    '        recognized = [item for item in candidates if item[0] >= 0.40]\n'
                    '        selected, identity_source = None, None\n'
                    '        if recognized:\n'
                    '            selected = max(recognized, key=lambda item: item[0])\n'
                    '            direct_matches += 1\n'
                    '            anchor_best = selected[0] if anchor_best is None else max(anchor_best, '
                    'selected[0])\n'
                    '            identity_source = "reference_match"\n'
                    '        elif anchor is not None and time - anchor[2] <= 0.30:\n'
                    '            linked = [item for item in candidates\n'
                    '                      if bbox_iou(item[1].bbox, anchor[0]) >= 0.20\n'
                    '                      and float(np.dot(item[2], anchor[1])) >= 0.45]\n'
                    '            if linked:\n'
                    '                selected = max(linked, key=lambda item: float(np.dot(item[2], '
                    'anchor[1])))\n'
                    '                identity_source = "face_continuity"\n'
                    '        for similarity, face, embedding in candidates:\n'
                    '            observations.append({"frame": int(index), "similarity": similarity,\n'
                    '                                 "bbox": face.bbox.tolist()})\n'
                    '        if selected is None:\n'
                    '            continue\n'
                    '        similarity, face, embedding = selected\n'
                    '        anchor = (face.bbox.copy(), embedding, time)\n'
                    '        if not segment.start <= time <= segment.end:\n'
                    '            continue\n'
                    '        target_frames.append({"frame": int(index), "identity_source": identity_source,\n'
                    '                              "similarity": similarity})\n'
                    '        landmarks = getattr(face, "landmark_3d_68", None)\n'
                    '        if landmarks is not None and np.all(np.isfinite(landmarks)):\n'
                    '            # Standard 68-point inner mouth: aperture / width in 3D landmark '
                    'coordinates.\n'
                    '            width = float(np.linalg.norm(landmarks[60] - landmarks[64]))\n'
                    '            if width > 1e-6:\n'
                    '                apertures.append(float(np.linalg.norm(landmarks[62] - landmarks[66]) / '
                    'width))\n'
                    '    spread = float(np.percentile(apertures, 90) - np.percentile(apertures, 10)) if '
                    'len(apertures) >= 5 else 0.0\n'
                    '    motion_hint = direct_matches >= 2 and len(apertures) >= 5 and spread >= 0.03\n'
                    '    segment.evidence.append(Evidence("target_face_visible", 0, 0,\n'
                    '        {"best_similarity": best, "identity_anchor_similarity": anchor_best,\n'
                    '         "target_visible_hint": bool(target_frames), "tracked_target_frames": '
                    'target_frames,\n'
                    '         "active_speaker_verified": False}))\n'
                    '    segment.evidence.append(Evidence("visual_context", 0, 0,\n'
                    '        {"frames_read": frames_read, "observations": observations,\n'
                    '         "note": "Face identity and visibility do not identify police or prove '
                    'speech."}))\n'
                    '    segment.evidence.append(Evidence("target_mouth_motion", 1.0 if motion_hint else '
                    '0.0,\n'
                    '        0.20 if motion_hint else 0.0,\n'
                    '        {"direct_identity_matches": direct_matches, "mouth_samples": len(apertures),\n'
                    '         "aperture_spread": spread, "active_speaker_verified": False,\n'
                    '         "note": "Weak landmark motion hint; no lipreading or audio-visual '
                    'synchronization model."}))\n'
                    '\n'
                    '\n'
                    'def resolve_segment(segment, target_track, mapping_confidence):\n'
                    '    """Only the resolver assigns final identity; visibility alone cannot flip it."""\n'
                    '    raw = segment.baseline.raw_speaker_track\n'
                    '    known = raw != "Unknown_Speaker"\n'
                    '    prior_weight = 0.55 * mapping_confidence if known else 0.0\n'
                    '    prior = 1.0 if raw == target_track else -1.0\n'
                    '    score, weight = prior * prior_weight, prior_weight\n'
                    '    reasons = [f"baseline={raw}; mapping strength={mapping_confidence:.3f}"]\n'
                    '    voice = None\n'
                    '    response = None\n'
                    '    mouth_motion = None\n'
                    '    echo = None\n'
                    '    visible = None\n'
                    '    for item in segment.evidence:\n'
                    '        # Presence/context describes the scene, not the active speaker.\n'
                    '        if item.source in ("target_face_visible", "visual_context"):\n'
                    '            if item.source == "target_face_visible":\n'
                    '                visible = item\n'
                    '            continue\n'
                    '        if item.source == "echo_question":\n'
                    '            echo = item\n'
                    '            continue\n'
                    '        if item.source == "target_mouth_motion":\n'
                    '            mouth_motion = item\n'
                    '            continue\n'
                    '        if item.source == "question_response":\n'
                    '            response = item\n'
                    '            continue\n'
                    '        contribution = item.target_score * item.confidence\n'
                    '        score += contribution\n'
                    '        weight += item.confidence\n'
                    '        if item.source == "local_voice":\n'
                    '            voice = item\n'
                    '        if item.confidence:\n'
                    '            reasons.append(f"{item.source}: {contribution:+.3f}")\n'
                    '    normalized = score / max(weight, 1e-9)\n'
                    '    # Role phrases cannot establish or contradict identity without acoustic support.\n'
                    '    acoustic_support = prior_weight > 0.08 or (voice is not None and voice.confidence > '
                    '0.2)\n'
                    '    strong_conflict = (voice is not None and voice.confidence >= 0.5\n'
                    '                      and voice.target_score * prior < -0.4 and known)\n'
                    '    # A strong reference match plus an independent track match can correct '
                    'diarization.\n'
                    '    details = voice.details if voice is not None else {}\n'
                    '    matched_track = details.get("best_track")\n'
                    '    verified_correction = (strong_conflict and voice.confidence >= 0.5\n'
                    '                          and abs(voice.target_score) >= 0.4\n'
                    '                          and details.get("track_margin", 0.0) >= 0.10\n'
                    '                          and matched_track is not None\n'
                    '                          and details.get("track_similarities", {}).get(matched_track, '
                    '-1.0) >= 0.30\n'
                    '                          and ((voice.target_score > 0 and matched_track == '
                    'target_track)\n'
                    '                               or (voice.target_score < 0 and matched_track != '
                    'target_track)))\n'
                    '    insufficient_short_audio = (segment.end - segment.start < 0.4\n'
                    '                                and (voice is None or voice.confidence == 0))\n'
                    '    response_track = response.details.get("candidate_track") if response is not None '
                    'else None\n'
                    '    profiles = details.get("track_similarities", {})\n'
                    '    weak_padded_voice = (segment.end - segment.start < 0.4 and voice is not None\n'
                    '                         and voice.confidence <= 0.30 and len(profiles) >= 2\n'
                    '                         and max(profiles.values()) < 0.30)\n'
                    '    response_inference = ((insufficient_short_audio or weak_padded_voice) and '
                    'response_track is not None\n'
                    '                          and response.confidence > 0)\n'
                    '    visual_inference = (mouth_motion is not None and mouth_motion.confidence > 0\n'
                    '                        and voice is not None and voice.confidence > 0\n'
                    '                        and mapping_confidence >= 0.75\n'
                    '                        and len(details.get("track_similarities", {})) >= 2\n'
                    '                        and details.get("track_margin", 1.0) < 0.05\n'
                    '                        and max(details["track_similarities"].values()) < 0.35)\n'
                    '    echo_inference = (echo is not None and echo.details["candidate_track"] == '
                    'target_track\n'
                    '                      and visible is not None and '
                    'visible.details.get("target_visible_hint", False)\n'
                    '                      and len(profiles) >= 2 and max(profiles.values()) < 0.30\n'
                    '                      and matched_track == target_track and voice.confidence < 0.5)\n'
                    '    if echo_inference:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("weak repeated-question inference with face continuity and weak '
                    'supporting voice profile; not voice-verified")\n'
                    '    elif visual_inference:\n'
                    '        final = "Target_Speaker"\n'
                    '        reasons.append("weak visible-target mouth-motion inference; independent voice '
                    'profiles are ambiguous")\n'
                    '    elif response_inference:\n'
                    '        final = "Target_Speaker" if response_track == target_track else response_track\n'
                    '        reasons.append(f"weak question/answer inference to {response_track}; not '
                    'voice-verified")\n'
                    '    elif verified_correction:\n'
                    '        final = "Target_Speaker" if matched_track == target_track else matched_track\n'
                    '        reasons.append(f"independent voice profile supports correction to '
                    '{matched_track}")\n'
                    '    elif insufficient_short_audio or not acoustic_support or abs(normalized) <= 0.18 or '
                    'strong_conflict:\n'
                    '        final = "Uncertain"\n'
                    '        reasons.append("weak, balanced, or conflicting acoustic evidence")\n'
                    '    elif normalized > 0:\n'
                    '        final = "Target_Speaker"\n'
                    '    else:\n'
                    '        final = raw if known and raw != target_track else "NonTarget_Unknown"\n'
                    '    segment.final_speaker = final\n'
                    '    # Avoid baseline-only certainty and account for weak global separation.\n'
                    '    segment.final_confidence = float(min(abs(normalized), weight / 1.55))\n'
                    '    if verified_correction:\n'
                    '        segment.final_confidence = float(min(voice.confidence, '
                    'abs(voice.target_score),\n'
                    '                                             details["track_margin"] / 0.20))\n'
                    '    if echo_inference:\n'
                    '        segment.final_confidence = 0.20\n'
                    '    elif visual_inference:\n'
                    '        segment.final_confidence = 0.25\n'
                    '    elif response_inference:\n'
                    '        segment.final_confidence = min(0.25, response.confidence)\n'
                    '    elif insufficient_short_audio:\n'
                    '        segment.final_confidence = 0.0\n'
                    '        reasons.append("short utterance has no usable local voice evidence")\n'
                    '    if segment.end - segment.start < 0.4:\n'
                    '        segment.final_confidence = min(segment.final_confidence, 0.30)\n'
                    '    segment.reasons = reasons\n'
                    '\n'
                    '\n'
                    'def main():\n'
                    '    parser = argparse.ArgumentParser(description="Resolve a supplied target voice in '
                    'any video.")\n'
                    '    parser.add_argument("video", nargs="?", default="short.mp4")\n'
                    '    parser.add_argument("--voice-priors", default="voice_embeddings.npy")\n'
                    '    parser.add_argument("--face-priors", default="face_embeddings.npy")\n'
                    '    parser.add_argument("--output", default="diarization_evidence.json")\n'
                    '    parser.add_argument("--batch-size", type=int, default=16)\n'
                    '    parser.add_argument("--cache-dir", help="Reuse completed stages for matching '
                    'inputs/code/runtime")\n'
                    '    parser.add_argument("--transcription-coverage", choices=("vad", "full"), '
                    'default="vad",\n'
                    '                        help="Experimental full coverage includes noise/silence; review '
                    'for hallucinations")\n'
                    '    args = parser.parse_args()\n'
                    '    if args.batch_size < 1:\n'
                    '        parser.error("--batch-size must be positive")\n'
                    '    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")\n'
                    '    if not HF_TOKEN:\n'
                    '        raise RuntimeError("Set HF_TOKEN (or HUGGINGFACE_TOKEN) before running.")\n'
                    '    device = "cuda" if torch.cuda.is_available() else "cpu"\n'
                    '    compute_type = "float16" if torch.cuda.is_available() else "int8"\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 1. CORE PIPELINE INITIALIZATION\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Initializing Core Tracking Engines...")\n'
                    '    def release_gpu():\n'
                    '        gc.collect()\n'
                    '        if device == "cuda":\n'
                    '            torch.cuda.empty_cache()\n'
                    '\n'
                    '    fingerprint = {"inputs": {name: file_digest(path) for name, path in\n'
                    '        (("video", args.video), ("voice", args.voice_priors), ("face", '
                    'args.face_priors))},\n'
                    '        "code": file_digest(__file__), "runtime_helper": '
                    'file_digest(Path(__file__).with_name("cloud_runtime.py")),\n'
                    '        "device": device, "batch_size": args.batch_size,\n'
                    '        "versions": {name: version(name) for name in\n'
                    '            ("torch", "torchaudio", "whisperx", "speechbrain", "insightface", '
                    '"numpy")},\n'
                    '        "onnxruntime": ort.__version__, "providers": ort.get_available_providers()}\n'
                    '    cache = StageCache(args.cache_dir, fingerprint)\n'
                    '    print(f"WhisperX/SpeechBrain device: {device}")\n'
                    '\n'
                    '    # Load Priors Matrix\n'
                    '    voice_priors = np.load(args.voice_priors, allow_pickle=False)\n'
                    '    face_priors = np.load(args.face_priors, allow_pickle=False)\n'
                    '    if voice_priors.ndim == 1: voice_priors = np.expand_dims(voice_priors, axis=0)\n'
                    '    if face_priors.ndim == 1: face_priors = np.expand_dims(face_priors, axis=0)\n'
                    '\n'
                    '    def reference_centroid(samples, label):\n'
                    '        if samples.ndim != 2:\n'
                    '            raise ValueError(f"{label}: expected a vector or matrix of target '
                    'samples")\n'
                    '        norms = np.linalg.norm(samples, axis=1)\n'
                    '        valid = np.all(np.isfinite(samples), axis=1) & (norms > 0)\n'
                    '        if not np.any(valid):\n'
                    '            raise ValueError(f"{label}: no valid target samples")\n'
                    '        normalized = samples[valid] / norms[valid, None]\n'
                    '        print(f"{label}: using {len(normalized)} of {len(samples)} target reference '
                    'samples")\n'
                    '        return normalize_vector(np.mean(normalized, axis=0))\n'
                    '\n'
                    '    target_voice_vector = reference_centroid(voice_priors, "Voice references")\n'
                    '    target_face_centroid = reference_centroid(face_priors, "Face references")\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 2. AUDIO & VIDEO DATA PREP\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Preparing media data tracks...")\n'
                    '    video_path = args.video\n'
                    '    audio_loaded = whisperx.load_audio(video_path)\n'
                    '    cap = cv2.VideoCapture(video_path)\n'
                    '    fps = cap.get(cv2.CAP_PROP_FPS)\n'
                    '\n'
                    '    waveform, sample_rate = torchaudio.load(video_path)\n'
                    '    if sample_rate != 16000:\n'
                    '        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, '
                    'new_freq=16000)\n'
                    '        waveform = resampler(waveform)\n'
                    '    waveform = torch.mean(waveform, dim=0, keepdim=True)\n'
                    '    total_samples = waveform.shape[1]\n'
                    '\n'
                    '    # =====================================================================\n'
                    '    # 3. GENERAL TRANSCRIPTION & LAYERING\n'
                    '    # =====================================================================\n'
                    '    print("⏳ Processing WhisperX Text Script...")\n'
                    '    def transcribe():\n'
                    '        options = {"vad_model": create_full_audio_vad()} if args.transcription_coverage '
                    '== "full" else {}\n'
                    '        model = whisperx.load_model("large-v2", device, compute_type=compute_type, '
                    '**options)\n'
                    '        try:\n'
                    '            return model.transcribe(audio_loaded, batch_size=args.batch_size)\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    asr_result = cache.get("transcription_" + args.transcription_coverage, transcribe)\n'
                    '\n'
                    '    def align():\n'
                    '        model, metadata = '
                    'whisperx.load_align_model(language_code=asr_result["language"], device=device)\n'
                    '        try:\n'
                    '            return whisperx.align(asr_result["segments"], model, metadata, '
                    'audio_loaded,\n'
                    '                                  device, return_char_alignments=False)\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    aligned_result = cache.get("alignment_" + args.transcription_coverage, align)\n'
                    '    print("⏳ Generating Unsupervised Voice Tracks...")\n'
                    '\n'
                    '    def diarize():\n'
                    '        model = DiarizationPipeline(token=HF_TOKEN, device=device)\n'
                    '        try:\n'
                    '            # Preserve original whole-video track IDs; no independent chunk '
                    'clustering.\n'
                    '            return model(audio_loaded)[["start", "end", '
                    '"speaker"]].to_dict(orient="records")\n'
                    '        finally:\n'
                    '            del model\n'
                    '            release_gpu()\n'
                    '\n'
                    '    diarize_segments = pd.DataFrame(cache.get("diarization", diarize))\n'
                    '    embedding_model = SpeakerRecognition.from_hparams(\n'
                    '        source="speechbrain/spkrec-ecapa-voxceleb", '
                    'savedir="pretrained_models/spkrec-ecapa-voxceleb",\n'
                    '        run_opts={"device": device})\n'
                    '    face_analyzer, face_providers = create_face_analyzer(device, ort, FaceAnalysis)\n'
                    '\n'
                    '    target_voice_vector = normalize_vector(target_voice_vector)\n'
                    '    target_face_centroid = normalize_vector(target_face_centroid)\n'
                    '    embedding_cache = {}\n'
                    '\n'
                    '    def audio_embedding(start, end):\n'
                    '        key = (float(start), float(end))\n'
                    '        if key in embedding_cache:\n'
                    '            return embedding_cache[key]\n'
                    '        left = max(0, int(start * 16000))\n'
                    '        right = min(total_samples, int(end * 16000))\n'
                    '        checkpoint_name = f"voice_{left}_{right}"\n'
                    '        saved = cache.read(checkpoint_name)\n'
                    '        if saved is not None:\n'
                    '            result = np.asarray(saved, dtype=np.float32)\n'
                    '            embedding_cache[key] = result\n'
                    '            return result\n'
                    '        result = None\n'
                    '        if right - left >= 6400:\n'
                    '            try:\n'
                    '                with torch.no_grad():\n'
                    '                    result = normalize_vector(embedding_model.encode_batch(\n'
                    '                        waveform[:, left:right].to(device)).flatten().cpu().numpy())\n'
                    '                if result.shape != target_voice_vector.shape:\n'
                    '                    raise ValueError("Voice prior dimensions do not match ECAPA '
                    'output")\n'
                    '            except ValueError:\n'
                    '                raise\n'
                    '            except Exception as exc:\n'
                    '                print(f"Voice embedding unavailable at {start:.2f}-{end:.2f}: {exc}")\n'
                    '        if result is not None:\n'
                    '            cache.write(checkpoint_name, result.tolist())\n'
                    '        embedding_cache[key] = result\n'
                    '        return result\n'
                    '\n'
                    '    cluster_scores = defaultdict(list)\n'
                    '    cluster_embeddings = defaultdict(list)\n'
                    '    for _, row in diarize_segments.iterrows():\n'
                    '        start, end = float(row["start"]), float(row["end"])\n'
                    '        if end - start < 0.6:\n'
                    '            continue\n'
                    '        emb = audio_embedding(start, end)\n'
                    '        if emb is not None:\n'
                    '            track = str(row["speaker"])\n'
                    '            cluster_scores[track].append(float(np.dot(target_voice_vector, emb)))\n'
                    '            cluster_embeddings[track].append((start, end, emb))\n'
                    '    means = {track: float(np.mean(scores)) for track, scores in '
                    'cluster_scores.items()}\n'
                    '    ranked = sorted(means, key=means.get, reverse=True)\n'
                    '    target_track = ranked[0] if ranked else None\n'
                    '    target_mean = means[target_track] if ranked else 0.0\n'
                    '    # No invented competitor when only one cluster has usable speech.\n'
                    '    other_mean = means[ranked[1]] if len(ranked) > 1 else None\n'
                    '    separation = target_mean - other_mean if other_mean is not None else 0.0\n'
                    '    mapping_confidence = min(1.0, max(0.0, separation / 0.15))\n'
                    '    print("\\n--- Baseline voice affinity ---")\n'
                    '    for track in ranked:\n'
                    '        print(f"{track}: {means[track]:.3f} ({len(cluster_scores[track])} samples)")\n'
                    '    print(f"Target candidate: {target_track}; mapping '
                    'strength={mapping_confidence:.3f}")\n'
                    '\n'
                    '    assigned = whisperx.assign_word_speakers(diarize_segments, aligned_result)\n'
                    '    timeline = []\n'
                    '    for source in assigned["segments"]:\n'
                    '        raw = str(source.get("speaker") or "Unknown_Speaker")\n'
                    '        base = "Target_Speaker" if raw == target_track else ("Unknown" if raw == '
                    '"Unknown_Speaker" else raw)\n'
                    '        timeline.append(TimelineSegment(float(source["start"]), float(source["end"]),\n'
                    '                        source["text"].strip(), Baseline(raw, base), '
                    'source.get("words", [])))\n'
                    '\n'
                    '    def collect_voice(segment, previous=None, following=None):\n'
                    '        crop_start, crop_end = short_voice_crop(segment, previous, following, '
                    'total_samples / 16000)\n'
                    '        emb = audio_embedding(crop_start, crop_end)\n'
                    '        similarity = float(np.dot(target_voice_vector, emb)) if emb is not None else '
                    'None\n'
                    '        strength = min(1.0, max(0.0, (segment.end - segment.start) / 1.2)) * '
                    'mapping_confidence\n'
                    '        if segment.end - segment.start < 0.4:\n'
                    '            strength = min(strength, 0.30)\n'
                    '        target_score = 0.0\n'
                    '        if similarity is not None and separation >= 0.03:\n'
                    '            target_score = float(np.clip((similarity - (target_mean + other_mean) / 2) '
                    '/ separation, -1, 1))\n'
                    '        else:\n'
                    '            strength = 0.0\n'
                    '        # Exclude intersecting speech from profiles so a segment cannot validate '
                    'itself.\n'
                    '        track_similarities = {}\n'
                    '        profile_counts = {}\n'
                    '        if emb is not None:\n'
                    '            for track, samples in cluster_embeddings.items():\n'
                    '                independent = [vector for start, end, vector in samples\n'
                    '                               if end <= crop_start or start >= crop_end]\n'
                    '                if len(independent) < 2:\n'
                    '                    continue\n'
                    '                centroid = normalize_vector(np.mean(independent, axis=0))\n'
                    '                track_similarities[track] = float(np.dot(emb, centroid))\n'
                    '                profile_counts[track] = len(independent)\n'
                    '        candidates = sorted(track_similarities, key=track_similarities.get, '
                    'reverse=True)\n'
                    '        best_track = candidates[0] if len(candidates) >= 2 else None\n'
                    '        margin = (track_similarities[candidates[0]] - '
                    'track_similarities[candidates[1]]\n'
                    '                  if len(candidates) >= 2 else 0.0)\n'
                    '        segment.evidence.append(Evidence("local_voice", target_score, strength,\n'
                    '            {"similarity": similarity, "crop_start": crop_start, "crop_end": crop_end,\n'
                    '             "target_mean": target_mean, "competitor_mean": other_mean,\n'
                    '             "track_similarities": track_similarities, "independent_profile_counts": '
                    'profile_counts,\n'
                    '             "best_track": best_track, "track_margin": margin}))\n'
                    '\n'
                    '    def collect_visual(segment):\n'
                    '        name = f"visual_{segment.start:.6f}_{segment.end:.6f}"\n'
                    '        saved = cache.read(name)\n'
                    '        if saved is not None:\n'
                    '            segment.evidence.extend(Evidence(**item) for item in saved)\n'
                    '            return\n'
                    '        offset = len(segment.evidence)\n'
                    '        collect_visual_evidence(segment, cap, fps, face_analyzer, '
                    'target_face_centroid)\n'
                    '        cache.write(name, [asdict(item) for item in segment.evidence[offset:]])\n'
                    '\n'
                    '    def collect_semantic(segment):\n'
                    '        # Without an explicit role-to-identity mapping, words cannot identify a '
                    'person.\n'
                    '        # Keep semantic/context observations neutral for arbitrary videos and targets.\n'
                    '        segment.evidence.append(Evidence("semantic_context", 0.0, 0.0,\n'
                    '            {"identity_mapping": None,\n'
                    '             "note": "No role or phrase is assumed to identify the supplied '
                    'target."}))\n'
                    '\n'
                    '    try:\n'
                    '        all_tracks = {str(track) for track in '
                    'diarize_segments["speaker"].dropna().unique()}\n'
                    '        for index, segment in enumerate(timeline):\n'
                    '            previous = timeline[index - 1] if index else None\n'
                    '            following = timeline[index + 1] if index + 1 < len(timeline) else None\n'
                    '            collect_voice(segment, previous, following)\n'
                    '            collect_visual(segment)\n'
                    '            collect_semantic(segment)\n'
                    '            add_question_response_evidence(segment, previous, all_tracks, target_track, '
                    'mapping_confidence)\n'
                    '            add_brief_exchange_evidence(segment, previous, all_tracks, target_track, '
                    'mapping_confidence)\n'
                    '            add_echo_question_evidence(segment, previous, all_tracks, target_track, '
                    'mapping_confidence)\n'
                    '            resolve_segment(segment, target_track, mapping_confidence)\n'
                    '            print(f"Resolved segment {index + 1}/{len(timeline)} at '
                    '{segment.end:.1f}s", flush=True)\n'
                    '        print("\\n--- Evidence-Based Speaker Resolution ---")\n'
                    '        for segment in timeline:\n'
                    '            print(f"[{segment.start:.2f}s - {segment.end:.2f}s] {segment.final_speaker} '
                    '"\n'
                    '                  f"(strength={segment.final_confidence:.2f}): {segment.text}")\n'
                    '            print(f"    baseline={segment.baseline.speaker}; '
                    'raw={segment.baseline.raw_speaker_track}")\n'
                    '            for item in segment.evidence:\n'
                    '                print(f"    {item.source}: score={item.target_score:+.2f}, '
                    'strength={item.confidence:.2f}, {item.details}")\n'
                    '        with open(args.output, "w", encoding="utf-8") as output:\n'
                    '            json.dump({"target_candidate": target_track, "cluster_voice_means": means,\n'
                    '                       "mapping_strength": mapping_confidence,\n'
                    '                       "runtime": {"device": device, "face_providers": face_providers,\n'
                    '                                   "transcription_coverage": '
                    'args.transcription_coverage},\n'
                    '                       "confidence_is_calibrated": False,\n'
                    '                       "segments": [asdict(segment) for segment in timeline]}, output, '
                    'indent=2, ensure_ascii=False)\n'
                    '    finally:\n'
                    '        cap.release()\n'
                    '\n'
                    '\n'
                    'if __name__ == "__main__":\n'
                    '    main()\n',
 'cloud_runtime.py': '"""Small runtime helpers; attribution rules do not live here."""\n'
                     'import hashlib\n'
                     'import json\n'
                     'import os\n'
                     'from pathlib import Path\n'
                     '\n'
                     '\n'
                     'def file_digest(path):\n'
                     '    digest = hashlib.sha256()\n'
                     '    with open(path, "rb") as stream:\n'
                     '        for block in iter(lambda: stream.read(1024 * 1024), b""):\n'
                     '            digest.update(block)\n'
                     '    return digest.hexdigest()\n'
                     '\n'
                     '\n'
                     'class StageCache:\n'
                     '    """Atomic, JSON-only checkpoints, isolated by inputs/code/runtime fingerprint."""\n'
                     '    def __init__(self, directory, fingerprint):\n'
                     '        self.root = None\n'
                     '        if directory:\n'
                     '            key = hashlib.sha256(json.dumps(fingerprint, '
                     'sort_keys=True).encode()).hexdigest()\n'
                     '            self.root = Path(directory) / key\n'
                     '            self.root.mkdir(parents=True, exist_ok=True)\n'
                     '            self.write("manifest", fingerprint)\n'
                     '\n'
                     '    def read(self, name):\n'
                     '        if self.root is None:\n'
                     '            return None\n'
                     '        path = self.root / (name + ".json")\n'
                     '        if not path.exists():\n'
                     '            return None\n'
                     '        return json.loads(path.read_text())\n'
                     '\n'
                     '    def write(self, name, value):\n'
                     '        if self.root is None:\n'
                     '            return\n'
                     '        path = self.root / (name + ".json")\n'
                     '        temporary = path.with_suffix(".tmp")\n'
                     '        temporary.write_text(json.dumps(value, ensure_ascii=False))\n'
                     '        os.replace(temporary, path)\n'
                     '\n'
                     '    def get(self, name, compute):\n'
                     '        value = self.read(name)\n'
                     '        if value is not None:\n'
                     '            print(f"Reusing checkpoint: {name}")\n'
                     '            return value\n'
                     '        value = compute()\n'
                     '        self.write(name, value)\n'
                     '        return value\n'
                     '\n'
                     '\n'
                     'def create_face_analyzer(device, ort, factory):\n'
                     '    if device == "cuda" and hasattr(ort, "preload_dlls"):\n'
                     '        ort.preload_dlls()\n'
                     '    use_cuda = device == "cuda" and "CUDAExecutionProvider" in '
                     'ort.get_available_providers()\n'
                     '    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if use_cuda else '
                     '["CPUExecutionProvider"]\n'
                     '    analyzer = factory(name="buffalo_l", providers=providers)\n'
                     '    analyzer.prepare(ctx_id=0 if use_cuda else -1, det_size=(640, 640))\n'
                     '    actual = {name: model.session.get_providers() for name, model in '
                     'analyzer.models.items()\n'
                     '              if getattr(model, "session", None) is not None}\n'
                     '    print(f"Face analysis actual providers: {actual}")\n'
                     '    if device == "cuda" and (not actual or any("CUDAExecutionProvider" not in value '
                     'for value in actual.values())):\n'
                     '        print("WARNING: one or more face models are using CPU; check '
                     'onnxruntime-gpu/CUDA libraries.")\n'
                     '    return analyzer, actual\n'
                     '\n'
                     '\n'
                     'def full_audio_chunks(sample_count, sample_rate, chunk_size=30):\n'
                     '    """Cover every sample with bounded windows; do not infer whether it is speech."""\n'
                     '    if sample_count <= 0 or sample_rate <= 0 or chunk_size <= 0:\n'
                     '        raise ValueError("Audio length, sample rate and chunk size must be positive")\n'
                     '    step = max(1, int(sample_rate * chunk_size))\n'
                     '    return [{"start": left / sample_rate, "end": min(left + step, sample_count) / '
                     'sample_rate}\n'
                     '            for left in range(0, sample_count, step)]\n'
                     '\n'
                     '\n'
                     'def create_full_audio_vad():\n'
                     '    """WhisperX coverage adapter for controlled experiments, not a speech '
                     'detector."""\n'
                     '    from whisperx.vads.vad import Vad\n'
                     '\n'
                     '    class FullAudio(Vad):\n'
                     '        @staticmethod\n'
                     '        def preprocess_audio(audio):\n'
                     '            return audio\n'
                     '\n'
                     '        def __call__(self, inputs):\n'
                     '            return {"sample_count": len(inputs["waveform"]), "sample_rate": '
                     'inputs["sample_rate"]}\n'
                     '\n'
                     '        @staticmethod\n'
                     '        def merge_chunks(segments, chunk_size, onset, offset):\n'
                     '            return full_audio_chunks(segments["sample_count"], '
                     'segments["sample_rate"], chunk_size)\n'
                     '\n'
                     '    return FullAudio(.5)\n',
 'recover_transcript_gaps.py': '"""Recover review candidates only inside uncovered transcript intervals.\n'
                               '\n'
                               'Existing segments are copied unchanged. Candidates are separate, have no '
                               'speaker\n'
                               'identity, and require review. Detection of a transcript gap does not prove '
                               'speech.\n'
                               '"""\n'
                               'import argparse\n'
                               'import copy\n'
                               'import hashlib\n'
                               'import json\n'
                               'import math\n'
                               'from pathlib import Path\n'
                               'import re\n'
                               'import subprocess\n'
                               'import numpy as np\n'
                               '\n'
                               '\n'
                               'def uncovered_intervals(segments, duration, minimum_gap=2.0):\n'
                               '    cursor=0.0; gaps=[]\n'
                               "    for segment in sorted(segments,key=lambda s:s['start']):\n"
                               "        start=max(0.0,min(duration,float(segment['start'])))\n"
                               "        end=max(start,min(duration,float(segment['end'])))\n"
                               '        if start-cursor>=minimum_gap: gaps.append((cursor,start))\n'
                               '        cursor=max(cursor,end)\n'
                               '    if duration-cursor>=minimum_gap:gaps.append((cursor,duration))\n'
                               '    return gaps\n'
                               '\n'
                               '\n'
                               'def recovery_windows(gap,duration,size=25.0,overlap=12.0,context=.5):\n'
                               '    if not all(math.isfinite(x) for x in (size,overlap)) or size<=0 or not '
                               "0<=overlap<size:raise ValueError('Invalid window size/overlap')\n"
                               "    if not math.isfinite(context) or context<0:raise ValueError('Context "
                               "must be finite and nonnegative')\n"
                               '    left=max(0.,gap[0]-context); right=min(duration,gap[1]+context)\n'
                               '    if right-left<=size:return [(left,right)]\n'
                               '    starts=[];start=left\n'
                               '    while start+size<right:\n'
                               '        starts.append(start); start+=size-overlap\n'
                               '    final=max(left,right-size)\n'
                               '    if not starts or abs(final-starts[-1])>1e-6:starts.append(final)\n'
                               '    return [(s,min(s+size,right)) for s in starts]\n'
                               '\n'
                               '\n'
                               "def word_key(text):return re.sub(r'[^\\w]+','',text.casefold())\n"
                               '\n'
                               '\n'
                               'def collect_candidates(observations,gap):\n'
                               '    # Retain all eligible words for review; repeated words need distinct '
                               'windows.\n'
                               '    clusters=[]\n'
                               '    for word in sorted(observations,key=lambda '
                               "w:(w['start'],w['window_index'])):\n"
                               "        if word['start']<gap[0] or word['end']>gap[1] or "
                               "word['end']<word['start']:continue\n"
                               "        key=word_key(word['word'])\n"
                               '        if not key:continue\n'
                               "        matches=[c for c in clusters if c['key']==key and "
                               "abs(c['anchor']-(word['start']+word['end'])/2)<=.6 and word['window_index'] "
                               "not in {x['window_index'] for x in c['observations']}]\n"
                               '        if matches:\n'
                               '            closest=min(matches,key=lambda '
                               "c:abs(c['anchor']-(word['start']+word['end'])/2));closest['observations'].append(word)\n"
                               '        '
                               "else:clusters.append({'key':key,'anchor':(word['start']+word['end'])/2,'observations':[word]})\n"
                               '    # Keep words from one decoder window together; never splice hypotheses.\n'
                               '    support={}\n'
                               '    for cluster in clusters:\n'
                               "        indices=sorted({w['window_index'] for w in "
                               "cluster['observations']})\n"
                               "        for w in cluster['observations']:\n"
                               '            '
                               "support[(w['window_index'],w['start'],w['end'],w['word'])]=indices\n"
                               '    hypotheses=[]\n'
                               "    for index in sorted({w['window_index'] for w in observations}):\n"
                               '        runs=[]\n'
                               "        for w in sorted((w for w in observations if w['window_index']==index "
                               "and w['start']>=gap[0] and w['end']<=gap[1] and w['end']>=w['start'] and "
                               "word_key(w['word'])),key=lambda w:w['start']):\n"
                               '            '
                               "word=dict(w,supporting_windows=support.get((index,w['start'],w['end'],w['word']),[index]))\n"
                               "            if runs and word['start']-runs[-1]['end']<=.65:\n"
                               '                '
                               "runs[-1]['words'].append(word);runs[-1]['end']=max(runs[-1]['end'],word['end'])\n"
                               '            '
                               "else:runs.append({'start':word['start'],'end':word['end'],'words':[word],'window_index':index})\n"
                               '        for run in runs:\n'
                               "            if run['end']<=run['start']:continue\n"
                               "            repeated=sum(len(w['supporting_windows'])>=2 for w in "
                               "run['words'])\n"
                               "            run.update(text=''.join(w['word'] for w in "
                               "run['words']).strip(),supported_word_count=repeated,supported_word_fraction=repeated/len(run['words']),repeated_in_overlapping_windows=repeated/len(run['words'])>=.5,review_required=True,speaker='Uncertain')\n"
                               '            hypotheses.append(run)\n'
                               '    selected=[]\n'
                               '    for run in sorted(hypotheses,key=lambda '
                               "r:(r['supported_word_count'],sum(w['probability'] for w in "
                               "r['words'])/len(r['words']),r['end']-r['start']),reverse=True):\n"
                               '        if '
                               "any(min(run['end'],chosen['end'])-max(run['start'],chosen['start'])>.15 for "
                               'chosen in selected):continue\n'
                               '        selected.append(run)\n'
                               "    return sorted(selected,key=lambda r:r['start'])\n"
                               '\n'
                               '\n'
                               'def preserve_baseline(baseline,candidates):\n'
                               '    result=copy.deepcopy(baseline)\n'
                               "    result['gap_recovery_candidates']=copy.deepcopy(candidates)\n"
                               "    result['gap_recovery_note']='Existing segments unchanged; candidates "
                               'require review and have no attributed speaker. Repeated decoding is not '
                               "ground truth.'\n"
                               '    return result\n'
                               '\n'
                               '\n'
                               'def main():\n'
                               '    parser=argparse.ArgumentParser(description=__doc__)\n'
                               "    parser.add_argument('video',type=Path)\n"
                               "    parser.add_argument('--baseline',required=True,type=Path)\n"
                               "    parser.add_argument('--output-dir',required=True,type=Path)\n"
                               "    parser.add_argument('--minimum-gap',type=float,default=2.)\n"
                               "    parser.add_argument('--window-seconds',type=float,default=25.)\n"
                               "    parser.add_argument('--overlap-seconds',type=float,default=12.)\n"
                               '    '
                               "parser.add_argument('--context-seconds',type=float,default=.5,help='Audio "
                               "context on each side; words outside the gap are never added')\n"
                               "    parser.add_argument('--language',default='en',help='Language of the "
                               "working transcript')\n"
                               '    args=parser.parse_args()\n'
                               "    if args.output_dir.exists():parser.error('Choose a new output directory; "
                               "existing results are never overwritten')\n"
                               '    if not math.isfinite(args.minimum_gap) or '
                               "args.minimum_gap<=0:parser.error('Minimum gap must be positive')\n"
                               '    '
                               'try:recovery_windows((0.,1.),1.,args.window_seconds,args.overlap_seconds,args.context_seconds)\n'
                               '    except ValueError as error:parser.error(str(error))\n'
                               '    baseline=json.loads(args.baseline.read_text())\n'
                               '    '
                               "raw=subprocess.check_output(['ffmpeg','-nostdin','-hide_banner','-loglevel','error','-i',str(args.video),'-vn','-ar','16000','-ac','1','-f','f32le','pipe:1'])\n"
                               "    audio=np.frombuffer(raw,dtype='<f4').copy();duration=len(audio)/16000\n"
                               '    '
                               "gaps=uncovered_intervals(baseline['segments'],duration,args.minimum_gap)\n"
                               '    args.output_dir.mkdir(parents=True)\n'
                               '    candidates=[];decodes=[];model=None;device=None\n'
                               '    if gaps:\n'
                               '        import torch\n'
                               '        from faster_whisper import WhisperModel\n'
                               "        device='cuda' if torch.cuda.is_available() else 'cpu'\n"
                               "        model=WhisperModel('large-v2',device=device,compute_type='float16' "
                               "if device=='cuda' else 'int8',cpu_threads=4)\n"
                               '    for gap_index,gap in enumerate(gaps):\n'
                               '        observations=[]\n'
                               '        for window_index,(left,right) in '
                               'enumerate(recovery_windows(gap,duration,args.window_seconds,args.overlap_seconds,args.context_seconds)):\n'
                               '            '
                               'segments,info=model.transcribe(audio[int(left*16000):int(right*16000)],language=args.language,vad_filter=False,beam_size=5,condition_on_previous_text=False,word_timestamps=True)\n'
                               '            rows=[]\n'
                               '            for segment in segments:\n'
                               '                eligible=bool(segment.avg_logprob>=-1.0 and '
                               'segment.no_speech_prob<=.6 and segment.compression_ratio<=2.4)\n'
                               '                '
                               "row={'start':left+segment.start,'end':left+segment.end,'text':segment.text,'avg_logprob':float(segment.avg_logprob),'no_speech_prob':float(segment.no_speech_prob),'compression_ratio':float(segment.compression_ratio),'quality_filter_passed':eligible,'words':[]}\n"
                               '                for word in segment.words or []:\n'
                               '                    '
                               "item={'start':left+word.start,'end':left+word.end,'word':word.word,'probability':float(word.probability),'window_index':window_index}\n"
                               "                    row['words'].append(item)\n"
                               "                    item['low_confidence']=bool(word.probability<.4)\n"
                               '                    if eligible:observations.append(item)\n'
                               '                rows.append(row)\n'
                               '            '
                               "decodes.append({'gap_index':gap_index,'window_index':window_index,'window_start':left,'window_end':right,'segments':rows})\n"
                               '            '
                               "(args.output_dir/'window_decodes.json').write_text(json.dumps(decodes,indent=2)+'\\n')\n"
                               "            print('Decoded "
                               "gap',gap_index+1,'window',window_index+1,f'{left:.2f}-{right:.2f}',flush=True)\n"
                               '        for candidate in collect_candidates(observations,gap):\n'
                               "            candidate['gap_index']=gap_index;candidates.append(candidate)\n"
                               '    output=preserve_baseline(baseline,candidates)\n'
                               "    assert output['segments']==baseline['segments'],'Existing transcript "
                               "changed'\n"
                               "    offset=float(baseline.get('source_offset_seconds',0))\n"
                               '    '
                               "output['gap_recovery_settings']={'model':'large-v2','device':device,'vad_filter':False,'condition_on_previous_text':False,'window_seconds':args.window_seconds,'overlap_seconds':args.overlap_seconds,'context_seconds':args.context_seconds,'minimum_gap':args.minimum_gap,'gaps':gaps,'baseline_sha256':hashlib.sha256(args.baseline.read_bytes()).hexdigest(),'video_sha256':hashlib.sha256(args.video.read_bytes()).hexdigest()}\n"
                               '    '
                               "(args.output_dir/'transcript_with_candidates.json').write_text(json.dumps(output,indent=2)+'\\n')\n"
                               '    lines=[]\n'
                               '    for s in '
                               'baseline[\'segments\']:lines.append((s[\'start\'],f"[{s[\'start\']+offset:.2f}-{s[\'end\']+offset:.2f}] '
                               '{s.get(\'final_speaker\',s.get(\'speaker\',\'Unknown\'))}: {s[\'text\']}"))\n'
                               '    for s in candidates:\n'
                               '        support=f"overlap support '
                               '{s[\'supported_word_count\']}/{len(s[\'words\'])} words" if '
                               "s['supported_word_count'] else 'single decode'\n"
                               '        '
                               'lines.append((s[\'start\'],f"[{s[\'start\']+offset:.2f}-{s[\'end\']+offset:.2f}] '
                               'REVIEW ({support}; speaker unknown): {s[\'text\']}"))\n'
                               "    (args.output_dir/'review_transcript.txt').write_text('\\n'.join(text for "
                               "_,text in sorted(lines))+'\\n')\n"
                               "    print('Completed; preserved',len(baseline['segments']),'existing "
                               "segments;',len(candidates),'review candidates',flush=True)\n"
                               '\n'
                               "if __name__=='__main__':main()\n",
 'review_audio_window.py': '"""Optional local audio-window review using the project\'s existing models.\n'
                           '\n'
                           'Writes independent ASR/alignment/voice hypotheses, never edits a baseline.\n'
                           'No expected transcript text, named-video rules, clothing rules, or HF token.\n'
                           '"""\n'
                           'import argparse\n'
                           'import gc\n'
                           'import hashlib\n'
                           'import json\n'
                           'import math\n'
                           'from pathlib import Path\n'
                           'import subprocess\n'
                           'import time\n'
                           '\n'
                           '\n'
                           'def baseline_evidence(segments, start, end, offset=0):\n'
                           '    tracks = {}\n'
                           '    sources = []\n'
                           '    for index, s in enumerate(segments):\n'
                           '        '
                           "overlap=max(0,min(end,float(s['end'])+offset)-max(start,float(s['start'])+offset))\n"
                           '        if not overlap: continue\n'
                           '        '
                           "track=s.get('raw_speaker_track',s.get('baseline',{}).get('raw_speaker_track',s.get('base_track',s.get('speaker'))))\n"
                           "        sources.append({'segment_index':index,'raw_speaker_track':track,\n"
                           "            'baseline_speaker':s.get('final_speaker',s.get('speaker')),\n"
                           "            'overlap_seconds':overlap})\n"
                           '        if track is not None: tracks[track]=tracks.get(track,0)+overlap\n'
                           "    return {'source':'baseline_overlap','raw_track_overlap_seconds':tracks,\n"
                           "            'segments':sources,'identity_verified':False}\n"
                           '\n'
                           '\n'
                           'def decoder_sentence_bounds(decoded_segments, aligned_segments):\n'
                           '    """Link generated sentence text to its own decoder words, in sequence.\n'
                           '\n'
                           '    This matches two representations of the same ASR hypothesis, not supplied\n'
                           '    expected dialogue. Missing links yield None rather than invented timings.\n'
                           '    """\n'
                           '    text_parts=[];word_spans=[];base=0\n'
                           '    for segment in decoded_segments:\n'
                           "        body=' '.join(segment['text'].split());cursor=0\n"
                           "        for word in segment.get('words',[]):\n"
                           "            token=' '.join(word.get('word','').split())\n"
                           '            location=body.find(token,cursor) if token else -1\n'
                           '            if location<0:continue\n'
                           '            '
                           "word_spans.append((base+location,base+location+len(token),float(word['start']),float(word['end'])))\n"
                           '            cursor=location+len(token)\n'
                           '        text_parts.append(body);base+=len(body)+1\n'
                           "    text=' '.join(text_parts);cursor=0;bounds=[]\n"
                           '    for segment in aligned_segments:\n'
                           "        sentence=' "
                           "'.join(segment['text'].split());left=text.find(sentence,cursor) if sentence else "
                           '-1\n'
                           '        if left<0:\n'
                           '            bounds.append(None);continue\n'
                           '        right=left+len(sentence);cursor=right\n'
                           '        words=[w for w in word_spans if w[0]<right and w[1]>left]\n'
                           '        if not words or max(w[3] for w in words)<=min(w[2] for w in words):\n'
                           '            bounds.append(None)\n'
                           '        else:bounds.append((min(w[2] for w in words),max(w[3] for w in words)))\n'
                           '    return bounds\n'
                           '\n'
                           '\n'
                           'def resolve_voice(scores, min_similarity=.25, min_margin=.08):\n'
                           '    """Conservative review hypothesis; thresholds are not calibrated '
                           'confidence."""\n'
                           "    if not scores: return 'Uncertain'\n"
                           '    ranked=sorted(scores,key=scores.get,reverse=True)\n'
                           '    # A margin requires at least one competing profile. With only a target\n'
                           '    # reference, use similarity alone but keep the explicit review requirement.\n'
                           '    runner=scores[ranked[1]] if len(ranked)>1 else None\n'
                           "    if scores[ranked[0]] < min_similarity: return 'Uncertain'\n"
                           '    if runner is not None and scores[ranked[0]]-runner < min_margin: return '
                           "'Uncertain'\n"
                           '    return ranked[0]\n'
                           '\n'
                           '\n'
                           'def resolve_timing_evidence(variants, min_similarity=.25, min_margin=.08):\n'
                           '    """Use one evidence family: agree on the leading voice, with qualified '
                           'support.\n'
                           '\n'
                           '    Alternate crops are not independent votes and do not raise confidence.\n'
                           '    Conflicting leading identities abstain, even if one crop matches strongly.\n'
                           '    """\n'
                           '    usable=[scores for scores in variants if scores]\n'
                           "    if not usable:return 'Uncertain'\n"
                           '    leaders={max(scores,key=scores.get) for scores in usable}\n'
                           "    if len(leaders)!=1:return 'Uncertain'\n"
                           '    leader=next(iter(leaders))\n'
                           '    return leader if any(resolve_voice(scores,min_similarity,min_margin)==leader '
                           "for scores in usable) else 'Uncertain'\n"
                           '\n'
                           '\n'
                           'def decoder_quality_flags(segments, start, end):\n'
                           '    """Keep decoder warnings as evidence; word probability is not accuracy."""\n'
                           "    observed=[s for s in segments if s['start']<end and s['end']>start]\n"
                           '    flags=[]\n'
                           "    if any(s.get('no_speech_prob',0)>.6 for s in observed):\n"
                           "        flags.append('Decoder marks this passage as possible non-speech')\n"
                           "    if any(s.get('avg_logprob',0)<-1 for s in observed):\n"
                           "        flags.append('Low decoder support for wording')\n"
                           "    if any(s.get('compression_ratio',0)>2.4 for s in observed):\n"
                           "        flags.append('Decoder wording may be repetitive')\n"
                           "    return flags, [{'start':s['start'],'end':s['end'],**{k:s[k] for k in "
                           "('avg_logprob','no_speech_prob','compression_ratio') if k in s}} for s in "
                           'observed]\n'
                           '\n'
                           '\n'
                           'def sha(path):\n'
                           '    h=hashlib.sha256()\n'
                           "    with path.open('rb') as f:\n"
                           "        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)\n"
                           '    return h.hexdigest()\n'
                           '\n'
                           '\n'
                           'def main():\n'
                           '    p=argparse.ArgumentParser(description=__doc__)\n'
                           "    p.add_argument('--video',type=Path,required=True)\n"
                           "    p.add_argument('--start',type=float,required=True)\n"
                           "    p.add_argument('--duration',type=float,required=True)\n"
                           "    p.add_argument('--target-reference',type=Path,required=True)\n"
                           '    '
                           "p.add_argument('--other-reference',action='append',default=[],metavar='NAME=PATH')\n"
                           "    p.add_argument('--baseline',type=Path)\n"
                           "    p.add_argument('--baseline-offset',type=float,default=0,\n"
                           "                   help='Add this offset to baseline times; default assumes "
                           "absolute video times')\n"
                           "    p.add_argument('--output-dir',type=Path,required=True)\n"
                           "    p.add_argument('--model',default='large-v2')\n"
                           '    '
                           "p.add_argument('--asr-engine',choices=['native','bounded-whisperx'],default='native')\n"
                           '    '
                           "p.add_argument('--timing-source',choices=['consensus','decoder','alignment'],default='consensus')\n"
                           "    p.add_argument('--language',default='en')\n"
                           "    p.add_argument('--device',choices=['auto','cpu','cuda'],default='auto')\n"
                           "    p.add_argument('--threads',type=int,default=4)\n"
                           "    p.add_argument('--chunk-seconds',type=float,default=20)\n"
                           "    p.add_argument('--context-seconds',type=float,default=.5)\n"
                           "    p.add_argument('--min-similarity',type=float,default=.25)\n"
                           "    p.add_argument('--min-margin',type=float,default=.08)\n"
                           "    p.add_argument('--speechbrain-cache',type=Path)\n"
                           '    a=p.parse_args()\n'
                           '    if not math.isfinite(a.start) or a.start<0 or not math.isfinite(a.duration) '
                           'or a.duration<=0:\n'
                           "        p.error('Provide a nonnegative start and positive duration')\n"
                           '    if not math.isfinite(a.baseline_offset) or a.threads<1:\n'
                           "        p.error('Invalid offset or thread count')\n"
                           '    if not math.isfinite(a.chunk_seconds) or not 1<=a.chunk_seconds<=30:\n'
                           "        p.error('Chunk length must be between 1 and 30 seconds')\n"
                           '    if not -1<=a.min_similarity<=1 or not 0<=a.min_margin<=2:\n'
                           "        p.error('Invalid voice gates')\n"
                           "    references={'Target_Speaker':a.target_reference}\n"
                           '    for item in a.other_reference:\n'
                           "        if '=' not in item:p.error('Other reference must be NAME=PATH')\n"
                           "        name,path=item.split('=',1)\n"
                           '        if not name or name in references or name in '
                           "('Unknown','Uncertain'):p.error('Reference name must be unique')\n"
                           '        references[name]=Path(path)\n'
                           '    paths=[a.video,*references.values()]+([a.baseline] if a.baseline else [])\n'
                           '    for path in paths:\n'
                           "        if not path.is_file():p.error(f'File not found: {path}')\n"
                           '    planned=[a.output_dir/x for x in '
                           "('asr.json','alignment.json','review_hypotheses.json','review_transcript.txt')]\n"
                           '    if any(path.resolve() in [x.resolve() for x in paths] for path in planned):\n'
                           "        p.error('Outputs must be separate from input files')\n"
                           '    if any(path.exists() for path in planned):\n'
                           "        p.error('Choose an empty output directory to preserve earlier "
                           "experiments')\n"
                           '    import numpy as np\n'
                           '    import torch\n'
                           '    import whisperx\n'
                           '    from bounded_silero_vad import make_vad\n'
                           '    torch.set_num_threads(a.threads)\n'
                           "    device=('cuda' if torch.cuda.is_available() else 'cpu') if a.device=='auto' "
                           'else a.device\n'
                           "    if device=='cuda' and not torch.cuda.is_available():p.error('CUDA is "
                           "unavailable')\n"
                           '    a.output_dir.mkdir(parents=True,exist_ok=True)\n'
                           '    started=time.monotonic()\n'
                           '    '
                           "raw=subprocess.check_output(['ffmpeg','-nostdin','-hide_banner','-loglevel','error',\n"
                           "        '-ss',str(a.start),'-i',str(a.video),'-t',str(a.duration),'-vn',\n"
                           "        '-ar','16000','-ac','1','-f','f32le','pipe:1'])\n"
                           "    audio=np.frombuffer(raw,dtype='<f4').copy()\n"
                           "    if not len(audio) or not np.isfinite(audio).all():p.error('No valid audio "
                           "decoded')\n"
                           "    if a.asr_engine=='native':\n"
                           '        from faster_whisper import WhisperModel\n'
                           "        asr=WhisperModel(a.model,device=device,compute_type='float16' if "
                           "device=='cuda' else 'int8',cpu_threads=a.threads)\n"
                           '        decoded,_=asr.transcribe(audio,language=a.language,beam_size=5,\n'
                           '            '
                           'vad_filter=False,condition_on_previous_text=False,word_timestamps=True)\n'
                           '        '
                           "result={'language':a.language,'segments':[{'start':float(s.start),'end':float(s.end),'text':s.text,\n"
                           '            '
                           "'avg_logprob':float(s.avg_logprob),'no_speech_prob':float(s.no_speech_prob),\n"
                           "            'compression_ratio':float(s.compression_ratio),\n"
                           '            '
                           "'words':[{'start':float(w.start),'end':float(w.end),'word':w.word,'probability':float(w.probability)} "
                           'for w in s.words or []]} for s in decoded]}\n'
                           '    else:\n'
                           "        asr=whisperx.load_model(a.model,device,compute_type='float16' if "
                           "device=='cuda' else 'int8',\n"
                           '            language=a.language,vad_model=make_vad(context=a.context_seconds))\n'
                           '        '
                           'result=asr.transcribe(audio,batch_size=4,chunk_size=a.chunk_seconds,language=a.language)\n'
                           "    (a.output_dir/'asr.json').write_text(json.dumps(result,indent=2)+'\\n')\n"
                           '    del asr;gc.collect()\n'
                           "    if device=='cuda':torch.cuda.empty_cache()\n"
                           "    if result['segments']:\n"
                           '        '
                           'aligner,metadata=whisperx.load_align_model(language_code=a.language,device=device)\n'
                           '        '
                           "aligned=whisperx.align(result['segments'],aligner,metadata,audio,device,return_char_alignments=False)\n"
                           '        del aligner;gc.collect()\n'
                           "        if device=='cuda':torch.cuda.empty_cache()\n"
                           "    else:aligned={'segments':[],'word_segments':[]}\n"
                           '    '
                           "(a.output_dir/'alignment.json').write_text(json.dumps(aligned,indent=2)+'\\n')\n"
                           '    from speechbrain.inference.speaker import SpeakerRecognition\n'
                           '    '
                           "options={'source':'speechbrain/spkrec-ecapa-voxceleb','run_opts':{'device':device}}\n"
                           "    if a.speechbrain_cache:options['savedir']=str(a.speechbrain_cache)\n"
                           '    encoder=SpeakerRecognition.from_hparams(**options)\n'
                           '    def unit(v):\n'
                           '        v=np.asarray(v,dtype=np.float32).reshape(-1)\n'
                           '        if not np.isfinite(v).all() or np.linalg.norm(v)<1e-8:raise '
                           "ValueError('Invalid embedding')\n"
                           '        return v/np.linalg.norm(v)\n'
                           '    profiles={}\n'
                           '    for name,path in references.items():\n'
                           '        values=np.load(path,allow_pickle=False)\n'
                           '        if values.ndim==1:values=values[None,:]\n'
                           "        if values.ndim!=2 or not len(values):raise ValueError('Reference must "
                           "contain one or more embeddings')\n"
                           '        profiles[name]=unit(np.mean([unit(v) for v in values],axis=0))\n'
                           '    baseline=json.loads(a.baseline.read_text()) if a.baseline else '
                           "{'segments':[]}\n"
                           '    rows=[]\n'
                           '    '
                           "decoder_bounds=decoder_sentence_bounds(result['segments'],aligned['segments'])\n"
                           "    for index,s in enumerate(aligned['segments']):\n"
                           '        '
                           "alignment_left,alignment_right=float(s['start']),float(s['end']);scores={};reasons=[]\n"
                           '        bounds=decoder_bounds[index]\n'
                           "        use_decoder=a.timing_source in ('decoder','consensus') and bounds is not "
                           'None\n'
                           '        left,right=bounds if use_decoder else (alignment_left,alignment_right)\n'
                           "        if a.timing_source in ('decoder','consensus') and bounds is "
                           "None:reasons.append('Decoder word boundaries unavailable; alignment fallback "
                           "needs review')\n"
                           '        if right-left>=.4:\n'
                           '            '
                           'crop=torch.from_numpy(audio[round(left*16000):round(right*16000)]).unsqueeze(0).to(device)\n'
                           '            with '
                           'torch.no_grad():v=unit(encoder.encode_batch(crop).detach().cpu().numpy())\n'
                           '            for name,profile in profiles.items():\n'
                           "                if v.shape!=profile.shape:raise ValueError('Reference embedding "
                           "model/dimension mismatch')\n"
                           '                scores[name]=float(v@profile)\n'
                           "        else:reasons.append('Voice crop shorter than 0.4 seconds')\n"
                           "        variants=[{'source':'decoder' if use_decoder else "
                           "'alignment','start':a.start+left,'end':a.start+right,'scores':scores}]\n"
                           "        if a.timing_source=='consensus' and use_decoder and "
                           'alignment_right-alignment_left>=.4 and (abs(left-alignment_left)>1/16000 or '
                           'abs(right-alignment_right)>1/16000):\n'
                           '            '
                           'crop=torch.from_numpy(audio[round(alignment_left*16000):round(alignment_right*16000)]).unsqueeze(0).to(device)\n'
                           '            with '
                           'torch.no_grad():alternate=unit(encoder.encode_batch(crop).detach().cpu().numpy())\n'
                           '            '
                           "variants.append({'source':'alignment','start':a.start+alignment_left,'end':a.start+alignment_right,'scores':{name:float(alternate@profile) "
                           'for name,profile in profiles.items()}})\n'
                           "        speaker=resolve_timing_evidence([v['scores'] for v in "
                           "variants],a.min_similarity,a.min_margin) if a.timing_source=='consensus' else "
                           'resolve_voice(scores,a.min_similarity,a.min_margin)\n'
                           '        '
                           "quality_flags,quality_signals=decoder_quality_flags(result['segments'],*(bounds "
                           'if bounds is not None else (left,right)))\n'
                           '        reasons.extend(quality_flags)\n'
                           '        near_edge=left<=.25 or right>=len(audio)/16000-.25\n'
                           "        if near_edge:reasons.append('Near audio-window boundary; wording or "
                           "timing may be incomplete')\n"
                           "        if speaker=='Uncertain':reasons.append('Insufficient voice similarity or "
                           "separation between profiles')\n"
                           "        if len(profiles)==1:reasons.append('No competing voice reference; "
                           "target-only match needs review')\n"
                           '        absolute_start,absolute_end=a.start+left,a.start+right\n'
                           '        '
                           "rows.append({'index':index,'start':absolute_start,'end':absolute_end,'text':s['text'],\n"
                           '            '
                           "'speaker_hypothesis':speaker,'transcription_status':'decoder_warning' if "
                           'quality_flags else '
                           "'review_hypothesis','review_required':True,'near_window_boundary':near_edge,'reasons':reasons,\n"
                           '            '
                           "'evidence':[{'source':'decoder_support','flags':quality_flags,'signals':quality_signals,'word_probability_is_accuracy':False},{'source':'timing_comparison','selected_source':'decoder' "
                           "if use_decoder else 'alignment',\n"
                           '                '
                           "'alignment_start':a.start+alignment_left,'alignment_end':a.start+alignment_right,\n"
                           "                'decoder_start':a.start+bounds[0] if bounds else "
                           "None,'decoder_end':a.start+bounds[1] if bounds else "
                           "None},baseline_evidence(baseline['segments'],absolute_start,absolute_end,a.baseline_offset),\n"
                           '                '
                           "{'source':'local_voice','similarities':scores,'timing_variants':variants,'variants_are_independent_votes':False,'min_similarity':a.min_similarity,\n"
                           '                 '
                           "'min_margin':a.min_margin,'identity_probability_calibrated':False}],\n"
                           "            'alignment_words':[{**w,**({'start':a.start+w['start']} if 'start' "
                           'in w else {}),\n'
                           "                      **({'end':a.start+w['end']} if 'end' in w else {})} for w "
                           "in s.get('words',[])]})\n"
                           "    document={'baseline_modified':False,'experimental':True,'segments':rows,\n"
                           '        '
                           "'provenance':{'video':str(a.video.resolve()),'video_sha256':sha(a.video),\n"
                           "            'window_start':a.start,'decoded_duration':len(audio)/16000,\n"
                           '            '
                           "'models':{'asr':a.model,'voice':'speechbrain/spkrec-ecapa-voxceleb'},\n"
                           '            '
                           "'device':device,'requested_timing_source':a.timing_source,'asr_engine':a.asr_engine,'vad':'disabled' "
                           "if a.asr_engine=='native' else 'bounded_silero','chunk_seconds':a.chunk_seconds "
                           "if a.asr_engine=='bounded-whisperx' else None,\n"
                           "            'context_seconds':a.context_seconds if "
                           "a.asr_engine=='bounded-whisperx' else "
                           "None,'references':{k:{'path':str(v.resolve()),'sha256':sha(v)} for k,v in "
                           'references.items()},\n'
                           "            'baseline':str(a.baseline.resolve()) if a.baseline else None,\n"
                           "            'baseline_sha256':sha(a.baseline) if a.baseline else "
                           "None,'elapsed_seconds':time.monotonic()-started}}\n"
                           '    '
                           "(a.output_dir/'review_hypotheses.json').write_text(json.dumps(document,indent=2)+'\\n')\n"
                           '    '
                           '(a.output_dir/\'review_transcript.txt\').write_text(\'\\n\'.join(f"[{r[\'start\']:.2f}–{r[\'end\']:.2f}] '
                           "{r['speaker_hypothesis']}{' [decoder warning]' if "
                           'r[\'transcription_status\']==\'decoder_warning\' else \'\'}: {r[\'text\']}" for '
                           "r in rows)+'\\n')\n"
                           "    print(f'Wrote {len(rows)} review hypotheses to {a.output_dir}; baseline "
                           "preserved.')\n"
                           '\n'
                           "if __name__=='__main__':main()\n",
 'review_transcript_regions.py': '"""Review doubtful transcript regions without modifying the baseline.\n'
                                 '\n'
                                 'This is a batch companion to review_audio_window.py.  It loads each large\n'
                                 'model once, checkpoints transcription/alignment per region, and writes '
                                 'only\n'
                                 'supplemental hypotheses.  Selection uses confidence/duration/gaps, never\n'
                                 'expected dialogue or speaker-specific video rules.\n'
                                 '"""\n'
                                 'from __future__ import annotations\n'
                                 '\n'
                                 'import argparse\n'
                                 'import gc\n'
                                 'import hashlib\n'
                                 'import json\n'
                                 'import math\n'
                                 'from pathlib import Path\n'
                                 'import subprocess\n'
                                 'import time\n'
                                 '\n'
                                 '\n'
                                 'def sha256(path: Path) -> str:\n'
                                 '    digest = hashlib.sha256()\n'
                                 '    with path.open("rb") as source:\n'
                                 '        for block in iter(lambda: source.read(1024 * 1024), b""):\n'
                                 '            digest.update(block)\n'
                                 '    return digest.hexdigest()\n'
                                 '\n'
                                 '\n'
                                 'def media_duration(path: Path) -> float:\n'
                                 '    value = subprocess.check_output([\n'
                                 '        "ffprobe", "-v", "error", "-show_entries", "format=duration",\n'
                                 '        "-of", "default=noprint_wrappers=1:nokey=1", str(path)\n'
                                 '    ], text=True).strip()\n'
                                 '    duration = float(value)\n'
                                 '    if not math.isfinite(duration) or duration <= 0:\n'
                                 '        raise ValueError("Invalid media duration")\n'
                                 '    return duration\n'
                                 '\n'
                                 '\n'
                                 'def select_review_regions(segments, duration, confidence=0.35,\n'
                                 '                          short_seconds=1.0, minimum_gap=5.0,\n'
                                 '                          context=3.0, merge_gap=2.0,\n'
                                 '                          maximum_window=30.0, overlap=4.0,\n'
                                 '                          extra_regions=()):\n'
                                 '    """Select and bound review windows. Extra regions are external '
                                 'controls."""\n'
                                 '    if not (0 <= confidence <= 1 and short_seconds >= 0 and minimum_gap >= '
                                 '0\n'
                                 '            and context >= 0 and merge_gap >= 0 and maximum_window > 0\n'
                                 '            and 0 <= overlap < maximum_window and duration > 0):\n'
                                 '        raise ValueError("Invalid region selection settings")\n'
                                 '    ordered = sorted(segments, key=lambda row: (float(row["start"]), '
                                 'float(row["end"])))\n'
                                 '    candidates = []\n'
                                 '    for index, row in enumerate(ordered):\n'
                                 '        start, end = float(row["start"]), float(row["end"])\n'
                                 '        if not (0 <= start <= end <= duration + 0.5):\n'
                                 '            raise ValueError("Baseline contains invalid segment times")\n'
                                 '        speaker = row.get("final_speaker", row.get("speaker", '
                                 '"Uncertain"))\n'
                                 '        strength = float(row.get("final_confidence", 0.0) or 0.0)\n'
                                 '        reasons = []\n'
                                 '        if speaker in ("Uncertain", "Unknown", "Unknown_Speaker", None):\n'
                                 '            reasons.append("uncertain_speaker")\n'
                                 '        if strength < confidence:\n'
                                 '            reasons.append("weak_identity_evidence")\n'
                                 '        if end - start <= short_seconds:\n'
                                 '            reasons.append("short_utterance")\n'
                                 '        if reasons:\n'
                                 '            candidates.append({"start": max(0, start-context),\n'
                                 '                               "end": min(duration, end+context),\n'
                                 '                               "reasons": reasons,\n'
                                 '                               "baseline_indices": [index]})\n'
                                 '    previous = 0.0\n'
                                 '    for index, row in enumerate(ordered):\n'
                                 '        start = float(row["start"])\n'
                                 '        if start - previous >= minimum_gap:\n'
                                 '            candidates.append({"start": max(0, previous-context),\n'
                                 '                               "end": min(duration, start+context),\n'
                                 '                               "reasons": ["transcript_gap"],\n'
                                 '                               "baseline_indices": []})\n'
                                 '        previous = max(previous, float(row["end"]))\n'
                                 '    if duration - previous >= minimum_gap:\n'
                                 '        candidates.append({"start": max(0, previous-context), "end": '
                                 'duration,\n'
                                 '                           "reasons": ["transcript_gap"], '
                                 '"baseline_indices": []})\n'
                                 '    for start, end in extra_regions:\n'
                                 '        if not (0 <= start < end <= duration):\n'
                                 '            raise ValueError("Extra review region is outside the video")\n'
                                 '        candidates.append({"start": start, "end": end,\n'
                                 '                           "reasons": ["external_review_control"],\n'
                                 '                           "baseline_indices": []})\n'
                                 '    candidates.sort(key=lambda row: (row["start"], row["end"]))\n'
                                 '    merged = []\n'
                                 '    for item in candidates:\n'
                                 '        if merged and item["start"] <= merged[-1]["end"] + merge_gap:\n'
                                 '            merged[-1]["end"] = max(merged[-1]["end"], item["end"])\n'
                                 '            merged[-1]["reasons"] = sorted(set(merged[-1]["reasons"] + '
                                 'item["reasons"]))\n'
                                 '            merged[-1]["baseline_indices"] = '
                                 'sorted(set(merged[-1]["baseline_indices"] + item["baseline_indices"]))\n'
                                 '        else:\n'
                                 '            merged.append(dict(item))\n'
                                 '    windows = []\n'
                                 '    for item in merged:\n'
                                 '        left = item["start"]\n'
                                 '        while left < item["end"] - 1e-6:\n'
                                 '            right = min(left + maximum_window, item["end"])\n'
                                 '            windows.append({"index": len(windows), "start": left, "end": '
                                 'right,\n'
                                 '                            "reasons": item["reasons"],\n'
                                 '                            "baseline_indices": '
                                 'item["baseline_indices"]})\n'
                                 '            if right >= item["end"]:\n'
                                 '                break\n'
                                 '            left = right - overlap\n'
                                 '    return windows\n'
                                 '\n'
                                 '\n'
                                 'def parse_region(value):\n'
                                 '    try:\n'
                                 '        start, end = (float(part) for part in value.split(":", 1))\n'
                                 '    except Exception as error:\n'
                                 '        raise argparse.ArgumentTypeError("Region must be START:END") from '
                                 'error\n'
                                 '    if not (math.isfinite(start) and math.isfinite(end) and 0 <= start < '
                                 'end):\n'
                                 '        raise argparse.ArgumentTypeError("Region must be finite and '
                                 'increasing")\n'
                                 '    return start, end\n'
                                 '\n'
                                 '\n'
                                 'def main():\n'
                                 '    parser = argparse.ArgumentParser(description=__doc__)\n'
                                 '    parser.add_argument("--video", type=Path, required=True)\n'
                                 '    parser.add_argument("--baseline", type=Path, required=True)\n'
                                 '    parser.add_argument("--target-reference", type=Path, required=True)\n'
                                 '    parser.add_argument("--other-reference", action="append", default=[], '
                                 'metavar="NAME=PATH")\n'
                                 '    parser.add_argument("--output-dir", type=Path, required=True)\n'
                                 '    parser.add_argument("--cache-dir", type=Path, required=True)\n'
                                 '    parser.add_argument("--extra-region", action="append", '
                                 'type=parse_region, default=[])\n'
                                 '    parser.add_argument("--weak-confidence", type=float, default=0.35)\n'
                                 '    parser.add_argument("--short-seconds", type=float, default=1.0)\n'
                                 '    parser.add_argument("--minimum-gap", type=float, default=5.0)\n'
                                 '    parser.add_argument("--context-seconds", type=float, default=3.0)\n'
                                 '    parser.add_argument("--maximum-window-seconds", type=float, '
                                 'default=30.0)\n'
                                 '    parser.add_argument("--window-overlap-seconds", type=float, '
                                 'default=4.0)\n'
                                 '    parser.add_argument("--model", default="large-v2")\n'
                                 '    parser.add_argument("--language", default="en")\n'
                                 '    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), '
                                 'default="auto")\n'
                                 '    parser.add_argument("--threads", type=int, default=4)\n'
                                 '    parser.add_argument("--min-similarity", type=float, default=0.25)\n'
                                 '    parser.add_argument("--min-margin", type=float, default=0.08)\n'
                                 '    parser.add_argument("--speechbrain-cache", type=Path)\n'
                                 '    args = parser.parse_args()\n'
                                 '    input_paths = [args.video, args.baseline, args.target_reference]\n'
                                 '    references = {"Target_Speaker": args.target_reference}\n'
                                 '    for item in args.other_reference:\n'
                                 '        if "=" not in item:\n'
                                 '            parser.error("Other reference must be NAME=PATH")\n'
                                 '        name, path = item.split("=", 1)\n'
                                 '        if not name or name in references or name in ("Unknown", '
                                 '"Uncertain"):\n'
                                 '            parser.error("Reference names must be unique")\n'
                                 '        references[name] = Path(path)\n'
                                 '        input_paths.append(Path(path))\n'
                                 '    input_paths.append(args.baseline)\n'
                                 '    for path in input_paths:\n'
                                 '        if not path.is_file():\n'
                                 '            parser.error(f"Missing input: {path}")\n'
                                 '    baseline_bytes = args.baseline.read_bytes()\n'
                                 '    baseline_hash = hashlib.sha256(baseline_bytes).hexdigest()\n'
                                 '    baseline = json.loads(baseline_bytes)\n'
                                 '    if not isinstance(baseline.get("segments"), list):\n'
                                 '        parser.error("Baseline must contain a segments list")\n'
                                 '    duration = media_duration(args.video)\n'
                                 '    windows = select_review_regions(\n'
                                 '        baseline["segments"], duration, args.weak_confidence,\n'
                                 '        args.short_seconds, args.minimum_gap, args.context_seconds, 2.0,\n'
                                 '        args.maximum_window_seconds, args.window_overlap_seconds,\n'
                                 '        args.extra_region)\n'
                                 '    configuration = {\n'
                                 '        "video_sha256": sha256(args.video), "baseline_sha256": '
                                 'baseline_hash,\n'
                                 '        "references": {name: sha256(path) for name, path in '
                                 'references.items()},\n'
                                 '        "model": args.model, "language": args.language,\n'
                                 '        "weak_confidence": args.weak_confidence, "short_seconds": '
                                 'args.short_seconds,\n'
                                 '        "minimum_gap": args.minimum_gap, "context_seconds": '
                                 'args.context_seconds,\n'
                                 '        "maximum_window_seconds": args.maximum_window_seconds,\n'
                                 '        "window_overlap_seconds": args.window_overlap_seconds,\n'
                                 '        "extra_regions": args.extra_region, "windows": windows}\n'
                                 '    fingerprint = hashlib.sha256(json.dumps(configuration, '
                                 'sort_keys=True).encode()).hexdigest()\n'
                                 '    args.output_dir.mkdir(parents=True, exist_ok=True)\n'
                                 '    cache = args.cache_dir / fingerprint\n'
                                 '    cache.mkdir(parents=True, exist_ok=True)\n'
                                 '    (args.output_dir / '
                                 '"selection.json").write_text(json.dumps(configuration, indent=2)+"\\n")\n'
                                 '    print(f"Selected {len(windows)} windows, {sum(w[\'end\']-w[\'start\'] '
                                 'for w in windows)/60:.1f} decoded minutes")\n'
                                 '\n'
                                 '    import numpy as np\n'
                                 '    import torch\n'
                                 '    torch.set_num_threads(args.threads)\n'
                                 '    device = ("cuda" if torch.cuda.is_available() else "cpu") if '
                                 'args.device == "auto" else args.device\n'
                                 '    if device == "cuda" and not torch.cuda.is_available():\n'
                                 '        parser.error("CUDA was requested but is unavailable")\n'
                                 '    raw = subprocess.check_output(["ffmpeg", "-nostdin", "-hide_banner", '
                                 '"-loglevel", "error",\n'
                                 '        "-i", str(args.video), "-vn", "-ar", "16000", "-ac", "1", "-f", '
                                 '"f32le", "pipe:1"])\n'
                                 '    audio = np.frombuffer(raw, dtype="<f4").copy()\n'
                                 '    if not len(audio) or not np.isfinite(audio).all():\n'
                                 '        raise ValueError("Decoded audio is empty or invalid")\n'
                                 '\n'
                                 '    asr_records = {}\n'
                                 '    missing = []\n'
                                 '    for window in windows:\n'
                                 '        path = cache / f"asr-{window[\'index\']:04d}.json"\n'
                                 '        if path.exists():\n'
                                 '            asr_records[window["index"]] = json.loads(path.read_text())\n'
                                 '        else:\n'
                                 '            missing.append(window)\n'
                                 '    if missing:\n'
                                 '        from faster_whisper import WhisperModel\n'
                                 '        model = WhisperModel(args.model, device=device,\n'
                                 '            compute_type="float16" if device == "cuda" else "int8", '
                                 'cpu_threads=args.threads)\n'
                                 '        for count, window in enumerate(missing, 1):\n'
                                 '            left, right = window["start"], window["end"]\n'
                                 '            decoded, _ = '
                                 'model.transcribe(audio[round(left*16000):round(right*16000)],\n'
                                 '                language=args.language, beam_size=5, vad_filter=False,\n'
                                 '                condition_on_previous_text=False, word_timestamps=True)\n'
                                 '            rows = []\n'
                                 '            for segment in decoded:\n'
                                 '                rows.append({"start": float(segment.start), "end": '
                                 'float(segment.end),\n'
                                 '                    "text": segment.text, "avg_logprob": '
                                 'float(segment.avg_logprob),\n'
                                 '                    "no_speech_prob": float(segment.no_speech_prob),\n'
                                 '                    "compression_ratio": '
                                 'float(segment.compression_ratio),\n'
                                 '                    "words": [{"start": float(word.start), "end": '
                                 'float(word.end),\n'
                                 '                               "word": word.word, "probability": '
                                 'float(word.probability)}\n'
                                 '                              for word in segment.words or []]})\n'
                                 '            record = {"window": window, "segments": rows, "language": '
                                 'args.language}\n'
                                 '            path = cache / f"asr-{window[\'index\']:04d}.json"\n'
                                 '            path.write_text(json.dumps(record, indent=2)+"\\n")\n'
                                 '            asr_records[window["index"]] = record\n'
                                 '            print(f"Transcribed review window {count}/{len(missing)}", '
                                 'flush=True)\n'
                                 '        del model\n'
                                 '        gc.collect()\n'
                                 '        if device == "cuda": torch.cuda.empty_cache()\n'
                                 '\n'
                                 '    import whisperx\n'
                                 '    aligned_records = {}\n'
                                 '    missing = []\n'
                                 '    for window in windows:\n'
                                 '        path = cache / f"alignment-{window[\'index\']:04d}.json"\n'
                                 '        if path.exists():\n'
                                 '            aligned_records[window["index"]] = '
                                 'json.loads(path.read_text())\n'
                                 '        else:\n'
                                 '            missing.append(window)\n'
                                 '    if missing:\n'
                                 '        aligner, metadata = '
                                 'whisperx.load_align_model(language_code=args.language, device=device)\n'
                                 '        for count, window in enumerate(missing, 1):\n'
                                 '            record = asr_records[window["index"]]\n'
                                 '            left, right = window["start"], window["end"]\n'
                                 '            if record["segments"]:\n'
                                 '                aligned = whisperx.align(record["segments"], aligner, '
                                 'metadata,\n'
                                 '                    audio[round(left*16000):round(right*16000)], device,\n'
                                 '                    return_char_alignments=False)\n'
                                 '            else:\n'
                                 '                aligned = {"segments": [], "word_segments": []}\n'
                                 '            output = {"window": window, **aligned}\n'
                                 '            path = cache / f"alignment-{window[\'index\']:04d}.json"\n'
                                 '            path.write_text(json.dumps(output, indent=2)+"\\n")\n'
                                 '            aligned_records[window["index"]] = output\n'
                                 '            print(f"Aligned review window {count}/{len(missing)}", '
                                 'flush=True)\n'
                                 '        del aligner\n'
                                 '        gc.collect()\n'
                                 '        if device == "cuda": torch.cuda.empty_cache()\n'
                                 '\n'
                                 '    from speechbrain.inference.speaker import SpeakerRecognition\n'
                                 '    from review_audio_window import (baseline_evidence, '
                                 'decoder_quality_flags,\n'
                                 '        decoder_sentence_bounds, resolve_timing_evidence)\n'
                                 '    options = {"source": "speechbrain/spkrec-ecapa-voxceleb", "run_opts": '
                                 '{"device": device}}\n'
                                 '    if args.speechbrain_cache:\n'
                                 '        options["savedir"] = str(args.speechbrain_cache)\n'
                                 '    encoder = SpeakerRecognition.from_hparams(**options)\n'
                                 '\n'
                                 '    def unit(value):\n'
                                 '        value = np.asarray(value, dtype=np.float32).reshape(-1)\n'
                                 '        norm = np.linalg.norm(value)\n'
                                 '        if not np.isfinite(value).all() or norm < 1e-8:\n'
                                 '            raise ValueError("Invalid voice embedding")\n'
                                 '        return value / norm\n'
                                 '\n'
                                 '    profiles = {}\n'
                                 '    for name, path in references.items():\n'
                                 '        values = np.load(path, allow_pickle=False)\n'
                                 '        if values.ndim == 1:\n'
                                 '            values = values[None, :]\n'
                                 '        if values.ndim != 2 or not len(values):\n'
                                 '            raise ValueError("Reference must contain voice embeddings")\n'
                                 '        profiles[name] = unit(np.mean([unit(value) for value in values], '
                                 'axis=0))\n'
                                 '    rows = []\n'
                                 '    for window in windows:\n'
                                 '        left = window["start"]\n'
                                 '        decoded = asr_records[window["index"]]\n'
                                 '        aligned = aligned_records[window["index"]]\n'
                                 '        decoder_bounds = decoder_sentence_bounds(decoded["segments"], '
                                 'aligned["segments"])\n'
                                 '        for local_index, (segment, bounds) in '
                                 'enumerate(zip(aligned["segments"], decoder_bounds)):\n'
                                 '            alignment_bounds = (float(segment["start"]), '
                                 'float(segment["end"]))\n'
                                 '            primary = bounds if bounds is not None else alignment_bounds\n'
                                 '            variants = []\n'
                                 '            for source, crop in (("decoder", bounds), ("alignment", '
                                 'alignment_bounds)):\n'
                                 '                if crop is None or crop[1]-crop[0] < 0.4:\n'
                                 '                    continue\n'
                                 '                if variants and '
                                 'all(abs(crop[i]-variants[0][f"local_{\'start\' if i == 0 else \'end\'}"]) '
                                 '< 1/16000 for i in (0, 1)):\n'
                                 '                    continue\n'
                                 '                waveform = torch.from_numpy(audio[\n'
                                 '                    '
                                 'round((left+crop[0])*16000):round((left+crop[1])*16000)]).unsqueeze(0).to(device)\n'
                                 '                with torch.no_grad():\n'
                                 '                    voice = '
                                 'unit(encoder.encode_batch(waveform).detach().cpu().numpy())\n'
                                 '                scores = {}\n'
                                 '                for name, profile in profiles.items():\n'
                                 '                    if voice.shape != profile.shape:\n'
                                 '                        raise ValueError("Voice reference dimension/model '
                                 'mismatch")\n'
                                 '                    scores[name] = float(voice @ profile)\n'
                                 '                variants.append({"source": source, "local_start": crop[0], '
                                 '"local_end": crop[1],\n'
                                 '                                 "scores": scores})\n'
                                 '            hypothesis = resolve_timing_evidence([item["scores"] for item '
                                 'in variants],\n'
                                 '                                                  args.min_similarity, '
                                 'args.min_margin)\n'
                                 '            quality_flags, quality_signals = decoder_quality_flags(\n'
                                 '                decoded["segments"], *(bounds if bounds is not None else '
                                 'alignment_bounds))\n'
                                 '            absolute_start, absolute_end = left+primary[0], '
                                 'left+primary[1]\n'
                                 '            rows.append({"window_index": window["index"], '
                                 '"local_segment_index": local_index,\n'
                                 '                "start": absolute_start, "end": absolute_end, "text": '
                                 'segment["text"],\n'
                                 '                "review_speaker_hypothesis": hypothesis, '
                                 '"review_required": True,\n'
                                 '                "near_window_boundary": primary[0] <= .25 or primary[1] >= '
                                 'window["end"]-left-.25,\n'
                                 '                "evidence": [\n'
                                 '                    {"source": "review_selection", "reasons": '
                                 'window["reasons"]},\n'
                                 '                    {"source": "baseline_overlap", **baseline_evidence(\n'
                                 '                        baseline["segments"], absolute_start, '
                                 'absolute_end)},\n'
                                 '                    {"source": "decoder_support", "flags": quality_flags,\n'
                                 '                     "signals": quality_signals, '
                                 '"word_probability_is_accuracy": False},\n'
                                 '                    {"source": "local_voice", "timing_variants": '
                                 'variants,\n'
                                 '                     "variants_are_independent_votes": False,\n'
                                 '                     "identity_probability_calibrated": False,\n'
                                 '                     "min_similarity": args.min_similarity, "min_margin": '
                                 'args.min_margin}],\n'
                                 '                "alignment_words": [{**word,\n'
                                 '                    **({"start": left+word["start"]} if "start" in word '
                                 'else {}),\n'
                                 '                    **({"end": left+word["end"]} if "end" in word else '
                                 '{})}\n'
                                 '                    for word in segment.get("words", [])]})\n'
                                 '    assert hashlib.sha256(args.baseline.read_bytes()).hexdigest() == '
                                 'baseline_hash\n'
                                 '    result = {"baseline_modified": False, "baseline_sha256": '
                                 'baseline_hash,\n'
                                 '              "selection_fingerprint": fingerprint, "segments": rows}\n'
                                 '    (args.output_dir / '
                                 '"review_hypotheses.json").write_text(json.dumps(result, indent=2)+"\\n")\n'
                                 '    (args.output_dir / "review_transcript.txt").write_text("\\n".join(\n'
                                 '        f"[{row[\'start\']:.2f}-{row[\'end\']:.2f}] '
                                 '{row[\'review_speaker_hypothesis\']}: {row[\'text\']}"\n'
                                 '        for row in rows)+"\\n")\n'
                                 '    counts = {}\n'
                                 '    for row in rows:\n'
                                 '        counts[row["review_speaker_hypothesis"]] = '
                                 'counts.get(row["review_speaker_hypothesis"], 0)+1\n'
                                 '    summary = {"baseline_segment_count": len(baseline["segments"]),\n'
                                 '        "baseline_modified": False, "review_window_count": len(windows),\n'
                                 '        "decoded_review_minutes": sum(w["end"]-w["start"] for w in '
                                 'windows)/60,\n'
                                 '        "review_hypothesis_count": len(rows), "review_label_counts": '
                                 'counts,\n'
                                 '        "duplicate_overlap_hypotheses_retained": True,\n'
                                 '        "note": "Review hypotheses are supplemental and require '
                                 'comparison; no automatic replacement."}\n'
                                 '    (args.output_dir / '
                                 '"comparison_summary.json").write_text(json.dumps(summary, '
                                 'indent=2)+"\\n")\n'
                                 '    print(json.dumps(summary, indent=2))\n'
                                 '\n'
                                 '\n'
                                 'if __name__ == "__main__":\n'
                                 '    main()\n',
 'test_cloud_runtime.py': 'import tempfile\n'
                          'import unittest\n'
                          'from types import SimpleNamespace\n'
                          'from cloud_runtime import StageCache, create_face_analyzer, full_audio_chunks, '
                          'create_full_audio_vad\n'
                          '\n'
                          '\n'
                          'class CloudRuntimeTests(unittest.TestCase):\n'
                          '    def test_cache_reuses_completed_stage_and_isolates_changed_inputs(self):\n'
                          '        with tempfile.TemporaryDirectory() as directory:\n'
                          '            cache = StageCache(directory, {"video": "one", "code": "one"})\n'
                          '            self.assertEqual(cache.get("stage", lambda: {"speaker": '
                          '"SPEAKER_02"}), {"speaker": "SPEAKER_02"})\n'
                          '            repeated = StageCache(directory, {"code": "one", "video": "one"})\n'
                          '            self.assertEqual(repeated.get("stage", lambda: self.fail("stage '
                          'reran")), {"speaker": "SPEAKER_02"})\n'
                          '            self.assertIsNone(StageCache(directory, {"video": "two", "code": '
                          '"one"}).read("stage"))\n'
                          '            self.assertIsNone(StageCache(directory, {"video": "one", "code": '
                          '"two"}).read("stage"))\n'
                          '            self.assertFalse(list(cache.root.glob("*.tmp")))\n'
                          '\n'
                          '    def test_disabled_cache_does_not_save(self):\n'
                          '        cache = StageCache(None, {})\n'
                          '        cache.write("stage", {"ok": True})\n'
                          '        self.assertIsNone(cache.read("stage"))\n'
                          '\n'
                          '    def test_face_provider_selection_and_actual_fallback(self):\n'
                          '        for device, available, expected, context in (\n'
                          '            ("cpu", ["CUDAExecutionProvider", "CPUExecutionProvider"], '
                          '["CPUExecutionProvider"], -1),\n'
                          '            ("cuda", ["CPUExecutionProvider"], ["CPUExecutionProvider"], -1),\n'
                          '            ("cuda", ["CUDAExecutionProvider", "CPUExecutionProvider"], '
                          '["CUDAExecutionProvider", "CPUExecutionProvider"], 0)):\n'
                          '            calls = []\n'
                          '            def factory(**kwargs):\n'
                          '                calls.append(kwargs)\n'
                          '                return SimpleNamespace(prepare=lambda **options: '
                          'calls.append(options),\n'
                          '                    models={"recognition": '
                          'SimpleNamespace(session=SimpleNamespace(get_providers=lambda: '
                          '["CPUExecutionProvider"]))})\n'
                          '            ort = SimpleNamespace(get_available_providers=lambda: available, '
                          'preload_dlls=lambda: None)\n'
                          '            _, actual = create_face_analyzer(device, ort, factory)\n'
                          '            self.assertEqual(calls[0]["providers"], expected)\n'
                          '            self.assertEqual(calls[1]["ctx_id"], context)\n'
                          '            self.assertEqual(actual["recognition"], ["CPUExecutionProvider"])\n'
                          '\n'
                          '\n'
                          '    def test_full_coverage_has_no_gaps_and_bounds_final_window(self):\n'
                          '        chunks = full_audio_chunks(65 * 16000, 16000)\n'
                          '        self.assertEqual(chunks, [{"start": 0, "end": 30}, {"start": 30, "end": '
                          '60}, {"start": 60, "end": 65}])\n'
                          '        for left, right in zip(chunks, chunks[1:]):\n'
                          '            self.assertEqual(left["end"], right["start"])\n'
                          '        self.assertAlmostEqual(full_audio_chunks(16001, 16000)[-1]["end"], '
                          '1.0000625)\n'
                          '        with self.assertRaises(ValueError):\n'
                          '            full_audio_chunks(0, 16000)\n'
                          '\n'
                          '    def test_whisperx_adapter_uses_requested_chunk_size(self):\n'
                          '        import numpy as np\n'
                          '        adapter = create_full_audio_vad()\n'
                          '        audio = np.zeros(25 * 16000)\n'
                          '        self.assertIs(adapter.preprocess_audio(audio), audio)\n'
                          '        detected = adapter({"waveform": audio, "sample_rate": 16000})\n'
                          '        self.assertEqual(adapter.merge_chunks(detected, 10, .5, .36),\n'
                          '            [{"start": 0, "end": 10}, {"start": 10, "end": 20}, {"start": 20, '
                          '"end": 25}])\n'
                          '\n'
                          '    def test_pipeline_reuses_stages_voice_and_visual_evidence(self):\n'
                          '        import contextlib\n'
                          '        import io\n'
                          '        import json\n'
                          '        from pathlib import Path\n'
                          '        from unittest.mock import patch, Mock\n'
                          '        import numpy as np\n'
                          '        import torch\n'
                          '        import chainofrules as pipeline\n'
                          '        with tempfile.TemporaryDirectory() as directory:\n'
                          '            root = Path(directory)\n'
                          '            (root/"video.mp4").write_bytes(b"fake media")\n'
                          '            np.save(root/"voice.npy", np.ones((2, 192), dtype=np.float32))\n'
                          '            np.save(root/"face.npy", np.ones((2, 512), dtype=np.float32))\n'
                          '            records = [{"start": 0., "end": 1., "speaker": "SPEAKER_00"},\n'
                          '                       {"start": 1., "end": 2., "speaker": "SPEAKER_01"}]\n'
                          '            assigned = {"segments": [{"start": 0., "end": 1., "text": "Hello.", '
                          '"speaker": "SPEAKER_00"}]}\n'
                          '            whisper = Mock(); whisper.transcribe.return_value = {"language": '
                          '"en", "segments": []}\n'
                          '            voice = Mock(); voice.encode_batch.return_value = torch.ones((1, 1, '
                          '192))\n'
                          '            detector = Mock(return_value=pipeline.pd.DataFrame(records))\n'
                          '            cap = Mock(); cap.get.return_value = 30\n'
                          '            argv = ["chainofrules.py", str(root/"video.mp4"), "--voice-priors", '
                          'str(root/"voice.npy"),\n'
                          '                    "--face-priors", str(root/"face.npy"), "--output", '
                          'str(root/"result.json"),\n'
                          '                    "--cache-dir", str(root/"cache")]\n'
                          '            with patch("sys.argv", argv), patch.dict("os.environ", {"HF_TOKEN": '
                          '"test-placeholder"}), \\\n'
                          '                 patch.object(pipeline.torch.cuda, "is_available", '
                          'return_value=False), \\\n'
                          '                 patch.object(pipeline.whisperx, "load_audio", '
                          'return_value=np.zeros(32000)), \\\n'
                          '                 patch.object(pipeline.torchaudio, "load", '
                          'return_value=(torch.zeros(1, 32000), 16000)), \\\n'
                          '                 patch.object(pipeline.cv2, "VideoCapture", return_value=cap), '
                          '\\\n'
                          '                 patch.object(pipeline.whisperx, "load_model", '
                          'return_value=whisper) as load, \\\n'
                          '                 patch.object(pipeline.whisperx, "load_align_model", '
                          'return_value=(Mock(), {})) as align_load, \\\n'
                          '                 patch.object(pipeline.whisperx, "align", return_value=assigned), '
                          '\\\n'
                          '                 patch.object(pipeline.whisperx, "assign_word_speakers", '
                          'return_value=assigned), \\\n'
                          '                 patch.object(pipeline, "DiarizationPipeline", '
                          'return_value=detector) as diarize_load, \\\n'
                          '                 patch.object(pipeline.SpeakerRecognition, "from_hparams", '
                          'return_value=voice) as voice_load, \\\n'
                          '                 patch.object(pipeline, "create_face_analyzer", '
                          'return_value=(Mock(), {})), \\\n'
                          '                 patch.object(pipeline, "collect_visual_evidence") as visual, \\\n'
                          '                 contextlib.redirect_stdout(io.StringIO()):\n'
                          '                pipeline.main()\n'
                          '                first = json.loads((root/"result.json").read_text())\n'
                          '                pipeline.main()\n'
                          '                second = json.loads((root/"result.json").read_text())\n'
                          '                self.assertEqual(first, second)\n'
                          '                self.assertEqual(load.call_count, 1)\n'
                          '                self.assertEqual(align_load.call_count, 1)\n'
                          '                self.assertEqual(diarize_load.call_count, 1)\n'
                          '                self.assertEqual(voice.encode_batch.call_count, 2)\n'
                          '                self.assertEqual(visual.call_count, 1)\n'
                          '                self.assertEqual(voice_load.call_args.kwargs["run_opts"], '
                          '{"device": "cpu"})\n'
                          '                self.assertEqual(cap.release.call_count, 2)\n'
                          '                # Coverage changes only ASR/alignment caches, not track '
                          'identity.\n'
                          '                with patch("sys.argv", argv + ["--transcription-coverage", '
                          '"full"]):\n'
                          '                    pipeline.main()\n'
                          '                self.assertEqual(load.call_count, 2)\n'
                          '                self.assertIn("vad_model", load.call_args.kwargs)\n'
                          '                self.assertEqual(align_load.call_count, 2)\n'
                          '                self.assertEqual(diarize_load.call_count, 1)\n'
                          '                self.assertEqual(voice.encode_batch.call_count, 2)\n'
                          '                self.assertEqual(visual.call_count, 1)\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    unittest.main()\n',
 'test_review_regions.py': 'import unittest\n'
                           'from review_transcript_regions import select_review_regions\n'
                           '\n'
                           '\n'
                           'class RegionTests(unittest.TestCase):\n'
                           '    def test_good_long_segment_is_not_selected(self):\n'
                           '        rows = [{"start": 1, "end": 4, "final_speaker": "Target_Speaker",\n'
                           '                 "final_confidence": .9}]\n'
                           '        self.assertEqual(select_review_regions(rows, 5), [])\n'
                           '\n'
                           '    def test_uncertain_short_and_gap_are_selected_without_changing_rows(self):\n'
                           '        rows = [{"start": 5, "end": 5.5, "final_speaker": "Uncertain",\n'
                           '                 "final_confidence": .1},\n'
                           '                {"start": 20, "end": 24, "final_speaker": "SPEAKER_04",\n'
                           '                 "final_confidence": .8}]\n'
                           '        snapshot = [dict(row) for row in rows]\n'
                           '        result = select_review_regions(rows, 30, context=1, minimum_gap=5)\n'
                           '        self.assertEqual(rows, snapshot)\n'
                           '        self.assertTrue(any("uncertain_speaker" in row["reasons"] for row in '
                           'result))\n'
                           '        self.assertTrue(any("transcript_gap" in row["reasons"] for row in '
                           'result))\n'
                           '\n'
                           '    def test_windows_are_bounded_and_external_controls_are_data(self):\n'
                           '        rows = [{"start": 1, "end": 99, "final_speaker": "Uncertain",\n'
                           '                 "final_confidence": 0}]\n'
                           '        result = select_review_regions(rows, 100, context=0, minimum_gap=200,\n'
                           '                                       maximum_window=30, overlap=4,\n'
                           '                                       extra_regions=[(40, 50)])\n'
                           '        self.assertTrue(all(0 < row["end"]-row["start"] <= 30 for row in '
                           'result))\n'
                           '        self.assertTrue(any("external_review_control" in row["reasons"] for row '
                           'in result))\n'
                           '\n'
                           '    def test_invalid_region_is_rejected(self):\n'
                           '        with self.assertRaises(ValueError):\n'
                           '            select_review_regions([], 10, extra_regions=[(9, 11)])\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    unittest.main()\n',
 'test_short_answers.py': '"""Behavior checks for conversational attribution, independent of model '
                          'downloads."""\n'
                          'import unittest\n'
                          'import numpy as np\n'
                          'from chainofrules import (Baseline, Evidence, TimelineSegment, short_voice_crop,\n'
                          '                          add_question_response_evidence, '
                          'add_echo_question_evidence, add_brief_exchange_evidence, resolve_segment, '
                          'collect_visual_evidence)\n'
                          '\n'
                          '\n'
                          'class ShortAnswerTests(unittest.TestCase):\n'
                          '    def question(self, text="Do you have any weapons?", strength=0.9, '
                          'speaker="SPEAKER_00"):\n'
                          '        segment = TimelineSegment(10.0, 12.83, text, Baseline(speaker, speaker))\n'
                          '        segment.final_speaker, segment.final_confidence = speaker, strength\n'
                          '        return segment\n'
                          '\n'
                          '    def reply(self, start=12.89, end=12.99, text="No.", raw="SPEAKER_00"):\n'
                          '        segment = TimelineSegment(start, end, text, Baseline(raw, raw))\n'
                          '        segment.evidence.append(Evidence("local_voice", 0.0, 0.0))\n'
                          '        return segment\n'
                          '\n'
                          '    def infer(self, reply, question, tracks=("SPEAKER_00", "SPEAKER_01"), '
                          'mapping=1.0):\n'
                          '        add_question_response_evidence(reply, question, set(tracks), '
                          '"SPEAKER_01", mapping)\n'
                          '        resolve_segment(reply, "SPEAKER_01", mapping)\n'
                          '        return reply\n'
                          '\n'
                          '    def test_brief_answer_has_weak_alternative_identity(self):\n'
                          '        reply = self.infer(self.reply(), self.question())\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertLessEqual(reply.final_confidence, 0.25)\n'
                          '        self.assertTrue(any("not voice-verified" in reason for reason in '
                          'reply.reasons))\n'
                          '\n'
                          '    def test_target_question_does_not_force_target_answer(self):\n'
                          '        reply = self.infer(self.reply(raw="SPEAKER_01"), '
                          'self.question(speaker="Target_Speaker"))\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_three_speakers_leave_answer_unresolved(self):\n'
                          '        reply = self.infer(self.reply(), self.question(), ("SPEAKER_00", '
                          '"SPEAKER_01", "SPEAKER_02"))\n'
                          '        self.assertEqual(reply.final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_no_question_or_weak_question_leaves_answer_unresolved(self):\n'
                          '        for question in (None, self.question("You are on private property."),\n'
                          '                         self.question("Why are you here?"), '
                          'self.question(strength=0.35)):\n'
                          '            with self.subTest(question=question):\n'
                          '                self.assertEqual(self.infer(self.reply(), '
                          'question).final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_gap_overlap_and_long_answer_are_not_inferred(self):\n'
                          '        for reply in (self.reply(13.5, 13.6), self.reply(12.8, 12.9),\n'
                          '                      self.reply(12.89, 14.5), self.reply(text="No one should be '
                          'doing that.")):\n'
                          '            with self.subTest(reply=reply):\n'
                          '                result = self.infer(reply, self.question())\n'
                          '                self.assertFalse(any(item.source == "question_response" for item '
                          'in result.evidence))\n'
                          '                if reply.end - reply.start < 0.4:\n'
                          '                    self.assertEqual(result.final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_weak_target_mapping_does_not_infer(self):\n'
                          '        self.assertEqual(self.infer(self.reply(), self.question(), '
                          'mapping=0.4).final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_available_voice_is_not_overridden_by_conversation(self):\n'
                          '        reply = self.reply()\n'
                          '        reply.evidence = [Evidence("local_voice", -1.0, 0.5)]\n'
                          '        self.assertEqual(self.infer(reply, self.question()).final_speaker, '
                          '"SPEAKER_00")\n'
                          '\n'
                          '    def test_padded_short_audio_does_not_claim_high_confidence(self):\n'
                          '        reply = self.reply(raw="SPEAKER_01")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.64, 0.28)]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertLessEqual(reply.final_confidence, 0.30)\n'
                          '\n'
                          '    def test_independent_voice_corrects_short_baseline_conflict(self):\n'
                          '        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", '
                          'text="Question")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.57, 0.55,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n'
                          '             "track_similarities": {"SPEAKER_00": 0.31, "SPEAKER_01": 0.16}})]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_poor_profile_match_cannot_correct_identity(self):\n'
                          '        reply = self.reply(start=0.25, end=0.92, raw="SPEAKER_01", '
                          'text="Question")\n'
                          '        reply.evidence = [Evidence("local_voice", -0.8, 0.55,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": 0.15,\n'
                          '             "track_similarities": {"SPEAKER_00": 0.20, "SPEAKER_01": 0.05}})]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "Uncertain")\n'
                          '\n'
                          '    def test_face_presence_alone_does_not_override_voice(self):\n'
                          '        reply = self.reply(start=28.0, end=29.0)\n'
                          '        reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n'
                          '            {"track_margin": 0.03, "track_similarities": {"SPEAKER_00": 0.23, '
                          '"SPEAKER_01": 0.20}}),\n'
                          '            Evidence("target_face_visible", 1.0, 1.0)]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '\n'
                          '    def test_mouth_hint_is_tentative_only_with_ambiguous_profiles(self):\n'
                          '        for margin, expected in ((0.03, "Target_Speaker"), (0.20, '
                          '"SPEAKER_00")):\n'
                          '            reply = self.reply(start=28.0, end=29.0)\n'
                          '            reply.evidence = [Evidence("local_voice", -0.8, 0.7,\n'
                          '                {"track_margin": margin, "track_similarities": {"SPEAKER_00": '
                          '0.23, "SPEAKER_01": 0.20}}),\n'
                          '                Evidence("target_mouth_motion", 1.0, 0.2)]\n'
                          '            resolve_segment(reply, "SPEAKER_01", 1.0)\n'
                          '            self.assertEqual(reply.final_speaker, expected)\n'
                          '            if margin < 0.05:\n'
                          '                self.assertLessEqual(reply.final_confidence, 0.25)\n'
                          '\n'
                          '    def test_face_identity_continues_through_head_turn_but_not_bbox_jump(self):\n'
                          '        class Capture:\n'
                          '            index = 0\n'
                          '            def get(self, prop): return 20\n'
                          '            def set(self, prop, value): self.index = int(value)\n'
                          '            def read(self): return True, np.zeros((200, 200, 3), dtype=np.uint8)\n'
                          '        class Face:\n'
                          '            pass\n'
                          '        for jump in (False, True):\n'
                          '            capture = Capture()\n'
                          '            class Analyzer:\n'
                          '                def get(self, frame):\n'
                          '                    face = Face()\n'
                          '                    face.embedding = np.array([0.7, np.sqrt(1-0.7**2)]) if '
                          'capture.index < 2 else np.array([0.3, np.sqrt(1-0.3**2)])\n'
                          '                    face.bbox = np.array([10,10,80,100]) if not jump or '
                          'capture.index < 2 else np.array([120,120,190,200])\n'
                          '                    points = np.zeros((68,3))\n'
                          '                    points[64,0] = 10\n'
                          '                    points[66,1] = 0.4 if capture.index % 2 else 1.2\n'
                          '                    face.landmark_3d_68 = points\n'
                          '                    return [face]\n'
                          '            reply = self.reply(start=0.5, end=1.4)\n'
                          '            collect_visual_evidence(reply, capture, 8.0, Analyzer(), '
                          'np.array([1.0, 0.0]))\n'
                          '            motion = next(item for item in reply.evidence if item.source == '
                          '"target_mouth_motion")\n'
                          '            self.assertEqual(motion.confidence > 0, not jump)\n'
                          '\n'
                          '    def test_crop_respects_both_neighbors(self):\n'
                          '        following = TimelineSegment(13.01, 15.6, "Next", Baseline("SPEAKER_00", '
                          '"SPEAKER_00"))\n'
                          '        start, end = short_voice_crop(self.reply(), self.question(), following, '
                          '30.0)\n'
                          '        self.assertAlmostEqual(start, 12.83)\n'
                          '        self.assertAlmostEqual(end, 13.01)\n'
                          '        self.assertLess(end-start, 0.4)\n'
                          '\n'
                          '    def test_overlapping_timing_does_not_trim_reply(self):\n'
                          '        preceding = self.question()\n'
                          '        preceding.end = 12.92\n'
                          '        reply = self.reply()\n'
                          '        self.assertEqual(short_voice_crop(reply, preceding, None, 30.0), '
                          '(reply.start, reply.end))\n'
                          '\n'
                          '\n'
                          '    def test_echo_requires_weak_supporting_voice_and_visible_target(self):\n'
                          '        previous = self.question("You are being arrested for criminal '
                          'loitering.")\n'
                          '        reply = self.reply(start=13.3, end=13.88, text="Criminal loitering?")\n'
                          '        reply.evidence = [Evidence("local_voice", -.8, .48,\n'
                          '            {"best_track": "SPEAKER_01", "track_margin": .07,\n'
                          '             "track_similarities": {"SPEAKER_00": .05, "SPEAKER_01": .12}}),\n'
                          '            Evidence("target_face_visible", 0, 0, {"target_visible_hint": '
                          'True})]\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, '
                          '"SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '        reply.evidence = [e for e in reply.evidence if e.source != '
                          '"target_face_visible"]\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertNotEqual(reply.final_speaker, "Target_Speaker")\n'
                          '\n'
                          '    def '
                          'test_echo_does_not_override_clear_voice_or_choose_among_three_people(self):\n'
                          '        previous = self.question("You are being arrested for criminal '
                          'loitering.")\n'
                          '        reply = self.reply(start=13.3, end=13.88, text="Criminal loitering?")\n'
                          '        reply.evidence = [Evidence("local_voice", -1, .6,\n'
                          '            {"best_track": "SPEAKER_00", "track_margin": .4,\n'
                          '             "track_similarities": {"SPEAKER_00": .5, "SPEAKER_01": .1}}),\n'
                          '            Evidence("target_face_visible", 0, 0, {"target_visible_hint": '
                          'True})]\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01"}, '
                          '"SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '        reply.evidence = []\n'
                          '        add_echo_question_evidence(reply, previous, {"SPEAKER_00", "SPEAKER_01", '
                          '"SPEAKER_02"}, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.evidence, [])\n'
                          '\n'
                          '    def test_padded_but_weak_voice_allows_tentative_answer(self):\n'
                          '        reply = self.reply()\n'
                          '        reply.evidence = [Evidence("local_voice", -1, .28,\n'
                          '            {"best_track": "SPEAKER_00", "track_similarities": {"SPEAKER_00": '
                          '.23, "SPEAKER_01": .08}})]\n'
                          '        self.infer(reply, self.question())\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '\n'
                          '\n'
                          '    def test_acknowledgement_requires_independent_agreement(self):\n'
                          '        previous = self.question("Your request has been accepted.")\n'
                          '        reply = self.reply(text="Oh, okay.")\n'
                          '        reply.evidence = [Evidence("local_voice", -.7, .28,\n'
                          '            {"best_track": "SPEAKER_01", "track_similarities": {"SPEAKER_00": 0, '
                          '"SPEAKER_01": .1}})]\n'
                          '        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "Target_Speaker")\n'
                          '        self.assertEqual(reply.final_confidence, .2)\n'
                          '        for strength, best, tracks in ((.6, "SPEAKER_01", {"SPEAKER_00", '
                          '"SPEAKER_01"}),\n'
                          '                                      (.28, "SPEAKER_00", {"SPEAKER_00", '
                          '"SPEAKER_01"}),\n'
                          '                                      (.28, "SPEAKER_01", {"SPEAKER_00", '
                          '"SPEAKER_01", "SPEAKER_02"})):\n'
                          '            reply.evidence = [Evidence("local_voice", -.7, strength,\n'
                          '                {"best_track": best, "track_similarities": {"SPEAKER_00": 0, '
                          '"SPEAKER_01": .1}})]\n'
                          '            add_brief_exchange_evidence(reply, previous, tracks, "SPEAKER_01", '
                          '1)\n'
                          '            self.assertEqual(len(reply.evidence), 1)\n'
                          '\n'
                          '    def test_confirmation_requires_face_baseline_and_no_inference_chain(self):\n'
                          '        previous = self.question("Really?", .30, "SPEAKER_01")\n'
                          '        previous.final_speaker = "Target_Speaker"\n'
                          '        previous.evidence = [Evidence("target_face_visible", 0, 0, '
                          '{"target_visible_hint": True})]\n'
                          '        reply = self.reply(text="Yeah.", raw="SPEAKER_01")\n'
                          '        voice = Evidence("local_voice", -1, .15,\n'
                          '            {"best_track": "SPEAKER_00", "track_similarities": {"SPEAKER_00": '
                          '.06, "SPEAKER_01": .02}})\n'
                          '        reply.evidence = [voice]\n'
                          '        add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '        resolve_segment(reply, "SPEAKER_01", 1)\n'
                          '        self.assertEqual(reply.final_speaker, "SPEAKER_00")\n'
                          '        self.assertEqual(reply.final_confidence, .20)\n'
                          '        for face, reasons in (([], []), (previous.evidence, ["weak '
                          'question/answer inference"])):\n'
                          '            previous.evidence, previous.reasons = face, reasons\n'
                          '            reply.evidence = [voice]\n'
                          '            add_brief_exchange_evidence(reply, previous, {"SPEAKER_00", '
                          '"SPEAKER_01"}, "SPEAKER_01", 1)\n'
                          '            self.assertEqual(len(reply.evidence), 1)\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    unittest.main()\n',
 'test_transcript_gaps.py': 'import unittest\n'
                            'from recover_transcript_gaps import '
                            'uncovered_intervals,recovery_windows,collect_candidates,preserve_baseline\n'
                            'class GapTests(unittest.TestCase):\n'
                            ' def test_union_handles_overlaps_and_edges(self):\n'
                            '  '
                            "self.assertEqual(uncovered_intervals([{'start':2,'end':5},{'start':4,'end':8},{'start':12,'end':14}],17),[(0.,2.),(8.,12.),(14.,17)])\n"
                            ' def test_windows_cover_gap_without_tiny_tail(self):\n'
                            '  windows=recovery_windows((10,30.00001),40,size=8,overlap=4)\n'
                            '  self.assertTrue(all(0<r-l<=8.000001 for l,r in '
                            'windows));self.assertEqual(windows[0][0],9.5);self.assertEqual(windows[-1][1],30.50001)\n'
                            '  self.assertTrue(all(b[0]<=a[1] for a,b in zip(windows,windows[1:])))\n'
                            ' def test_context_is_bounded_and_invalid_context_rejected(self):\n'
                            '  windows=recovery_windows((1,39),40,size=20,overlap=10,context=2)\n'
                            '  self.assertEqual(windows[0][0],0);self.assertEqual(windows[-1][1],40)\n'
                            "  for context in (-1,float('nan'),float('inf')):\n"
                            '   with '
                            'self.assertRaises(ValueError):recovery_windows((10,30),40,context=context)\n'
                            ' def test_repetition_needs_distinct_windows_and_stays_in_gap(self):\n'
                            '  '
                            "words=[{'start':4,'end':4.5,'word':'Height?','probability':.9,'window_index':0},{'start':4.1,'end':4.6,'word':'height','probability':.8,'window_index':1},{'start':1,'end':2,'word':'existing','probability':1,'window_index':1}]\n"
                            '  '
                            "candidates=collect_candidates(words,(3,6));self.assertEqual(len(candidates),1);self.assertTrue(candidates[0]['repeated_in_overlapping_windows']);self.assertTrue(candidates[0]['review_required'])\n"
                            ' def test_candidate_does_not_splice_disagreeing_windows(self):\n'
                            '  '
                            "words=[{'start':4,'end':4.2,'word':'one','probability':.9,'window_index':0},{'start':4.3,'end':4.5,'word':'answer','probability':.6,'window_index':0},{'start':4,'end':4.2,'word':'another','probability':.6,'window_index':1},{'start':4.3,'end':4.5,'word':'answer','probability':.9,'window_index':1}]\n"
                            '  '
                            "candidates=collect_candidates(words,(3,6));self.assertEqual(len(candidates),1);self.assertEqual(len({w['window_index'] "
                            "for w in candidates[0]['words']}),1)\n"
                            ' def test_zero_duration_word_kept_with_phrase_not_invented_timing(self):\n'
                            "  words=[{'start':4,'end':4.5,'word':' "
                            "How','probability':.9,'window_index':0},{'start':4.5,'end':4.5,'word':' "
                            "tall?','probability':.9,'window_index':0}]\n"
                            '  '
                            "result=collect_candidates(words,(3,6));self.assertEqual(result[0]['text'],'How "
                            "tall?');self.assertEqual(result[0]['words'][1]['start'],result[0]['words'][1]['end'])\n"
                            '  self.assertEqual(collect_candidates(words[1:],(3,6)),[])\n'
                            ' def test_keeps_baseline_nested_fields_and_identity_unchanged(self):\n'
                            '  '
                            "original={'segments':[{'start':0,'end':1,'text':'works','final_speaker':'Target_Speaker','words':[{'word':'works'}],'reasons':['original']}]}\n"
                            '  '
                            "result=preserve_baseline(original,[{'text':'new','speaker':'Uncertain'}]);self.assertEqual(result['segments'],original['segments']);result['segments'][0]['words'][0]['word']='mutated';self.assertEqual(original['segments'][0]['words'][0]['word'],'works')\n"
                            "if __name__=='__main__':unittest.main()\n",
 'test_window_review.py': 'import unittest\n'
                          'from review_audio_window import '
                          'baseline_evidence,resolve_voice,decoder_sentence_bounds,resolve_timing_evidence,decoder_quality_flags\n'
                          'class WindowTests(unittest.TestCase):\n'
                          '    def test_confident_beep_words_do_not_prove_speech(self):\n'
                          '        '
                          "s=[{'start':0,'end':4.2,'no_speech_prob':.808,'avg_logprob':-.584,'words':[{'probability':.94}]}]\n"
                          '        flags,_=decoder_quality_flags(s,1,2)\n'
                          '        self.assertTrue(flags)\n'
                          '    def test_unrelated_decoder_warning_not_applied(self):\n'
                          '        '
                          "flags,_=decoder_quality_flags([{'start':0,'end':1,'no_speech_prob':.9}],3,4)\n"
                          '        self.assertEqual(flags,[])\n'
                          '    def test_consistent_crops_with_one_qualified_match(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.34,'Other':.18},{'Target':.27,'Other':.21}]),'Target')\n"
                          '    def test_conflicting_crops_do_not_choose_the_stronger_match(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.7,'Other':.1},{'Target':.1,'Other':.3}]),'Uncertain')\n"
                          '    def test_two_weak_crops_do_not_accumulate_confidence(self):\n'
                          '        '
                          "self.assertEqual(resolve_timing_evidence([{'Target':.2,'Other':.1},{'Target':.21,'Other':.1}]),'Uncertain')\n"
                          '    def test_decoder_words_follow_repeated_sentences_in_order(self):\n'
                          "        d=[{'text':'I agree. I "
                          "agree.','words':[{'word':'I','start':0,'end':.1},{'word':'agree.','start':.1,'end':1},{'word':'I','start':2,'end':2.1},{'word':'agree.','start':2.1,'end':3}]}]\n"
                          "        a=[{'text':'I agree.'},{'text':'I agree.'}]\n"
                          '        self.assertEqual(decoder_sentence_bounds(d,a),[(0,1),(2,3)])\n'
                          '    def test_missing_word_links_do_not_invent_timings(self):\n'
                          '        '
                          "self.assertEqual(decoder_sentence_bounds([{'text':'Hi.'}],[{'text':'Hi.'}]),[None])\n"
                          '    def test_raw_ids_preserved_and_offsets_applied(self):\n'
                          '        '
                          "a=[{'start':0,'end':2,'raw_speaker_track':'SPEAKER_03','final_speaker':'Target_Speaker'},{'start':2,'end':4,'raw_speaker_track':'SPEAKER_07','final_speaker':'SPEAKER_07'}]\n"
                          '        result=baseline_evidence(a,101,103,offset=100)\n'
                          '        '
                          "self.assertEqual(result['raw_track_overlap_seconds'],{'SPEAKER_03':1,'SPEAKER_07':1})\n"
                          "        self.assertEqual(a[0]['raw_speaker_track'],'SPEAKER_03')\n"
                          '    def test_current_pipeline_nested_baseline_schema(self):\n'
                          '        '
                          "s=[{'start':0,'end':2,'baseline':{'raw_speaker_track':'SPEAKER_08','speaker':'SPEAKER_08'},'final_speaker':'SPEAKER_08'}]\n"
                          '        '
                          "self.assertEqual(baseline_evidence(s,0,1)['raw_track_overlap_seconds'],{'SPEAKER_08':1})\n"
                          '    def test_no_observation_is_not_negative_identity_evidence(self):\n'
                          "        self.assertEqual(baseline_evidence([],0,2)['segments'],[])\n"
                          "        self.assertEqual(resolve_voice({}),'Uncertain')\n"
                          '    def test_short_response_cannot_borrow_questioners_identity(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.22,'Officer':.21}),'Uncertain')\n"
                          '    def test_close_profiles_abstain_even_if_similarity_is_high(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.6,'Other':.57}),'Uncertain')\n"
                          '    def test_strong_separated_profile_produces_review_hypothesis(self):\n'
                          '        '
                          "self.assertEqual(resolve_voice({'Target_Speaker':.5,'Other':.1}),'Target_Speaker')\n"
                          "if __name__=='__main__':unittest.main()\n"}
for name, content in EMBEDDED_FILES.items():
    (WORK/name).write_text(content)

# Strip notebook-only display settings from every child process.
ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['MPLCONFIGDIR'] = str(BASE/'matplotlib')
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
ENV['PYTHONUNBUFFERED'] = '1'
ENV.pop('PYTHONPATH', None)

def checked(args, **kwargs):
    return subprocess.run(args, env=ENV, check=True, **kwargs)

print('1/4: Creating/checking environment', flush=True)
ready = False
if Path(PYTHON).exists():
    ready = subprocess.run([PYTHON, '-m', 'pip', '--version'], env=ENV,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    # Install bootstrap tools outside the runtime, without replacing notebook packages.
    bootstrap = BASE/'bootstrap-tools'
    checked([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap),
             'virtualenv>=20.26,<21', 'wrapt'])
    bootstrap_env = ENV.copy()
    bootstrap_env['PYTHONPATH'] = str(bootstrap)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-download', str(VENV)],
                   env=bootstrap_env, check=True)
checked([PYTHON, '-m', 'pip', '--version'])
print('2/4: Installing pipeline dependencies (including wrapt in the venv)', flush=True)
requirements = [
    'whisperx==3.8.6', 'speechbrain==1.1.1', 'insightface==2.0',
    'torch==2.8.0', 'torchaudio==2.8.0', 'numpy==2.5.3',
    'opencv-python==5.0.0.93', 'onnxruntime-gpu==1.23.2', 'wrapt']
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', *requirements])
# InsightFace's metadata requires the CPU-named distribution, even though GPU
# supplies the same import. Check dependency resolution before removing that overlap.
print('3/4: Removing overlapping ONNX packages and installing CUDA 12 build', flush=True)
checked([PYTHON, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'])
checked([PYTHON, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
         'onnxruntime-gpu==1.23.2'])
if shutil.which('ffmpeg') is None:
    raise RuntimeError('Install ffmpeg on this laptop and rerun setup. Kaggle normally includes it.')
library_dirs = subprocess.check_output([PYTHON, '-c',
    "import site,pathlib; print(':'.join(str(p) for d in site.getsitepackages() for p in pathlib.Path(d).glob('nvidia/*/lib')))"], env=ENV, text=True).strip()
ENV['LD_LIBRARY_PATH'] = library_dirs + ':' + ENV.get('LD_LIBRARY_PATH', '')
print('4/4: Verifying imports, GPU and behavior checks', flush=True)
verification = """import torch,onnxruntime as ort,wrapt
import chainofrules
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (local CPU)')
print('ONNX:', ort.__version__, ort.__file__)
print('Advertised providers:', ort.get_available_providers())
"""
if ON_KAGGLE:
    verification += "assert torch.cuda.is_available(), 'Kaggle GPU is not available to this runtime'\nassert 'CUDAExecutionProvider' in ort.get_available_providers(), 'GPU ONNX package missing'"
checked([PYTHON, '-c', verification], cwd=WORK)
checked([PYTHON, '-m', 'unittest', 'test_cloud_runtime', 'test_short_answers'], cwd=WORK)
print('Setup complete. Continue to credentials and tests.', flush=True)

import base64, hashlib
REFERENCE_FILES = {'face_embeddings.npy': '...',
 'opening_officer_reference.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE5MiwpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIApDR569eD+ePSVTQjzXL8Q8mFepvQvxDz3M/vy9YNoYvbW3or2vrIm9B/1hvZUsuD3xL569DlK0vZXXV76CoP49pTbJvXm8+D0rOMO7dQlDPgHbrb32ZTq9CKD5vQ9U37tiJZm9awdSPA8JgL2s9lw9o0yAPS0WeT0kG5e9Qyscvnywsr0gUcM9MYe9PWgwhT2T9/+8dN+VPE0xhL0qjMe9kvxuPYCZDL4TrUS9GluQPZgFOD1fWoc9hl6NPSuuzj07pmY9G28RvUyrVrx3J4o9nemUvf1lCb2RV1O9samYOx9Kwz3HZcS83eeqPRlqHrx3YXi9tTCWvIBTfT3P+nq97hzyu3W4qT1uji49wXiDPatGqT3xJ6Y9+7ytvRKQeD0oG5W9AEVSO0Rk2z0s18S8yngZvqcF+TwuJk07nuqgvFu6i7zfPdk8XlA7vIzIU72kDnQ7LuPhPflzkTy7ZaU9SQprvOXkfz2Rcaa971UaveED+r1wB3E9oWuXvfqJ6b04ZL+8glvUvf9vVzvUXMy9PJmRvGJhQz04Ou49Me+yvV+DsL3XHok97akwOyr1q7x9JoE8t7utvXZgrT1+4sS97rJQvWCahrzGHao9BmubPWbicbydw8o9UoElPaMZxT2Kjim8lFDIPAeF873jfsk9KHSFPQCjizwX4sw8R/MEPahYvr1tTkC9i+NzPR4WEz0djJY8bPG+PL661j2XY0S8ZmstPcr++r0yOwc8hiOaPAC8ar3XBI68ty0JvAQC2LyIdd69/njeu/eipT3X4pE9lrucvL/j3D0f7uA9prS+PRFYW71r4v+7YbtYPRYMaD09C4u8QvxFPX7KIb1GAbE99asPvp3whr3hlR46/mkAPvSJrTyyiaW8tCXrPKVQu7xzO7+96JEzPevqjjydtB6+tr9tPdb7Kr2T0hs9InaLPXhmhb1qw0Y9pWg+vdvlEr2TkHA9JC7gvdhyML0Dtlc95fy6PVnmz71jg+O9MjXBPethZz0e+hg9OvCCPXPA3ro=',
 'reference.json': 'ewogICJ2b2ljZSI6IHsKICAgICJhbmNob3Jfcm93IjogNSwKICAgICJhY2NlcHRlZF9yb3dzIjogWwogICAgICAwLAogICAgICAxLAogICAgICAyLAogICAgICAzLAogICAgICA0LAogICAgICA1LAogICAgICA2LAogICAgICA3LAogICAgICA4LAogICAgICA5LAogICAgICAxMCwKICAgICAgMTIsCiAgICAgIDEzLAogICAgICAxNCwKICAgICAgMTUsCiAgICAgIDE2LAogICAgICAxNywKICAgICAgMTgKICAgIF0sCiAgICAiZXhjbHVkZWRfcm93cyI6IFsKICAgICAgMTEKICAgIF0sCiAgICAic2ltaWxhcml0eV90b19hbmNob3IiOiB7CiAgICAgICIwIjogMC42NTg0ODI3ODk5OTMyODYxLAogICAgICAiMSI6IDAuNjU5MzkzNzg3Mzg0MDMzMiwKICAgICAgIjIiOiAwLjY4ODUyODQ3ODE0NTU5OTQsCiAgICAgICIzIjogMC42NzQyNzE4MjE5NzU3MDgsCiAgICAgICI0IjogMC43NzYxNTQzOTg5MTgxNTE5LAogICAgICAiNSI6IDEuMCwKICAgICAgIjYiOiAwLjc5MTA4Mzk5MTUyNzU1NzQsCiAgICAgICI3IjogMC42NjE5NjQ0NzYxMDg1NTEsCiAgICAgICI4IjogMC43NDI3Mzg3MjM3NTQ4ODI4LAogICAgICAiOSI6IDAuNzkwNDg1NjIwNDk4NjU3MiwKICAgICAgIjEwIjogMC43NzM1MjgyNzc4NzM5OTI5LAogICAgICAiMTEiOiAwLjIzODY0NTA0Njk0OTM4NjYsCiAgICAgICIxMiI6IDAuNzAzMjMyNTI2Nzc5MTc0OCwKICAgICAgIjEzIjogMC42NzExODE3MzgzNzY2MTc0LAogICAgICAiMTQiOiAwLjYwMDczMTEzNDQxNDY3MjksCiAgICAgICIxNSI6IDAuNjU5NjEzOTY2OTQxODMzNSwKICAgICAgIjE2IjogMC40NzI5MjMxMjk3OTY5ODE4LAogICAgICAiMTciOiAwLjYwODAxNTY1NjQ3MTI1MjQsCiAgICAgICIxOCI6IDAuNjAzNTE5NTU4OTA2NTU1MgogICAgfSwKICAgICJtaW5pbXVtX3NpbWlsYXJpdHkiOiAwLjQ1LAogICAgInNhbXBsZV9wcm92ZW5hbmNlIjogWwogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PWpQZWJKd3BJazRFIiwKICAgICAgICAic3RhcnQiOiAwLjAsCiAgICAgICAgImVuZCI6IDQuMCwKICAgICAgICAic3BlZWNoX2ZyYWN0aW9uIjogMS4wCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9alBlYkp3cElrNEUiLAogICAgICAgICJzdGFydCI6IDQuMCwKICAgICAgICAiZW5kIjogOC4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInN0YXJ0IjogOC4wLAogICAgICAgICJlbmQiOiAxMi4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInN0YXJ0IjogMjQuMCwKICAgICAgICAiZW5kIjogMjguMCwKICAgICAgICAic3BlZWNoX2ZyYWN0aW9uIjogMS4wCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9c0FrZWdjMGZwUzQiLAogICAgICAgICJzdGFydCI6IDAuMCwKICAgICAgICAiZW5kIjogNC4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInN0YXJ0IjogNC4wLAogICAgICAgICJlbmQiOiA4LjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAic3RhcnQiOiA4LjAsCiAgICAgICAgImVuZCI6IDEyLjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAic3RhcnQiOiAxMi4wLAogICAgICAgICJlbmQiOiAxNi4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInN0YXJ0IjogMTYuMCwKICAgICAgICAiZW5kIjogMjAuMCwKICAgICAgICAic3BlZWNoX2ZyYWN0aW9uIjogMS4wCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9c0FrZWdjMGZwUzQiLAogICAgICAgICJzdGFydCI6IDIwLjAsCiAgICAgICAgImVuZCI6IDI0LjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAic3RhcnQiOiAyNC4wLAogICAgICAgICJlbmQiOiAyOC4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInN0YXJ0IjogMjguMCwKICAgICAgICAiZW5kIjogMzAuMCwKICAgICAgICAic3BlZWNoX2ZyYWN0aW9uIjogMS4wCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9QVRpdzQxQlQybzQiLAogICAgICAgICJzdGFydCI6IDAuMCwKICAgICAgICAiZW5kIjogNC4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInN0YXJ0IjogNC4wLAogICAgICAgICJlbmQiOiA4LjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PUFUaXc0MUJUMm80IiwKICAgICAgICAic3RhcnQiOiA4LjAsCiAgICAgICAgImVuZCI6IDEyLjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PUFUaXc0MUJUMm80IiwKICAgICAgICAic3RhcnQiOiAxMi4wLAogICAgICAgICJlbmQiOiAxNi4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInN0YXJ0IjogMTYuMCwKICAgICAgICAiZW5kIjogMjAuMCwKICAgICAgICAic3BlZWNoX2ZyYWN0aW9uIjogMS4wCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9QVRpdzQxQlQybzQiLAogICAgICAgICJzdGFydCI6IDIwLjAsCiAgICAgICAgImVuZCI6IDI0LjAsCiAgICAgICAgInNwZWVjaF9mcmFjdGlvbiI6IDEuMAogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PUFUaXc0MUJUMm80IiwKICAgICAgICAic3RhcnQiOiAyNC4wLAogICAgICAgICJlbmQiOiAyOC4wLAogICAgICAgICJzcGVlY2hfZnJhY3Rpb24iOiAxLjAKICAgICAgfQogICAgXQogIH0sCiAgImZhY2UiOiB7CiAgICAiYW5jaG9yX3JvdyI6IDYsCiAgICAiYWNjZXB0ZWRfcm93cyI6IFsKICAgICAgMCwKICAgICAgMSwKICAgICAgMiwKICAgICAgMywKICAgICAgNCwKICAgICAgNSwKICAgICAgNiwKICAgICAgNywKICAgICAgOCwKICAgICAgOSwKICAgICAgMTAsCiAgICAgIDExLAogICAgICAxMywKICAgICAgMTQsCiAgICAgIDE1LAogICAgICAxNiwKICAgICAgMTcsCiAgICAgIDE4LAogICAgICAxOSwKICAgICAgMjAsCiAgICAgIDIxLAogICAgICAyMiwKICAgICAgMjMsCiAgICAgIDI0LAogICAgICAyNSwKICAgICAgMjYsCiAgICAgIDI3LAogICAgICAyOCwKICAgICAgMjksCiAgICAgIDMwLAogICAgICAzMSwKICAgICAgMzIsCiAgICAgIDMzCiAgICBdLAogICAgImV4Y2x1ZGVkX3Jvd3MiOiBbCiAgICAgIDEyCiAgICBdLAogICAgInNpbWlsYXJpdHlfdG9fYW5jaG9yIjogewogICAgICAiMCI6IDAuOTMwMjE1OTU0NzgwNTc4NiwKICAgICAgIjEiOiAwLjcyMjAzMDc1ODg1NzcyNywKICAgICAgIjIiOiAwLjkyMjMzODg0MzM0NTY0MjEsCiAgICAgICIzIjogMC45MzAwNjM2NjQ5MTMxNzc1LAogICAgICAiNCI6IDAuOTM5MDM1NzczMjc3MjgyNywKICAgICAgIjUiOiAwLjk0ODk5MDM0NTAwMTIyMDcsCiAgICAgICI2IjogMS4wLAogICAgICAiNyI6IDAuOTU0MTI1NDA0MzU3OTEwMiwKICAgICAgIjgiOiAwLjY3NjMyMjQ2MDE3NDU2MDUsCiAgICAgICI5IjogMC43NDEzNzYxNjE1NzUzMTc0LAogICAgICAiMTAiOiAwLjc1NDA2ODM3NDYzMzc4OTEsCiAgICAgICIxMSI6IDAuNzgxMTYzODcxMjg4Mjk5NiwKICAgICAgIjEyIjogMC40NDk3NzQwODY0NzUzNzIzLAogICAgICAiMTMiOiAwLjc1MjA1MTE3NDY0MDY1NTUsCiAgICAgICIxNCI6IDAuNzAxMDIyODYzMzg4MDYxNSwKICAgICAgIjE1IjogMC43NDAyMTg5MzczOTcwMDMyLAogICAgICAiMTYiOiAwLjY5OTE3MjczNTIxNDIzMzQsCiAgICAgICIxNyI6IDAuNzc0MDk1MTE4MDQ1ODA2OSwKICAgICAgIjE4IjogMC43MDI1MzEwOTkzMTk0NTgsCiAgICAgICIxOSI6IDAuNzIyODc1ODMzNTExMzUyNSwKICAgICAgIjIwIjogMC42ODAzMTM1ODcxODg3MjA3LAogICAgICAiMjEiOiAwLjY4ODgyMjM4ODY0ODk4NjgsCiAgICAgICIyMiI6IDAuNTk5NDA3ODUxNjk2MDE0NCwKICAgICAgIjIzIjogMC41Mjg0OTYxNDYyMDIwODc0LAogICAgICAiMjQiOiAwLjYxNzk4NjMyMTQ0OTI3OTgsCiAgICAgICIyNSI6IDAuNjA0OTg0NTgxNDcwNDg5NSwKICAgICAgIjI2IjogMC41OTAzNDA3MzM1MjgxMzcyLAogICAgICAiMjciOiAwLjYxMTAyMjgzMDAwOTQ2MDQsCiAgICAgICIyOCI6IDAuNTk0MzI4NzAxNDk2MTI0MywKICAgICAgIjI5IjogMC41Mzk4ODE3NjU4NDI0Mzc3LAogICAgICAiMzAiOiAwLjYyMzg0OTUxMTE0NjU0NTQsCiAgICAgICIzMSI6IDAuNjIzNzE2NTkyNzg4Njk2MywKICAgICAgIjMyIjogMC42MjA1MTAzMzk3MzY5Mzg1LAogICAgICAiMzMiOiAwLjU3NTA3MDUwMDM3Mzg0MDMKICAgIH0sCiAgICAibWluaW11bV9zaW1pbGFyaXR5IjogMC40NSwKICAgICJzYW1wbGVfcHJvdmVuYW5jZSI6IFsKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDAuMCwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDIuNSwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDUuMCwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDcuNSwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDEwLjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9alBlYkp3cElrNEUiLAogICAgICAgICJ0aW1lc3RhbXAiOiAxMi41LAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PWpQZWJKd3BJazRFIiwKICAgICAgICAidGltZXN0YW1wIjogMTUuMCwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDE3LjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9alBlYkp3cElrNEUiLAogICAgICAgICJ0aW1lc3RhbXAiOiAyMC4wLAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PWpQZWJKd3BJazRFIiwKICAgICAgICAidGltZXN0YW1wIjogMjIuNSwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgInRpbWVzdGFtcCI6IDI1LjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9alBlYkp3cElrNEUiLAogICAgICAgICJ0aW1lc3RhbXAiOiAyNy41LAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAidGltZXN0YW1wIjogMC4wLAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAidGltZXN0YW1wIjogNy41LAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAidGltZXN0YW1wIjogMTAuMCwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDEyLjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9c0FrZWdjMGZwUzQiLAogICAgICAgICJ0aW1lc3RhbXAiOiAxNS4wLAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAidGltZXN0YW1wIjogMTcuNSwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDIwLjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9c0FrZWdjMGZwUzQiLAogICAgICAgICJ0aW1lc3RhbXAiOiAyMi41LAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAidGltZXN0YW1wIjogMjUuMCwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1zQWtlZ2MwZnBTNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDI3LjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDEKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9QVRpdzQxQlQybzQiLAogICAgICAgICJ0aW1lc3RhbXAiOiAwLjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDIuNSwKICAgICAgICAicm9pIjogWwogICAgICAgICAgMCwKICAgICAgICAgIDAsCiAgICAgICAgICAxLAogICAgICAgICAgMC41CiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInNvdXJjZSI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PUFUaXc0MUJUMm80IiwKICAgICAgICAidGltZXN0YW1wIjogNS4wLAogICAgICAgICJyb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAwLjUKICAgICAgICBdCiAgICAgIH0sCiAgICAgIHsKICAgICAgICAic291cmNlIjogImh0dHBzOi8vd3d3LnlvdXR1YmUuY29tL3dhdGNoP3Y9QVRpdzQxQlQybzQiLAogICAgICAgICJ0aW1lc3RhbXAiOiA3LjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDEwLjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDEyLjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDE1LjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDE3LjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDIwLjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDIyLjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDI1LjAsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfSwKICAgICAgewogICAgICAgICJzb3VyY2UiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1BVGl3NDFCVDJvNCIsCiAgICAgICAgInRpbWVzdGFtcCI6IDI3LjUsCiAgICAgICAgInJvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfQogICAgXQogIH0sCiAgIm1ldGhvZCI6ICJtYWpvcml0eS1tZWRvaWQgY29uc2lzdGVuY3kgc2NyZWVuaW5nOyB1bnZlcmlmaWVkIGNhbmRpZGF0ZSIsCiAgImV2YWx1YXRpb25fYXVkaW9fdXNlZF9mb3JfZW5yb2xsbWVudCI6IGZhbHNlLAogICJzb3VyY2VfbWFuaWZlc3QiOiB7CiAgICAic291cmNlcyI6IFsKICAgICAgewogICAgICAgICJwYXRoIjogInNvdXJjZTFfaDI2NC5tcDQiLAogICAgICAgICJ1cmwiOiAiaHR0cHM6Ly93d3cueW91dHViZS5jb20vd2F0Y2g/dj1qUGViSndwSWs0RSIsCiAgICAgICAgIm9yaWdpbmFsX3N0YXJ0X3NlY29uZHMiOiAwLAogICAgICAgICJvZmZzZXRfc2Vjb25kcyI6IDAsCiAgICAgICAgImR1cmF0aW9uX3NlY29uZHMiOiAzMCwKICAgICAgICAiZmFjZV9yb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInBhdGgiOiAic291cmNlMl9oMjY0Lm1wNCIsCiAgICAgICAgInVybCI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PXNBa2VnYzBmcFM0IiwKICAgICAgICAib3JpZ2luYWxfc3RhcnRfc2Vjb25kcyI6IDEwLAogICAgICAgICJvZmZzZXRfc2Vjb25kcyI6IDAsCiAgICAgICAgImR1cmF0aW9uX3NlY29uZHMiOiAzMCwKICAgICAgICAiZmFjZV9yb2kiOiBbCiAgICAgICAgICAwLAogICAgICAgICAgMCwKICAgICAgICAgIDEsCiAgICAgICAgICAxCiAgICAgICAgXQogICAgICB9LAogICAgICB7CiAgICAgICAgInBhdGgiOiAic291cmNlM19oMjY0Lm1wNCIsCiAgICAgICAgInVybCI6ICJodHRwczovL3d3dy55b3V0dWJlLmNvbS93YXRjaD92PUFUaXc0MUJUMm80IiwKICAgICAgICAib3JpZ2luYWxfc3RhcnRfc2Vjb25kcyI6IDAsCiAgICAgICAgIm9mZnNldF9zZWNvbmRzIjogMCwKICAgICAgICAiZHVyYXRpb25fc2Vjb25kcyI6IDMwLAogICAgICAgICJmYWNlX3JvaSI6IFsKICAgICAgICAgIDAsCiAgICAgICAgICAwLAogICAgICAgICAgMSwKICAgICAgICAgIDAuNQogICAgICAgIF0KICAgICAgfQogICAgXQogIH0KfQo=',
 'voice_embeddings.npy': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE4LCAxOTIpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIApJis07uuwKPh5teLzOIL69MG+MPGhJpr0YthU8EmJxu7NtmD1b79y8wHhevQWwDr2r79M7hh/BPb/HiL33XKS8KYP+PU6IcTz8W2+9MlvbPQ7uRj3+ere9leYdPJst87uml009qjCRPYrcP7xy/BU9L1zmvUD9n7skUeA85cuoPUEEzrsM7L09RR8ZvJm0wzxRwx29G2MOPbuNjT2JqUC9zkrUvFhgZDwwKSC93AAdPWtd67xfM/M9H3SMvRU/4bv8MI69raZIPmCBdr2jbEq86iTEvQsqbDobspA9+599PUdlLb2xvSA9H8oIPo5umzxbf6a9ILg2PAQQiT3EghU9TMJFPQNcu7yeNHo+XD33vMllWryeYSW++6YwvdOKaT02YuA8V4ZHvT+Riz2Axzm9R13ePF5Pl72KmTA9qEIsu8AJVr1LJMi7RcpsuxDnsTuifME85bypPZeEJbwleMU9d9emPetSCbzDzbs9QrUDPudvCj4hPTe5ALQevYUpED3NPki9whYPvSFoCz1+59S8xMFauxqPJj1w/lc97l0LvaF9mrzow0Q8n2ETvdaDob232807KHP/PHE/3Lx4ToM8KtcLPQJVqLx+aZS9M3+hPSjpuT3lf+M9XXRZPYow7DzAWbK9DftDPS/FWj1O3wg7daqJPKB0sL3eXxW9De8KvtLyjr3cyl+96SWTvUJFC74IWsa7PRUSvUEECL314ri9jfgSPbO+s70JyZU9BtMgPkYEar20sOo8DWJAPkAO1r1BDDc9bEggPlBpBr2/kGO9I239vUCeLL2crc+9HIDovG6cozsam8052ryxvXfW3Tw4FvK6UYnFPRxLJroq/bo9KPyaPTmUBz30ZNO9b2evPAazaL23Jtu8U744vdQ5Mz6hP7093JTDvapiMT2UbOG8rMvWvWlYOz6X5OW94jxpvBivMz09nNK9b24UPK134LwqSjW+xRodvUowoj1yq3481Df2vJizLrwt6ou9yFNAveAVHb3BUyG+OAXUvUOoYjwZ2Go8vwOcPVvSXLzluL+9+MoUvUqCt7wnwUA9I9OCPPLVdT2qqua8FxDzvUgW6bsQFAe8IvrJPf9uVr2+9O68biYWPpOFYL1UAbm8yKOzPVvc2jx0joC9jMyoPANPdzsCNLW79rxEPWqNKL0C0+291ipFvpo8uD15M4A9nGHWPaTnpbp68z281Do1vaYWQT3ilRO+aOnHO4gGM7vpzB89rp6jvdoHRLwlP2y9LcHiPMPcw7y/UnA9Wa2ZPR9Sz7zhIYm9IWMbPv6YcL2ja7y8l6+Cu9T2Pb3oI0A94dCpPSk6xDvfpg67/gCvPe/GRD2yKX+8pLAyPYnGHT7VQBc9xQsqPSI1iL0xHSE+loejveAzNDpgqxW+099wPdFF7ryYooo6fY+jvXE6qz3CaWk8F6fUvXBZpLy+KY48UL+RPTZW17yIFhe9asVsPBxMGTwg86c9QeFIPWkuqby0p449cCF9PexXrL33BWy9k52rPd51Jj1X4Le9/0sVPHVibz1Co7g7heHwveH8Nr1CE6q9eSRyPXQJpTx80ZU96cLovP3uJL7k9D67NHfvOsBfs70/a7U9HjukPSMo3rxuTVc8SwfIPbonIj01epO9EqrcO/UWEz3l1dQ9aygFvS2VGz2t7OC9n0yiPUSuGzyGySM9ZjBGPV56Br5eTKi82eexvflxLbxr03K98js2vc0jCL3LNK28ZHqHvTv7pj3WqOW9b8YsvBmyhr1WbTc+2AUrPqT6pLw19ak951M/Ptiqw72GgAI9KtPWPUHWhzyiB7u8oS+DvcpcWr3EQ1m9e9xcvb6cSj0TNqG9N2j7vFeFPz36JBs9Wf8SPuPuZjuUi/Q9SiA0PHRHT72m8R+9kidGPfHbYzvFRPE5unOkvX/XiT3cyY49af8evFvXWj3sgra98k2AvMcVrz0QAhW+ZFpgPevGQD6EQN+9vhQgvSdxmz0fa9696Eg+PX3JnDzNCTu8qckiPU51LD25oBe9BF60u6dgwL1mAi++pHGOvR95HDyzhre8cP3fPY2HML3XsDC+G1RGPW6cTTu9yHe9ldvfPFTYkD2UkMW7VNKHuzBYSr0QFDa7ZzC/PTzjuDxCNyO98c9HPXuUTb06gNy82i52PfSyCT2+Owu+ZtvdPNin9rx+cqS9kSsPPTzYBz12Uaa9pprkvUjE2T2bN0e8OjSCPQ7QmryxKgA9kx4TvTgeWTxsjAG9pCj+PDUdqD1UMIO9CiZvvZ3xKr3O3ie95UC1vNhIBr5gRbg9gg/zvYizEL1tqsy9ZhbePSbqxL0kbYi9vVbIu6fWJb1eB7I9iauVPcAdV7sOJjk905LxPefMbr03SZM8h09kvJXM/T2lMms9OYHKu9Sym7t1DCA+ELqevfLwnz2Piu+9iCaCOpNJkLx935w7QqrGvc8gxD3iZKU8cL+Lutav4r0rpU49kwmGPYRAV72WK3u8kaiVPGO5Vry9wK49OM7ZPBALcj1QgNw9xKggPPgapb0+Zdm8AkoMPqhPgj3DEFu9oL1PvaJJ/DwH1gI8sIb7vUDqlLw67rs8p7k0PMpOwzvM6849eyeiPDCFor2Y3I+9mrugu/nSOT009lw8sBG8PSS5U73S4sQ9NT8BvO8zCr3egqG9N4a/vawGID2exKo914RMvLGKP7z06My9IDhJPdKZrD2HB4Y9hF3mu0+6H74kpnW9fUbevZ/QVL2vcxK7HQozPcJ14Lzpp+S8aJIdvQUaKT3WubK90SoYPcm/mrx37fw9MOhcPqZcW73cTQE8TxdUPo+YiLwXXdw98ALGPcLyG7sCu5q9IaTWvQdpdzzePRq++DFTPV8qND17O329h+H/vS6CiT3ETKi8jBOaPWiMB72zitI9o7cGPVIyFL1D6vu9O2SrPbaLCb0gPJG7X70hPYgqqz0u08k8EvPNvWuLQj0c4wi+Gf/1PStu6T2VDbm9Hp5tveN9mj2aIXq9nyRpvWajrDygpN+8IWKYvT5K87qc/PC7pBKbPeuwzzzD/qq9pQSXve3OBb2HiQy+ix2bvbMNWL1Nvga94BhbPUaubD1NtvW9CyCZPSgsyjxA5SQ8wCNbvdEqHj4cPjm9aEzhveiUAD0oUAi8Om2FPW9XVDxZ+5q8EaEZPFEY8b08fA29viMSPsFrGT1PA4a92HcPvNvS/Twl7ko9/3WbPIs/CTxRTH290AlhvpUN/Tw89na8C6AhPaBLdb3aG6k87k7GO3vZbb1YD9i9ucskOyrtiz0LZIa9E0jkvEuJDb3+MmS9vSCTPfOH1b0YZdE9x8AdvaOkx7x7RYS7jlvNPQylfTw/Fgi9giomvTt9ojyJOIE9L0iqPT2Oc73vbdq8NiwZPJ4Z9LxmCSA9yvu1PFIRuj0DXYM8UOnltUowlzx1vpw9q2y/OxKCoj3xpxu+nws/PTLO3jwtM9m8kQzavfLXPj0fn5M8P9RSvdBdnr2M/KS8KAuFPUgykTx85aw8yFEdPUwGpr0e/Ss+87CPPD4p/jtdY1I9H1bkPe954b3TVpI9RkTdPV8kxjxZUoy9MvDaPEipYr15OZW9wlZ+vRNxtbyC41O9gT+vPG4LUrugq0E9wlCyvLTRRr5zVfw8DBFIPb/noTzcUkW7ACQXvNciSjsZ7KY9mv3fPOpwkj0VYs29E7JSvdHpoTy5v749nULTO5/KGT0iy9K9vRrPPXNilz2jXZa7rcDIOVnlw73Q6bk7vZomvlsAwL3fQjw8+nRzvfI6Gb3VowC9WCl2PHMfCT2zAL+8xPk5PUJBSrzf6II9X9dePkF96rzyyAi96X8qPndCCr675gs9U6XgPTK98z03Yp08DTmnvYJmZ70i37i9X8aWvXShWD0s0u68rhfUvW1HQT1pfz+9EvkgPgbRr7vfMkI9yFexPc0kbjyMDfy9amTMPQNVur1P3o69joVlvId3jzx5pNM9Li2/vUYDBD1RKxa+BJi9Orv/1D2iOJ+9c2zDvFBJGj7/TVi9Jar0O9u98D1Or2q88kZTPUl46TyZT6u9fi1lPc7E7rs2+qy9olnJO2ycjL1Iauu9fDfkvZK97TzoXBg9By9zPejHET1RmbG9BW/+POX8zbpB1r298ueCvQH1Mj74HzI9u9yCvctpFDz4ZG87sXe2PXFnoD2jMyY9u9nQPU4vNL60g5g7C/pEvRmiVLs9/zu+wATDPJYin73puXU92a2Uuy3Rgz31hPq8GsTDvUNcUTzeH9k8C1EgPlp8+byYCpI9yoTIPRIarLz0URi97hOtvO2NsDywSZ49dWODvfZO870V9a48/E9JPRPCAD1bwCQ+BpHJO2vCrb0gR909vWUJPjvxk7zIegW900iQvYcfjb0vboA7DEk7PbnjxrwlVCI9cfkNPb5CaL0KqZy8fIkgvWBPXT1sC4o9f5NKPYVKu7wICtY9sGffvaPEir385ka+QELbO3a+zrpRCZq8CRJTvQONlz3m+j29JKmQvcyuAb6d9Lo896BgPN4jHr6woCk9rfYSvTNaET0WTTk+F0Hnua57iD2Q2oI9b8n6PBIGMD2//dq9hGgjPtFCVz0gpGm9kWIEPWJg+jvdJIi8g5UbvQtc3zrBRDw9QRFgPesGJz79PbE9GOg7vCtC4L17v4c8qt+xPCWkzLyxnfg87+8xPWFRLr2qqwM836BAPdClh73igAu9/ReyvGBvtD2C/UY9WWC8uyE9Ujy/Ori8IRQAvKliYj3IwME83LM5PX3EtL0TvHy9ti67vQaYPzvgdv47DRIyvQp6373DDHe9PRM0vdw9mTyj4cU8IFlbPWp0571/M8k9PaQZPm+4gr0QV9c8vhe0PUP0tb3GFbE9WBhHPf0IOT2v9/G9rVu2vX0sxLzeoM69KDg+PfLqgD27Fmi9Iu34vdK15j0ziIy9ZWpOPfFfV7t6NQM9DM4XPeUnsbx8rJu9TJIBvSHdqLruy3E8qNhIurcwqz1hmpU9YH66vfJzWj0HS7E80c5kPe5HHD7+LcC83WgSvS/EiT0c5MC91dcjOfuwAL2eU5a9Ir1+vPVXCT6zoy291J+lPX4ZXr1P14W93fs5vaqF473TAg6+LYeuvYTHxTxP3469eDh4PZBNHzsMwY69gFLlPb2T+TxKE+w8he2WvVO3FD7f3lA9PMWOvbmppbyv7DI87isEPhaEsj2IHQY9Rko0PXKGC75U0qg8xeyTvax9/btrpc29bfMpPT5pjr1vsZI8H9X1PFwPvzy4z7G9FDYevhY8tbzBXp492VtJPB+bfLr64oU9LNmtPcpXozxOlJS8TNBZPS0AST1tv6097S0Ivo4e/b0Spqi88fWiPcIR/blNRxQ+PrAwvcNjjr2la5a8jzkAPuvJYb2kFzm9ddBQvV8gRL3cfwc9rp5zPTrkTr2Mk0Y98zDIPEmoCL1rR6083nOlPVq6uD0EvD09IBMEu+f8X7zNhBg+I3sVvbtUp7sArkS+aYkQPdRFiTsa0dO8NreDvS4dPj10dYU8HERxvU0nEb16QUm9e2jqOxyZI75iB4I9ph4kvcMjpjy3aaM9jQS0vDXo/D3YE+Y9dpsTPu7XQb28WCk9xPEFPsQlkj2hCfC9q01ZvcIok7wCvma9U8uKvDDgdb34C687lcwAPeSwxT0uu5o8V5TFuzRYDb6ItRo9Y215PJZvi70H2vC81xKrPb+VG72IpKk8NMm2PcCU7TzIhdG9aOxSvAl5nD1nhgY9SCJcvJryHTyuYOe9syqQPehGyz2SnRo9gKMwutnxl70SLK69IyKUvV27wrzLDh48e0zzvClJ/Lzefxm95rFtvQSYPz0kmMG8ZtlxPeVzIr1pyPA9mNMqPh5gk70OXT891oVVPgxGYr1IWOa7LYQFPhS4xjyjV5G9pqgOvujeuLsgAOe99HybPa3Qoz3zxI69WmXevTKdhz3wzbY7H8OGPevdn70mFoi8XueQPRW/WTuh6r69qxXAvHvUdz1Bguu7hXETPdmI7z3438I9cM1hvUpWbzzedu28EQl8vFbT1D2YjYm9IuOAPISS5z0LDLK9M3gPO/qASD0CBw+9PJjuvOy6Pz1jGJK8NJRYPO/It70z9eO9dr20vYx1b71cYvu94i+DvVfO87x0cIS9WI+uPe5GWTypujK9vX0DPY7Dbz1UQYM7jsGIvb7lJD4KnuQ8ktdMvDAVZz28p247fFGmPYGhij2dVgO8meurPSiNg73Hsrm9z3ObvTKRfz15k+S9S/qaPdKPMDxkdwa9/PlxPK4sNT1lGqG9tzzuvZpAjDycepS8y9N8PWxd3DwF+Cg9MZsIPTww7DwReSW9z42EPb+CQz3q3KA7YOIyvZqL1r0d+My8HL6bPfyyjr2rvBo+vUMKvVTIMb026EM96tLqPd5YJ71H/pa6VLGgvTA1Uz04q1K8iIAcO0JoD74k9y89pYydvLvEUjzVNIu809UwvZAj1bxKWT88c+WwPQafgT0Mvz4+MfWBvbRjc72vX0S+Jeg2PT/sC7zvgKe954OnvVEMuz0vl7q8YpYyvE+Mjr0oUgO8j+ZLPH75/r0swI49mTfqO73zq7w8YAE+S8HdPL524T39FKU9YOEqPmQteb1AZoc8qLECPiFMbT0q87O9ZxVAPNRIMb3MeHy7eemhPNaAhTsjgle9mkcEPdXUBD34IKo90xUsPCB1NL4kGq49fXBXPDXLU71gLKu9RbnFPeKsajzIsB09W1Q+PUE7kj2WM4O9cN0HPD187zz1pGk8DuY+PTJsR7tr+Nu9E2PiPFhB+j3Sspc9JXyBu1zDr72HYyG8T03tvTdYTb202na9OljqvWg6qbztQPA846oCvc7jAT3CLYA9xsVyPbGsA756T+M9RFvXPR0vor0SIfE8cOtTPkACDL3m+AW8bOd2Po6scjyiHuu9xo9xuaUbFr3snrG984uuPUKpVj3QBli9iM2EvY1pzzt2vJI8X01hPGG8l7u58os9QSWEPcy16DyIGF29EgT9O41HPT0KaQE9gQVtPV17xj3UpbM9kb0ivZuQgzwjIW6948gYPeFGPD3xrUy9mE6RPAgQZT1VF+691ZVyPfMvtjt4+w2+NEFsvRJa8T0JN9c8p4pdPaN8+b33iNy8nsGAvWV2Q71nhRW+mBfevC6CWDsSDBE9IWWgPaejmT1urXK9CuZPPR9tVT0ofRq9e8gjvfoOhj3UFcs8EGxPPLjWEj2TbhO90qygPTQnJj0kEMI8BbcHPLoiwb1OKsY8ti1ZPVO3KT21WUe9eWD+PeYyMz0AW7q8xL8Cu3z/1LxCr2y9NIi3vcXD4byj6IA8/XtbPO7nlL2yi3Y7uUbpvGTQh7zON5m9WHw9vf5WRr1UIU28Ylu/vZ05q7xQHqm8wxM8PcXBv71qOQ4+KR6DvO8w1b0I0lG9CSliPbZgkryhtlM9sParvdiA5ryyJe48huYbvb+mOrxb6Au9QnOWva5+YD3iJqa84kEDPY3oHT5yHhE+/3sZvVkNoL0eu589pXuWvcteA71zl5+9t4itPfVhbD2h1iW8NiypvRZ7uD0fGIY8ltntPF/InbupMII7Hu+BPOXR5L0wa7q7tJH6vHzXH73W8d09R7/cPVtGUz2IYHs9C0g9Po2qzrwHwLW963WEPXNQpLwox4k7ry8ZPZjaNL1F2uS8/LNjvZz5zb3I5Fu9OnknvL6fVj3MCLo8SAPouvglH74Ukpi9OgwGPcRucbsB7y+9XeAPPUuHSzyBFOw8UWssPYOYvLzvpw++XNwWvVcIDD7PaZQ9UpRwuuCIuTsH1G69wbJiPYy+DT2L+Ac+35eTOhAzU71dmMU8jRTivST2nr3c1ia+ssy9ulEbR7zidhm9RWccvjU8JLwz46e9GUlRPaEoG765cAY+MBMfPl0eZr2vUig8rTxCPsPVQ701t5s9TcM3Pjed1DzJmYG9dDCwvIhZDrxQboG9ppxNPb5Xrz3DiCg9f2AfvlXv+T2PPK69EOg6PKtp57q/E6A8B/anvIUEfbyW7Se9nxWDPbO8oz0M3jQ9XPJCO5wdpzy7KOQ9uHYVvhHBNrzUO4a9Nm7NPVdX1D0GUqC9tB+2PIbiNj40av+93ob/vKSKVTzL9Ge94OqavdKHIzzsH6M8KEDtPI2xwL2z25a9abuRvRBTG731yxG+UqjQvZkVxr1tCh28g9XnPR9lPrzp7t69ToYMPh+TozxIxC88aytKO6WF5z1qCA69hUZIPbJmbr2BJr47FMkhPidgSrwjxc68Q6ckPmxNEr1EMT89JZSTvNwtijx2UoW93VjfPBsuNb1YEFY8+imsPIXngLwaATO9dk6RvTlN5TxmAhY+ZWgwPeGngr2HFaQ9M7wyvMxpWLtjM9e774WbPbHqBT2U23E9obm6vbCufr3DvGk9ic/YPVTwmrw8VBA+K9QTuyd5cb2RDpY9x2hRPbiS+by6fF69lBZRO4ZyND1Kpiy9ybFbPXPqC751Kqs8M3+ZvFI93buvYPu9YggIvFAESj2bIws9VCgNPQL5DL1AyMM9ybs9vJsOwjyiVw+8vt6QPd0hoD06hFE9dj7Zve2+l7zfFEu9ufMtvXZELT2nxHq8l8XcPLjQ8b0QqiI9+3eXvVf9+ryYd7c9DWZ3PW7Duj0yCQg+jzxVPvRSErzEBeE8+eO1PVqIgbzdFuG9JveUvVtWIj2ywuk7aiXnvFBPB73R2CA9uZYxPWKsYD29fHO7v5NOvc5oK76d9eU8efzRO/1r1ruEE8a9aP4kvZKcwbyzNlw8shDIPYmUH72Vo4u9DrhGvPjcND22KJk92IMxPRsgGL2GIN69wmhCPZ8KVz0R7NM9pmPmvLySmL1gMy69AAwbvmmUQr2mhDi9q7aBuoHJwL1rA1i8wo7nvaHDpD1M2Ma4/lmmPSUBFL3JSDo+cGKCPe1Ly7wnMKs7SaEiPhuas70snFC9zmU5PmE8EryGfKq93qgMvi4XvbxIOam9kVixPUNjdD3/sva9NLjmvdkroj2x2Qq93Cr+Pd8MijwzIXS9ClIAPh4NOj1mQrO9vDmnPBrZYzwXYUO9vUEsO5RMCD0zGGM97yR9vV0Ru7zSMIC9FSQDPoX7ij1MhAC7wejQvPRo1T2qcji+vDxJvKsprrzttbq8dB2hPDblBD0EmSm8m0w7Pfj0zr3S9eW8fMuWvTHekb1eKQy+a1mXvZx3kzwjSka9MI9VPTEKuz3Oera9GTHYPSm3mz1mlKO7b2fBvcxrUz5NKjs9FvHVvXnagz01qBK93xeIPbjB7j0Jks07cX0aPLdBir0JwtO9/GGPPRtbmzxosOG9T+oUPL1Wmb0bnls90euoPYKpGL3NSIa9zQ+FveJJGL0Ql0U9oKibPYEFlb1jlTY98XT7PW1LTr0bMju94Ozhu6kECj108oQ9HqWzvRzo0r1O6xq8xH5oPE5eLTt384s9Vg7FvHU6Mb2G25Q8oODWPQczFr0d24W9CfqivWvRRb3kS0A90VeOPdBnuL1VSNM8atTovJ4c0zw3KsG7NsRdPXJotTyyV5U8creQPaw5Ub1lL6k9kRCJvXbHdTzDtzu+PeGdPFSbRD348Tg8nUKivcBg5j3G8be8WrLHvMLAf71RkZy9DTLfPEsy1r2rQ868UZk9vVt6B72OhjE+IjRmvL7BrTyJ2hc90Ta0PbD6mL3gkL69KznYPdpLND00bTC9lV0evW+EAT2bJli9h1i2vUmjkL3XnoK9SPrQuiMOuT1M/3Q9i0a9PAgXMr5mPZ472KTqu0lmnL2mT727EH02vHI5AT3dbqy7k25XPBGYkjtUmYa9qRduvGXRij0vits8nmaCPdbmy7s7FSu+x4DsPH81Hj7O93w9gIUNPA23hL00OOK9bkHevUxXa7zcEBw94lsQvJ8Yyb0JK+m7qqxDvauREz0pPqg8suNqO+DXTL3D2bI98zkVPtCfEr1OTVo9I3pZPsdMLL1hgpq7MQxAPogj+z2SR5y8aFf3ves/S70xKPe9rdOfPRRsrzz2jAO+NWDRvbqpjD3Ee5C6DPKPPfD3RLzg0yM9u2nYPe052TofRjy95CAqPYyWdbzCO0y9smrIPBI2uT2wLNA8574EvTRyoT2+rm+93dOjPeZwFrqY13O9uvcmPe9JAT4RWKq9LkDXu2jjXD0jRqa9NMunvS25qj3mN1S9yQCjPUbXDr1LNwS9oh1zvfUHl72gmQ+++WGtvKUIEDwnWQS928AJPn1JUT3bh+a9yXAhPmwYr7uRAbK7l5h1vfRAjj0f2xk9zx/RPJqboD1I6mU9qJPpPbilzj2k41s8xu2dPZrvDL6Kgjs8HWoCvaN3Tz3zVZ+9D8+3O4pZE73fwFo9B+mMPPqhOzuN+oi9xdblvS2kBDyt8+g9fGNqPcXVC716dY49ReWGPd8dIzxsPDm902cRvNSSYD0n7aA9P6V2vZ95Nb21bW49phOfPbu33zxsQgw+dW1cvWp+R72WDDG6LdqwPXTXCb1q/g89tlidvXH3A71Wk9m8L2LePIMpYr0T9gk9pphrPVuJhDtkMZ29V6SUuw7ixj2+8w0+8vc+vY8ARL1s69s9DikUvpvE5ruUWPu95Ka1PYEEIj3DHVC9pSbnvStGcD2EAqc7k6N2vFoxID332DO7ZpGRPA9Zw7yKQzW9CMpGvc4bzbsE1CQ+EJTnvO41Lj0zhAY+Y9MOPjrYsr05Woa8cCvyPdvriz1EQaq9r3F1PMlWi7tgDo+9I8qHvT8ExL3IAsA8lhAbvM8Mxz3wv8U8+jSRvK9yA74YNyg9zKwPPVCTbz2osam9vF2sPI5zAT2LNDa8bWVsPW0MfDx0NMC9DRBDu9HcrD2L1wA9MEL9PAiHTr3BiRW+dS/XPeuQ5T2G3PE8JzREvTQGvb1b8Qk9tLjlvSPC8bpYDju8K8uRvI8BiL3b5pW9zTSXve4UjTv0bSC9xRHpu5LgVr2n2xM+PiQHPoirPL1XFx09bhjWPYHxoL3IRqc9wVY/PvGwVT1wF7C906ETuxRdA7xFr6i9x/GHujcpRT2vi869ZiUqvipz4j3BzO68ZfZwPO0xUT2lqDI8w88DPURwHrzF6tq9sCQXvZ0GLr078Iu9iSLeuwCEtT1RZOQ9fCAyvk3Inj116D49RXy+PeaBjz1rA526QSIIPU5lAT5jcok794Q2u8V1gz0cT5O9CtNivH4PIz0v5La7CoS2ua+tSb1CrIW9ui3rvWm0G70Eewy+vwGevXAXF708cci84IXpPYYWwbyBtic6MyeDPZ8/yry0wTK9CzpjPCvNIj6MhSK9SkJnvfF7k7xTdzK7MHHOPBqkFz6FRKe9nwjmPUYosb0TlIM7ZU4jvRyTgT12SdG9ZN+1PdQcQLzzoT49nUPkukAly7wqPEG9HQH7vRhldz0eEpQ9a/RuPb8wU70fPde8BjfEPdgmEj3Jp3e9owGXPWPHnTznjLU8AzIBvQfFmb3MO7Y7k3cYPrFUmrudzUo+oto1vblJlb3AwRW8d2o+PtSLj7xSZY29hImavZeBZjsbmY68VQdVPUe2+r2Z1lA9eSyWPQG2JL3gp4a90ueGPEGjwDtZBya9M7pdPckMVj1YQjk+pCgaPSNKGz7TYDq+0voaOxS3nj1hvFm8LL7zve9xyjwgWN274E8tvTspFL3NeSE9pUpvPLK9gL3fQdS7vi/4vLTT6DzuOrE9PMyyPQuZf7uaxZE9xf6kPGYUDDwwy8k9RsQhPi/zhz0z0w++8+6QO2rqEr2UOfC7uBgtPa3HoL1JIKi9gOQMvdvpBj54mKo9MdHevHSOBL7lBCq8URbzPDzecb0Nu0e8BuqWPRd3T72hAIM97qW0PPxQKL3QUwK9r2rWvdg3pj1+OWM8nDOrvHfKnb3UFxi9coVDPQn9Sj11PFO8GJpEPcBPM7380W28oX/cvU0JWDwcVXm9Do1ZvAtS+71HEAI8uWQwO3xFAj4qWby8mfhhPSlxCr40gls9ftorPqx6Xb2DrZA8xM26PbSJrb0yNHa8q3YHPk2SGT2z7A+9BA4JvtZEtr2z8xA93MjrPAmCQ7wNICy9/PyEvPCcHz1YmuQ9vYMePiW1Jb39D5e8GR5yPW7Yq7w08BC9sHfKvAivdbuGo+a8fh05PbRjCT767Aq78ElvvfJrtT3jfXE9/7EkPbs4xT3HTaO9UGMFvEgAKju7c6m95xbeug28Tb17Zdi9sRwvPAEWpj0mfka8ef6FPbyeqb2E9Py9v9ZEvQ6MmL0iGtS9WdB6vVXTYruhsL481ZKiPeUvPr0VjC69VyTSPbHr/jzbYVG9Q/8Ovcp1ID4kAVe8N6YKvE51QrxyGlM8gU1TPaJyxzuXqAg9kwEtPqk9HruhfJq9aP9WPTHPtD0vEcG9XSPNPIYsorwBE1q9qmy3PV2fKb0UN629RRXnvRiBZT0r0No9GuVKvJv9AT0IdJO8KU02Ph7ZhDzzUgm9WYqAPfcBrT1/RwK99+DQPKmk17wLnp884hGbPXhVtb2TPRw+g+XfuwUo6L28/FQ9A6kNPumtjr0NgqC8lWLPvbzYLr0w+5K9muewPccLpr07FV89CS7mPcFLKb3l/Yu9xifYPNaiuDyzP229zUeiPXJYUjsasQU+plQivd+Vzz1PUCy+JEO1vNtu7rvSTT49/430vPSch7xLP6q7zpFIvTDcJ71dZ4I9OHKBPX8Zi73uj+A7Kg5rvaqY7zxflmc9SAn7PNYloT34pVg9nuNGPegXbL1RE688WAcWPh7L9D2djMK9FOuQPfG877zH3pK9qjBmPLamb72r1rK7XtXFvBgMPjyklas9Gyz0O2N6Hb6nTsI8VNsuvVu2gjtEHig8X+DfPU9Zzbz4fYY9PxTvPFYpyb0VUEu9qc72u3wmmj2O3HA9fXhTPaUswTsrN3691iLUPYyUB7wylPs83/3JPBIinb2B2KK75RTRvXRuELzrpLm9GKnJvS3f973b24W8SfAovW4XJz2n9Pe9KGwZPVnuwL3DIyY9VDcNPlpa87z3U8m5WNMzPp40TL469Xg8xr/6PXjLxbsoSoa9CmC1vde/AL1mOki9x6wBPs+BoTx3a7G9loaYvUmKBz1Cnpc94BT+PZW/2bqowJG91hj+vOe1Yr1hH4+9RNUyvNzAG73BYQo95JkCPkYGLD4YgaM93IXIvc4fVj2DGDi9i/JgPANQ1TzEhgG9lS5IvTs4+TwgVM28mrBavTP4uL2fiKa9xNYiPLExJD2pTqe98JyuPUbpyL2pX2K9D/hTvcnMe7wl/ci9dhlovW+URjso+TW8cpTvPQYSijyJUJg8ftYfvVxEQD0wRGo8p/K3vReqFz5Hc009V2KrvTruUb1dbD+89IgiPYemhzw9H+m4JwMYPl3QJTyVBpK8Qn5nvBoJ2z23QUe8H5n7PFqFHzwL3vu8Nwa0Pf/SBb22awS+er0Kvo9UITyEWbg9F0ISvZNl3DxecuS8NsNyPcXTCD0wahU9GsrMPeINAD4iYXO9DNB2vX66yL0NdKM6CEpFPS9h+bzKMOY9lQB2vN8Z47z/yDO8AZBDPrlhHTxAipq86jQFvboWjDzMqFQ8YrhHvA9OFr407kC83lgrPmoexbzpztS7CDV+Pfa3gbyWPyU9Pwc+vfM0w7suWg0+SX4BvBAf+D3L/Qq+Ttd6vcMLPD35Yhs9O5ASvtzxSr0K00g9hhGCveGEF76ywbQ6mpIKPdaAgrw1Ao08IMlvPcTRIj3wcTq9sGu5PVCl2D2SB9O75BHLPFX3TL01g549QDIWPh+7nj31+IO9zTzRPbqndbxXC9G8wB+qvPkzJL3KPpi94MycvZAJlz0f+Xk909+HPdcgBL509EC9iBkxvanRJ73bvQ29U94tPoErqr0/Y6o9qSI5O1AXjL3ippa9ff9svXhmhzzbW/A9NZYaPRz6pL1gk0W9wj4WPQMZ1jqwHHY9mK/RPLdNDr4Jk7Q8ryL8vZFAhDyXQG67+GE0vTf0CL7/2k68uH/YvKD7hT0JWo+9pNasPVAVHLyVUkA9LhkpPnXzAL5TgC+8fMvVPfRR5b22NWc9YTkNPp0jtjv4tA69FwcSO3xSJj2NqNg8AhTqPfZlUT3jBFO9Dg3nvQmsRLxZuNU9MRdAPrIMIz1sbZI8NHkFPjjSML0AXZS9DB5ZPdbYsb295IY9J6q/u1buvT0e7by8B0KPvEZlCjzvFis7QrRsPV8qyj20I5C9uWqJvKq5dD1SG6S8kJU9vY7HDrxSvSa9CnP5Oh8AID0H90u9DHyUPC8VqL0bTMy9aw3uvFtxijwCirG9uNouvfZph7147F68U9wYPnsCo70oJBa9AVzmPBUj/TxxbYk9TUsPvWNarz26whW8t/LBvYEI1jxcJnA9akm1PRFIgz1zPqs86oZ/PayALD3+DB++E8u9Oo9/vjw/zQu90wbDPZ3srLuBwcW7eMaBPTzyeL0YM929h53cvRs1Rj1tCtg9CLeKvVlJ8Dx3+gY8junfPdPEYjz733q91ah1PWtzaD03KYe9GDZ2vcKkIL3V8ju9Rcg5PfOkEz1Wa9A93afovSxCm7sbKYO9Wa/vPZ1g0b068AG805FovSO0rz0CO0K7kUgVPeqzRb7i2xQ9i0kbPg38vL2H8rO9/yrrPJsCrTy/9xQ91dqEPMuvurzmVUE+vlUSPK7Bij2NygG++OZdvQkSxDw3tVG83KA4OYdVrj1JjeQ8D+inPGRPqr2F4ng95N+EPXwdvr3ZmNI8Q7RWvTJ5yT3TWEo9HFDavNuEmD3lWGo9fILRPDiCnDzVHIo9CkjpPVIKJz6wLU29h3tLPRk2CD1AB7682z9Mvf9RiLyFj7Y8Lqejvc6DBj4TBTk9GOpSPUEVbzyxiAk9r2mevAsWYL0ugDe8Oq37Pa+vhzyxh807pMi4PW9WNbsM8fu9lSsAOZODfz2ox8a808BRPbq+5b1GMDY8ho8KPWzvzTvdOKg9ymvwO0Agf72YizQ9s/mivVAZbTzxiCI9snWbu1qD6b0+Edi8XL49PAo4jD0WH5q9lZaHPXKZhL2lJ4Y9IeQOPk310bxOsXk9vnAfPov3SL3nmSk9tp4RPh0RvzrRLqy9ki/IvW/EyTzSqEm9Iw7DPNIUYz27F8i9bNMHvtblAT3vZtE9LX4MPlfyQbxB2Am9gnIrPWtdCL1d6hS9wu1fvahUvr2wAZa9+XgTPYc3dT5Q1809ZP2kvRXiBbyr9sg8AXhJPXiwgj1j1ga89K3GvHkq9T14OC284czivJW5JT081+29+eMgve/evz1scXK8tPDSu14ASL3CqV69Fk7UvRqsITzlU+K9M+qTvQVpzTx/cI48mY02PixuiL1a50y8TohrvM5RhD0ozaQ8W4bhvCjl3T2czMK8GPlQPU5Yzr1Tnx+9FGUgPn1klb1l3gG+Fy0mvH2hhr2KB5+9EwNoPHv0OTx58hi8RLzXPUhHAD7z4tQ8W6KHu0raor2CLhO9otuzvH7Ijz3ehaU9pO0Qvb13Yr29hvQ8SJC1PVs7jT2G5EY95kFpvV8usz2erg49iRp1vXvCAD39p+s9unspPrK/Dr2wXlg9XLYjveu0Nb1F92K7jKPaPeFIFbxHi8q9aZAXvXE1NDycn+Y8hb1iustsBr5N6Po8LW0JPmqUVb0MAzO9diAKPmtpFT0yS1C8mi4avZSTCb0y1Kc92Q2RPWvvrz0u+629B5KovWuOhz0ViA08cynpvS07yDsvEBA84QgMvjwPojwE9xa8hNmZO+UrPDyShwa+VYZMvTKfbj30cno8u10MPSVHTL2cXyy9RGVNPD1EJb0Vsam9oN4NPiQZADxR36W9jRhFO8+SQbnLjvq8efzGvNs4Ir2zzdi8io5UvWkkvz2Jn5G9EteSPdQbrb3xntU8KAWOPHXSob2IMrw8A6brPbh1SL3caVu8xSOEvYVb7jwWkyu+VSN/vSW8lz2UW7c9mXEPPaaeMLy38wK8TV1avRaztL0KLpU9XEjbPdBE77tbHtg8cGLavfdkjrxQnEC9pWCGPbSPX75t0iO8DiQHvaDwDj72VSe9OVPMPX5QLL1AsFI9rGH1PQDCe7zhEwy8PA+VPdlZEr6wsD28tFrjPROtTzwOlYe9ZY4IvgYFdbpmBGC9jd27vM7SPT2HgkS9RWKevTz8mLrH6y8947AzPgyHAL1TaYG9aRrBPVK6qr3wiyq8xf5WPJiMQ736dQ2+KmDdu5CPtD2+Jzw8tACSvXZlrb0Q+wK+myi3vIgkaDzMbSS9G3AUvXrJFj1FSii9IO2VvBzexT2cm7i95bzHuy+jVLxXjn29uvTUPXiWF71X69q80SmTvb9YsL1v0Q6+tDOvvZ7BQL2AJwG9XcQHveNPML3HN0C88tVkPCUSdrwNmoU77jO4vZ3T9D1cur69LXPSvZwhkr383to8pEqbPXplSD0PNzM9TPqaPS1rDr0BMUw812itvEmdEj2PGrO9eTmGPYRxVzwQD5s8oP8lPRLisL2GsBy8koINvh6ADD1CT6g93G0GOoT+MLtaK4W97yGIPfoOmDx4uow6T2uuvSGegD1uMG08Gj28vVCm2Dw7l8c9sC6sPLjE2LyPiHA9sSbMPYS7nL3AaTs7OQNwPnJ+/LxTj3S9GwauPJMC3rwb3b88G9E2Pb7aD77rIru9SREIPg+MfrxQXR49sxfzPVQMPD0HW289qCyGvApY6rw5Fys+4fLTPLRWvD3yoO69AjXRvZSkJz3/GFs7VL6evfrm6LrLv448U7DWvRVQjr0/OUm8GsVWPYH0v73ppOw8ThAJvWhwX7z5Yzy9gyl3PdMp1bxi17S8oJjUPeGLNr2QWGa9modFPgaN3jzs4Ju9Bl6tvL3vlLt6NBS9wWJZvdVwL7kMJ2O9laWuvC7f8TwnJ3I8z1Qdva6mfL2ccfe7O+Y8PHHKoL1uYu88DpyZPWWlDL6trJ08oBSbPFBctr0Uct+95nLBPV0Y1Ty04yE+leMAPGqzbTwDoRa8WW9tuwsMAj1wpvM83n5+PRfWV7wAmo48B+Q0vlPiEbwH6qG8aKdEPVU++r2NnMU82liJvYEq6j2lGO47R2ftPVRL/7zN09I8sghVPu45bb2Bjy+8n/M6PphCBr71TRS9wM/GPQ2XYDyvYnO8/V4LvmjVbDx/vGm95o67PGVbub1F0pa8xctWvVx2Wrxd00g9L44sPsUQVL12q/c8eZsHPiPzq738fC68TtY+PCun873bF5A94sQuvdlbYD1jKAW800zBvALzx7zRqNQ8lzrYPPgKHT5KOxC8tQNtOk/o6D12ywa+aah3Pfv3qD1VzZi9uXXlPEdeND2Y1Wu8VocgvBzhtb3+3+G9xF6pvE8GOb1dkcm91wLqvMX7ZjwFIFA9v8j1PVxf2DwgXhy9TBm9PEfeD7yPxSW7KYZGuztYXj7Jg0S7OmcMvU9hYr2mLGS8sGaoPcyOV7xRGmK9DifFPZdrnLyKo8e8oCQgPdk43TwWZBq9kGAiPkFt9TzFVnE9PLOiPAAeGT01dSa99oXivSO8Cb1p2VQ980R+PQUM/Du1tAm9q+PhPQ5n1zynLf29G2QjvZz0oj0BCC49PbvjOXuqpT0lLuk8kh7dPfYFnr2P8Fw9vuzEvMq2qbutfzY6mgMRPo3Ujb1mewO9w7a2vX8pKLzT1yA9DuWGPWRhkb2jlSs8gJeFPSzfJb3i7j69clOUPGPPoTzEja+8EBMbPbLDBT12QUQ+tDbRvAgcjLxcf8+9IAacvUarRDz6icI8VjbKvUx8rb2IYd28jGU+vVxNxDu9j0u8MZ5wvOePjL0SIx08iOvNvRWAVD05XiG7JKyaPbiouLwbbNu84jAHPRygur1XTIq7Jo7ZPTDDRTyIz6C9gyXeu71Ed7zXIao6ttYrvRiJA70foZi9DHjeO6JRDD6N1aI9MCuWuyBeIr0uZIk8XnzlvEqDBL5q/AK9UCa3PVn/2r38ejW9qKu7vJgXVr3CDtm7mBxHvYvZnz0YnVU8V0Buu33JHL3Ya6q9EXr2Oty7wjzuQZI9PvR5Pb19/runvwE9wW8bvrugM72t4169sdYgPG86Qr49V1Q8+AAIvS0a+D1FA0I8IzoZPvD4ib1qneA9nwhYPrc+v73zqD298v5SPgxov72jaNa9AHULPm41nD14pZe8e5gevrCkfL1nA4E890iAOkxiYL0e2oK9OYLmu1oI1zwK9Xk9OlMyPtNJUr38bUu9zFqGPR2FBr0A7BK9UDgcPKSIRb1XKY87ikMRPPVFzD1GuHw92Y63vDVkgz3bFMC6eNo1PF8j6TzJMAm+i+z/POrCsj3wxAK+bTHYvVGuv72UINy9gVkuPFXN6zvxmJ690hYXvc6Ujr2BboW9VzqMvKauur3UjCq+Ea6ovIamTz0='}
reference_hashes = {}
for name, encoded in REFERENCE_FILES.items():
    content = base64.b64decode(encoded)
    (REFERENCE/name).write_bytes(content)
    reference_hashes[name] = hashlib.sha256(content).hexdigest()
(RESULTS/'reference-hashes.json').write_text(json.dumps(reference_hashes, indent=2))
checked([PYTHON, '-m', 'unittest', 'test_transcript_gaps', 'test_window_review', 'test_review_regions'], cwd=WORK)
print('Improved reference ready:', reference_hashes)


## Credentials and optional checkpoint restore
The token is never embedded in this notebook or printed. Kaggle reads the authorized secret; on a laptop, use the environment variable or the hidden prompt. If restoring checkpoints, attach the downloaded archive as another private dataset. Completed stages with an exact fingerprint are reused; interrupted stages rerun.

In [ ]:
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    ENV['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
else:
    import getpass
    ENV['HF_TOKEN'] = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not ENV['HF_TOKEN']:
    raise RuntimeError('A token with diarization-model access is required.')
if ON_KAGGLE:
    for archive in Path('/kaggle/input').rglob('stage-checkpoints.zip'):
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                target = (BASE/member.filename).resolve()
                if not target.is_relative_to(CACHE.resolve()):
                    raise RuntimeError('Unexpected checkpoint archive path')
            zipped.extractall(BASE)
print('Credentials configured; token not displayed.')

In [ ]:
def export_checkpoints():
    if CACHE.exists():
        shutil.make_archive(str(BASE/'stage-checkpoints'), 'zip', CACHE.parent, CACHE.name)

def run_test(video_name, stem, batch_size=4):
    output = RESULTS/(stem+'_evidence.json')
    command = [PYTHON, str(WORK/'chainofrules.py'), str(Path(video_name) if Path(video_name).is_absolute() else DATA/video_name),
               '--voice-priors', str(REFERENCE/'voice_embeddings.npy'),
               '--face-priors', str(REFERENCE/'face_embeddings.npy'),
               '--output', str(output), '--cache-dir', str(CACHE),
               '--batch-size', str(batch_size)]
    started = time.monotonic()
    try:
        with (RESULTS/(stem+'.log')).open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                # Never save or display the secret even if a dependency prints it.
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush()
                print(line if len(line) < 1000 else line[:1000]+' ... [full line saved in log]\n', end='')
            if process.wait() != 0:
                raise RuntimeError('Test failed; see the log. For CUDA out-of-memory, retry with batch_size=1.')
    finally:
        export_checkpoints()
    result = json.loads(output.read_text())
    transcript = '\n'.join(f"[{s['start']:.2f}-{s['end']:.2f}] {s['final_speaker']} (strength={s['final_confidence']:.2f}): {s['text']}" for s in result['segments'])
    (RESULTS/(stem+'_transcript.txt')).write_text(transcript+'\n')
    with (RESULTS/'runtime-packages.txt').open('w') as packages:
        checked([PYTHON, '-m', 'pip', 'freeze'], stdout=packages)
    print(f'Elapsed: {(time.monotonic()-started)/60:.1f} minutes')
    print('Runtime:', result.get('runtime'))
    return result

def extract_clip(start, duration, name):
    clip = WORK/name
    checked(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error', '-y',
             '-ss', str(start), '-i', str(DATA/'video.mp4'), '-t', str(duration),
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-c:a', 'aac', str(clip)])
    return clip


def run_targeted_review():
    output_dir = RESULTS/'targeted-review'
    command = [PYTHON, str(WORK/'review_transcript_regions.py'),
        '--video', str(DATA/'video.mp4'),
        '--baseline', str(RESULTS/'full_video_evidence.json'),
        '--target-reference', str(REFERENCE/'voice_embeddings.npy'),
        '--other-reference', 'Opening_officer='+str(REFERENCE/'opening_officer_reference.npy'),
        '--output-dir', str(output_dir), '--cache-dir', str(CACHE/'targeted-review'),
        '--weak-confidence', str(REVIEW_WEAK_CONFIDENCE),
        '--short-seconds', str(REVIEW_SHORT_SECONDS),
        '--minimum-gap', '5', '--context-seconds', '3',
        '--maximum-window-seconds', '30', '--window-overlap-seconds', '4',
        '--device', 'cuda' if ON_KAGGLE else 'auto']
    if not ON_KAGGLE:
        command += ['--speechbrain-cache', str(DATA/'pretrained_models/spkrec-ecapa-voxceleb')]
    for start, end in REVIEW_CONTROL_REGIONS:
        command += ['--extra-region', f'{start}:{end}']
    baseline_before = (RESULTS/'full_video_evidence.json').read_bytes()
    started = time.monotonic()
    try:
        with (RESULTS/'targeted-review.log').open('w') as log:
            process = subprocess.Popen(command, cwd=WORK, env=ENV, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                line = line.replace(ENV['HF_TOKEN'], '[REDACTED]')
                log.write(line); log.flush(); print(line, end='')
            if process.wait() != 0:
                raise RuntimeError('Targeted review failed; see targeted-review.log. Checkpoints were retained.')
    finally:
        export_checkpoints()
    assert (RESULTS/'full_video_evidence.json').read_bytes() == baseline_before
    summary = json.loads((output_dir/'comparison_summary.json').read_text())
    print(f"Targeted review elapsed: {(time.monotonic()-started)/60:.1f} minutes")
    print(json.dumps(summary, indent=2))
    return summary


## Known-clip checks, baseline, then supplemental review

The 30-second opening clip validates the unchanged baseline and actual GPU providers first. Set `RUN_HEIGHT_CONTROL=True` only if you also want to repeat the previously validated 65-second height/weight control. The full video then runs once with the current pipeline. The review pass selects regions from that result and uses the same target voice reference plus an independently captured opening-speaker comparison reference. The name `Opening_officer` describes the supplied reference provenance; it is not police recognition.

Selected windows and their total duration are printed before review decoding. Every window is checkpointed. If Kaggle stops, download `stage-checkpoints.zip`, attach it with the video on the next session, and rerun the notebook. Duplicate hypotheses from overlapping windows remain visible for comparison rather than being silently merged.


In [ ]:
opening_clip = extract_clip(0, 30, 'opening_30s.mp4')
controls = [(opening_clip, 'opening', 0)]
if RUN_HEIGHT_CONTROL:
    second_clip = extract_clip(625, 65, 'height_weight_65s.mp4')
    controls.append((second_clip, 'height_weight', 625))
for clip, stem, offset in controls:
    result = run_test(str(clip), stem, BATCH_SIZE)
    (RESULTS/(stem+'_source_offset.json')).write_text(json.dumps({'source_offset_seconds':offset}))
    providers = result.get('runtime', {}).get('face_providers', {})
    if ON_KAGGLE and (not providers or any('CUDAExecutionProvider' not in p for p in providers.values())):
        raise RuntimeError('Actual face-model GPU check failed. Full video has not started; review the log.')
    print((RESULTS/(stem+'_transcript.txt')).read_text())
if RUN_FULL_VIDEO:
    full_video = run_test('video.mp4', 'full_video', BATCH_SIZE)
    if RUN_GAP_RECOVERY:
        recovery_dir = RESULTS/('gap-review-' + str(time.time_ns()))
        try:
            checked([PYTHON, str(WORK/'recover_transcript_gaps.py'), str(DATA/'video.mp4'),
                     '--baseline', str(RESULTS/'full_video_evidence.json'),
                     '--output-dir', str(recovery_dir), '--minimum-gap', '5',
                     '--window-seconds', '20', '--overlap-seconds', '10', '--context-seconds', '2'], cwd=WORK)
        finally:
            export_checkpoints()
        recovered = json.loads((recovery_dir/'transcript_with_candidates.json').read_text())
        assert recovered['segments'] == full_video['segments']
        print('Gap review saved separately:', recovery_dir)
    if RUN_TARGETED_REVIEW:
        targeted_review = run_targeted_review()
else:
    print('Full video disabled. Review the known-clip results, then set RUN_FULL_VIDEO=True.')


## Save results

Download both archives. `diarization-results.zip` contains the unchanged baseline, gap candidates, targeted review hypotheses, readable transcripts, logs, selection details, and the comparison summary. `stage-checkpoints.zip` permits exact review-window reuse after a Kaggle interruption.


In [ ]:
from IPython.display import FileLink, display
export_checkpoints()
shutil.make_archive(str(BASE/'diarization-results'), 'zip', RESULTS)
display(FileLink(str(BASE/'diarization-results.zip')))
display(FileLink(str(BASE/'stage-checkpoints.zip')))
print('Saved in:', BASE)